In [ ]:
"""
Document Reducer — Extracts PsO-relevant sections from Docling structured documents.

Takes a DoclingDocument object + brand name → outputs focused text excerpt.
Three strategies: heading match, table search, cross-reference following.
No LLM calls. Pure algorithmic document understanding.
"""
import re
from pathlib import Path
from typing import Optional
from docling_core.types.doc.document import DoclingDocument

# === BRAND ALIASES ===
BRAND_ALIASES = {
    "TREMFYA": ["tremfya", "guselkumab"],
    "STELARA": ["stelara", "ustekinumab"],
    "ENBREL": ["enbrel", "etanercept"],
    "AMJEVITA": ["amjevita", "adalimumab-atto", "adalimumab"],
    "COSENTYX": ["cosentyx", "secukinumab"],
    "SKYRIZI": ["skyrizi", "risankizumab"],
    "OTEZLA": ["otezla", "apremilast"],
    "SILIQ": ["siliq", "brodalumab"],
    "CIMZIA": ["cimzia", "certolizumab"],
    "BIMZELX": ["bimzelx", "bimekizumab"],
    "REMICADE": ["remicade", "infliximab"],
    "OTULFI": ["otulfi", "ustekinumab-aauz"],
    "ILUMYA": ["ilumya", "tildrakizumab"],
    "YESINTEK": ["yesintek", "ustekinumab-kfce"],
    "ACITRETIN": ["acitretin"],
}

# PsO keywords (word-boundary matched)
PSO_PATTERNS = [
    re.compile(r'\bplaque\s+psoriasis\b', re.IGNORECASE),
    re.compile(r'\bPsO\b'),
    re.compile(r'\bpsoriasis\b', re.IGNORECASE),
]

# Utility section headers — contain parameters that apply across indications
UTILITY_PATTERNS = [
    re.compile(r'\bduration\s+of\s+approval\b', re.IGNORECASE),
    re.compile(r'\bquantity\s+limit\b', re.IGNORECASE),
    re.compile(r'\breauthorization\b', re.IGNORECASE),
    re.compile(r'\brenewal\s+criteria\b', re.IGNORECASE),
    re.compile(r'\binitial\s+authorization\b', re.IGNORECASE),
    re.compile(r'\binitial\s+approval\b', re.IGNORECASE),
]

# Cross-reference pattern
GOTO_PATTERN = re.compile(r'Go\s+to\s+#\s*(\d+)', re.IGNORECASE)

# Criteria number at start of cell
CRITERIA_NUM_PATTERN = re.compile(r'^\s*(\d+)\.\s')


# === DOCUMENT INDEXER (Stage 1) ===

class DocumentIndex:
    """Indexed representation of a Docling document for fast traversal."""

    def __init__(self, doc: DoclingDocument):
        self.doc = doc
        self.items = list(doc.iterate_items())  # [(item, level), ...]
        self.headers = []      # [(position, text, level), ...]
        self.tables = []       # [(position, table_index, item), ...]
        self.text_items = []   # [(position, text, level, item_type), ...]

        self._build_index()

    def _build_index(self):
        """Single pass through the document to build all indexes."""
        table_counter = 0
        for pos, (item, level) in enumerate(self.items):
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''

            if item_type == 'SectionHeaderItem':
                self.headers.append((pos, text, level))
                self.text_items.append((pos, text, level, 'header'))

            elif item_type == 'TableItem':
                self.tables.append((pos, table_counter, item))
                table_counter += 1

            elif item_type in ('TextItem', 'ListItem'):
                self.text_items.append((pos, text, level, item_type))

    def get_table_dataframe(self, table_item):
        """Safely get a DataFrame from a table item."""
        try:
            return table_item.export_to_dataframe(self.doc)
        except:
            try:
                return table_item.export_to_dataframe()
            except:
                return None

    def get_section_text(self, start_pos, end_pos):
        """Get all text content between two positions."""
        parts = []
        for pos, (item, level) in enumerate(self.items):
            if pos < start_pos:
                continue
            if pos >= end_pos:
                break
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                parts.append(f"\n## {text}\n")
            elif item_type == 'TableItem':
                df = self.get_table_dataframe(item)
                if df is not None:
                    parts.append(df.to_markdown(index=False))
                    parts.append("")
            elif item_type == 'ListItem':
                parts.append(f"- {text}")
            elif text.strip():
                parts.append(text)
        return "\n".join(parts)


# === SECTION SCOPER ===

def find_section_scope(index: DocumentIndex, anchor_table_pos: int) -> tuple:
    """
    Find the section boundary around an anchor table.
    Uses adaptive gap detection — no magic numbers.

    Returns (start_pos, end_pos) defining the section.
    """
    # Get all table positions
    table_positions = [pos for pos, _, _ in index.tables]
    if not table_positions:
        return (0, len(index.items))

    # Find anchor in the table list
    anchor_idx = None
    for i, (pos, _, _) in enumerate(index.tables):
        if pos == anchor_table_pos:
            anchor_idx = i
            break
    if anchor_idx is None:
        return (0, len(index.items))

    # Collect seed: 10 tables in each direction from anchor
    seed_start = max(0, anchor_idx - 10)
    seed_end = min(len(table_positions), anchor_idx + 11)
    seed_positions = table_positions[seed_start:seed_end]

    # Calculate gaps between consecutive tables in seed
    gaps = []
    for i in range(1, len(seed_positions)):
        gaps.append(seed_positions[i] - seed_positions[i - 1])

    if not gaps:
        return (0, len(index.items))

    # Median gap as baseline
    sorted_gaps = sorted(gaps)
    median_gap = sorted_gaps[len(sorted_gaps) // 2]

    # Initial boundary: 3× median
    initial_threshold = max(median_gap * 3, 10)

    # Walk backward from anchor to find start
    start_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, 0, -1):
        gap = table_positions[i] - table_positions[i - 1]
        if gap > initial_threshold:
            break
        start_pos = table_positions[i - 1]

    # Walk forward from anchor to find end
    end_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, len(table_positions) - 1):
        gap = table_positions[i + 1] - table_positions[i]
        if gap > initial_threshold:
            break
        end_pos = table_positions[i + 1]

    # Refine: recompute with 2× largest intra-cluster gap
    cluster_positions = [p for p in table_positions if start_pos <= p <= end_pos]
    if len(cluster_positions) > 1:
        intra_gaps = [cluster_positions[i + 1] - cluster_positions[i]
                      for i in range(len(cluster_positions) - 1)]
        refined_threshold = max(max(intra_gaps) * 2, 10)

        # Re-walk with refined threshold
        start_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, 0, -1):
            gap = table_positions[i] - table_positions[i - 1]
            if gap > refined_threshold:
                break
            start_pos = table_positions[i - 1]

        end_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, len(table_positions) - 1):
            gap = table_positions[i + 1] - table_positions[i]
            if gap > refined_threshold:
                break
            end_pos = table_positions[i + 1]

    # Extend slightly beyond the last table to capture trailing text
    end_pos = min(end_pos + 5, len(index.items))
    # Extend start to capture preceding headers
    for pos, text, level in reversed(index.headers):
        if pos < start_pos and start_pos - pos < 20:
            start_pos = pos
            break

    return (start_pos, end_pos)


# === STRATEGY A: Heading Match ===

def strategy_heading_match(index: DocumentIndex, brand: str) -> list:
    """
    Find sections with PsO-related headings.
    Returns list of (start_pos, end_pos) ranges.
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    collected = []

    for i, (pos, text, level) in enumerate(index.headers):
        text_lower = text.lower()

        # Check if header mentions PsO
        is_pso = any(p.search(text) for p in PSO_PATTERNS)

        # Check if header mentions the brand
        is_brand = any(a in text_lower for a in aliases)

        # Check if header is a utility section (duration, quantity, reauth)
        is_utility = any(p.search(text) for p in UTILITY_PATTERNS)

        if is_pso or is_brand or is_utility:
            # Find end: next header at same or higher level
            end_pos = len(index.items)
            for j in range(i + 1, len(index.headers)):
                next_pos, next_text, next_level = index.headers[j]
                if next_level <= level:
                    end_pos = next_pos
                    break

            collected.append((pos, end_pos))

    return collected


# === STRATEGY B: Table Cell Search ===

def strategy_table_search(index: DocumentIndex, brand: str) -> list:
    """
    Find tables containing the brand name.
    Returns list of (start_pos, end_pos) ranges.
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    collected = []

    for pos, table_idx, table_item in index.tables:
        df = index.get_table_dataframe(table_item)
        if df is None:
            continue

        df_str = df.to_string().lower()
        if any(a in df_str for a in aliases):
            # Include the table and a few items before/after for context
            start = max(0, pos - 3)
            end = min(len(index.items), pos + 3)
            collected.append((start, end))

    return collected


# === STRATEGY C: Cross-Reference Following ===

def strategy_cross_reference(index: DocumentIndex, brand: str) -> list:
    """
    Follow "Go to #X" routing within the section scope.
    BFS traversal through criteria flowchart.
    Returns list of (start_pos, end_pos) ranges.
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    collected = []

    # Step 1: Find tables that mention both PsO and routing
    pso_routing_tables = []
    for pos, table_idx, table_item in index.tables:
        df = index.get_table_dataframe(table_item)
        if df is None:
            continue

        df_str = df.to_string().lower()
        has_pso = any(p.search(df_str) for p in PSO_PATTERNS)
        has_routing = bool(GOTO_PATTERN.search(df_str))

        if has_pso and has_routing:
            pso_routing_tables.append((pos, table_idx, table_item, df))

    if not pso_routing_tables:
        return collected

    # Step 2: For each PsO routing table, find the section scope
    for anchor_pos, anchor_idx, anchor_item, anchor_df in pso_routing_tables:
        scope_start, scope_end = find_section_scope(index, anchor_pos)

        # Collect all tables within scope
        scope_tables = []
        for pos, table_idx, table_item in index.tables:
            if scope_start <= pos <= scope_end:
                df = index.get_table_dataframe(table_item)
                if df is not None:
                    scope_tables.append((pos, table_idx, df))

        # Step 3: Find PsO routing entries in the anchor table
        refs_to_follow = set()
        for row_idx in range(len(anchor_df)):
            row_text = ' '.join(str(v) for v in anchor_df.iloc[row_idx].tolist()).lower()
            if any(p.search(row_text) for p in PSO_PATTERNS):
                # Extract "Go to #X" from this row
                full_row_text = ' '.join(str(v) for v in anchor_df.iloc[row_idx].tolist())
                refs = GOTO_PATTERN.findall(full_row_text)
                refs_to_follow.update(int(r) for r in refs)

        # Step 4: BFS — follow references
        visited = set()
        tables_to_include = {anchor_pos}
        queue = list(refs_to_follow)

        while queue:
            target_num = queue.pop(0)
            if target_num in visited:
                continue
            visited.add(target_num)

            # Search scope tables for this criteria number
            for tbl_pos, tbl_idx, tbl_df in scope_tables:
                for row_idx in range(len(tbl_df)):
                    cell_0 = str(tbl_df.iloc[row_idx, 0]).strip()

                    # Check both formats: bare number in col 0, or number prefix in text
                    match_bare = re.match(r'^(\d+)\.?\s*$', cell_0)
                    match_prefix = CRITERIA_NUM_PATTERN.match(cell_0)

                    found_num = None
                    if match_bare:
                        found_num = int(match_bare.group(1))
                    elif match_prefix:
                        found_num = int(match_prefix.group(1))

                    if found_num == target_num:
                        tables_to_include.add(tbl_pos)

                        # Check for further routing in this row
                        full_row = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
                        new_refs = GOTO_PATTERN.findall(full_row)
                        for r in new_refs:
                            r_int = int(r)
                            if r_int not in visited:
                                queue.append(r_int)

        # Step 5: Collect all table positions in the chain
        for tbl_pos in tables_to_include:
            start = max(scope_start, tbl_pos - 2)
            end = min(scope_end, tbl_pos + 2)
            collected.append((start, end))

    return collected


# === MERGE AND SERIALIZE ===

def merge_ranges(ranges: list) -> list:
    """Merge overlapping (start, end) ranges."""
    if not ranges:
        return []
    sorted_ranges = sorted(ranges, key=lambda x: x[0])
    merged = [sorted_ranges[0]]
    for start, end in sorted_ranges[1:]:
        prev_start, prev_end = merged[-1]
        if start <= prev_end + 3:
            merged[-1] = (prev_start, max(prev_end, end))
        else:
            merged.append((start, end))
    return merged


def serialize_items(index: DocumentIndex, ranges: list) -> str:
    """Convert collected item ranges back to readable text."""
    parts = []
    seen_positions = set()

    for start, end in ranges:
        section_text = index.get_section_text(start, end)
        # Avoid duplicate content
        if start not in seen_positions:
            parts.append(section_text)
            seen_positions.add(start)

    return "\n\n---\n\n".join(parts)


# === MAIN REDUCER ===

def reduce_document(doc: DoclingDocument, brand: str, indication: str = "PsO") -> str:
    """
    Main entry point. Takes a Docling document object and returns focused text
    containing only PA criteria for the given brand + indication.

    Args:
        doc: Docling structured document object
        brand: Target drug brand name (e.g., "CIMZIA")
        indication: Target indication (default "PsO")

    Returns:
        Focused text excerpt ready for LLM parameter extraction.
    """
    # Build index
    index = DocumentIndex(doc)

    total_items = len(index.items)
    total_tables = len(index.tables)
    print(f"  Reducer: {total_items} items, {total_tables} tables, {len(index.headers)} headers")

    # Run all three strategies
    ranges_a = strategy_heading_match(index, brand)
    ranges_b = strategy_table_search(index, brand)
    ranges_c = strategy_cross_reference(index, brand)

    print(f"  Strategy A (headings): {len(ranges_a)} ranges")
    print(f"  Strategy B (tables): {len(ranges_b)} ranges")
    print(f"  Strategy C (cross-ref): {len(ranges_c)} ranges")

    # Merge all ranges
    all_ranges = ranges_a + ranges_b + ranges_c
    if not all_ranges:
        print(f"  WARNING: No relevant content found. Returning full document.")
        return doc.export_to_markdown()

    merged = merge_ranges(all_ranges)
    print(f"  Merged: {len(merged)} ranges")

    # Serialize
    result = serialize_items(index, merged)

    # Stats
    full_text = doc.export_to_markdown()
    reduction = (1 - len(result) / len(full_text)) * 100 if full_text else 0
    print(f"  Output: {len(result):,} chars ({reduction:.0f}% reduction from {len(full_text):,})")

    return result


# === MAIN ===

if __name__ == "__main__":
    import sys
    import openpyxl

    OBJ_DIR = "work/doc_objects"
    EXCEL_PATH = "PA_Business_Rules.xlsx"
    OUT_DIR = Path("reduced")
    OUT_DIR.mkdir(exist_ok=True)

    mode = sys.argv[1] if len(sys.argv) > 1 else "test"

    if mode == "test":
        # Run on three representative files
        test_cases = [
            ("56061-4538520", "STELARA", "Simple single-drug policy"),
            ("283789-5060638", "ENBREL", "Formulary table"),
            ("66156-4274314", "CIMZIA", "Indication-organized flowchart"),
        ]
        for stem, brand, label in test_cases:
            obj_path = Path(OBJ_DIR) / f"{stem}.json"
            if not obj_path.exists():
                print(f"SKIP: {obj_path} not found")
                continue

            print(f"\n{'='*70}")
            print(f"{label}: {stem}.pdf | {brand}")
            print(f"{'='*70}")

            doc = DoclingDocument.load_from_json(obj_path)
            result = reduce_document(doc, brand)

            out_path = OUT_DIR / f"{stem}__{brand}.txt"
            out_path.write_text(result, encoding="utf-8")
            print(f"  Saved: {out_path}")

            # Preview
            print(f"\n  --- Preview (first 500 chars) ---")
            print(f"  {result[:500]}")
            print(f"  --- End preview ---")

        print(f"\nTest complete. Outputs in {OUT_DIR}/")

    elif mode == "batch":
        # Run on all 79 submissions
        print("Running reducer on all submissions...")
        wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
        ws = wb['Submissions']
        submissions = []
        for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
            if row[0] and row[1]:
                submissions.append((str(row[0]).strip(), str(row[1]).strip()))
        wb.close()

        doc_cache = {}
        for i, (filename, brand) in enumerate(submissions):
            stem = Path(filename).stem
            print(f"\n[{i+1}/{len(submissions)}] {filename} | {brand}")

            if stem not in doc_cache:
                obj_path = Path(OBJ_DIR) / f"{stem}.json"
                if not obj_path.exists():
                    print(f"  ERROR: {obj_path} not found")
                    continue
                doc_cache[stem] = DoclingDocument.load_from_json(obj_path)

            doc = doc_cache[stem]
            result = reduce_document(doc, brand)

            out_path = OUT_DIR / f"{stem}__{brand}.txt"
            out_path.write_text(result, encoding="utf-8")

        print(f"\nDone. Outputs in {OUT_DIR}/")

    else:
        print(f"Usage:")
        print(f"  python document_reducer.py test    — Run on 3 test files")
        print(f"  python document_reducer.py batch   — Run on all 79 submissions")

In [ ]:
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument

OBJ_DIR = "work/doc_objects"
OUT_DIR = Path("reduced")
OUT_DIR.mkdir(exist_ok=True)

test_cases = [
    ("56061-4538520", "STELARA", "Simple single-drug policy"),
    ("283789-5060638", "ENBREL", "Formulary table"),
    ("66156-4274314", "CIMZIA", "Indication-organized flowchart"),
]

for stem, brand, label in test_cases:
    obj_path = Path(OBJ_DIR) / f"{stem}.json"
    if not obj_path.exists():
        print(f"SKIP: {obj_path} not found")
        continue

    print(f"\n{'='*70}")
    print(f"{label}: {stem}.pdf | {brand}")
    print(f"{'='*70}")

    doc = DoclingDocument.load_from_json(obj_path)
    result = reduce_document(doc, brand)

    out_path = OUT_DIR / f"{stem}__{brand}.txt"
    out_path.write_text(result, encoding="utf-8")
    print(f"  Saved: {out_path}")

    print(f"\n  --- Preview (first 500 chars) ---")
    print(result[:500])
    print(f"  --- End preview ---")

print(f"\nTest complete. Outputs in {OUT_DIR}/")

In [ ]:
"""
Document Reducer v2 — Extracts PsO-relevant sections from Docling structured documents.

Architecture:
  Phase 1: Find the brand's section boundaries
  Phase 2: Within those boundaries, find PsO criteria + utility sections

Three strategies for Phase 2: heading match, table search, cross-reference following.
No LLM calls. Pure algorithmic document understanding.
"""
import re
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument

# === BRAND ALIASES ===
BRAND_ALIASES = {
    "TREMFYA": ["tremfya", "guselkumab"],
    "STELARA": ["stelara", "ustekinumab"],
    "ENBREL": ["enbrel", "etanercept"],
    "AMJEVITA": ["amjevita", "adalimumab-atto", "adalimumab"],
    "COSENTYX": ["cosentyx", "secukinumab"],
    "SKYRIZI": ["skyrizi", "risankizumab"],
    "OTEZLA": ["otezla", "apremilast"],
    "SILIQ": ["siliq", "brodalumab"],
    "CIMZIA": ["cimzia", "certolizumab"],
    "BIMZELX": ["bimzelx", "bimekizumab"],
    "REMICADE": ["remicade", "infliximab"],
    "OTULFI": ["otulfi", "ustekinumab-aauz"],
    "ILUMYA": ["ilumya", "tildrakizumab"],
    "YESINTEK": ["yesintek", "ustekinumab-kfce"],
    "ACITRETIN": ["acitretin"],
}

PSO_PATTERNS = [
    re.compile(r'\bplaque\s+psoriasis\b', re.IGNORECASE),
    re.compile(r'\bPsO\b'),
    re.compile(r'\bpsoriasis\b', re.IGNORECASE),
]

UTILITY_PATTERNS = [
    re.compile(r'\bduration\s+of\s+approval\b', re.IGNORECASE),
    re.compile(r'\bquantity\s+limit', re.IGNORECASE),
    re.compile(r'\breauthorization\b', re.IGNORECASE),
    re.compile(r'\brenewal\s+criteria\b', re.IGNORECASE),
    re.compile(r'\binitial\s+auth', re.IGNORECASE),
    re.compile(r'\binitial\s+approval\b', re.IGNORECASE),
    re.compile(r'\bdenial\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bapproval\s+criteria\b', re.IGNORECASE),
]

# Structural sub-headers that repeat in every section — NOT drug-class boundaries
STRUCTURAL_HEADERS = re.compile(
    r'^\s*('
    r'Products?\s+Affected|'
    r'Goal\(?s?\)?:?|'
    r'Length\s+of\s+Authorization:?|'
    r'Requires\s+PA:?|'
    r'Covered\s+Alternatives:?|'
    r'Approval\s+Criteria|'
    r'Renewal\s+Criteria|'
    r'References?:?|'
    r'Table\s+\d|'
    r'PA\s+Criteria|'
    r'P&T|'
    r'Effective\s+Date|'
    r'Notes?:?'
    r')\s*$',
    re.IGNORECASE
)

GOTO_PATTERN = re.compile(r'Go\s+to\s+#\s*(\d+)', re.IGNORECASE)
CRITERIA_NUM_PATTERN = re.compile(r'^\s*(\d+)\.\s')


# === DOCUMENT INDEX ===

class DocumentIndex:
    def __init__(self, doc: DoclingDocument):
        self.doc = doc
        self.items = list(doc.iterate_items())
        self.headers = []
        self.tables = []
        self.text_items = []
        self._build_index()

    def _build_index(self):
        table_counter = 0
        for pos, (item, level) in enumerate(self.items):
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                self.headers.append((pos, text, level))
            elif item_type == 'TableItem':
                self.tables.append((pos, table_counter, item))
                table_counter += 1
            if text.strip():
                self.text_items.append((pos, text, level, item_type))

    def get_table_df(self, table_item):
        try:
            return table_item.export_to_dataframe(self.doc)
        except:
            try:
                return table_item.export_to_dataframe()
            except:
                return None

    def get_text_between(self, start, end):
        parts = []
        for pos, (item, level) in enumerate(self.items):
            if pos < start:
                continue
            if pos >= end:
                break
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                parts.append(f"\n## {text}\n")
            elif item_type == 'TableItem':
                df = self.get_table_df(item)
                if df is not None:
                    parts.append(df.to_markdown(index=False))
                    parts.append("")
            elif item_type == 'ListItem':
                parts.append(f"- {text}")
            elif text.strip():
                parts.append(text)
        return "\n".join(parts)

    def is_structural_header(self, text):
        return bool(STRUCTURAL_HEADERS.match(text.strip()))


# === PHASE 1: FIND BRAND SECTION ===

def find_brand_section(index: DocumentIndex, brand: str) -> tuple:
    """
    Find the document region that contains PA criteria for this brand.
    Returns (start_pos, end_pos). For small docs, returns (0, total_items).
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    total = len(index.items)

    # Small documents: entire document is the section
    if total < 200:
        return (0, total)

    # Strategy 1: Find brand in a section header
    brand_header_pos = None
    for pos, text, level in index.headers:
        text_lower = text.lower().strip()
        if any(a == text_lower or a in text_lower.split() for a in aliases):
            # Verify it's a drug-name header, not a structural one
            if not index.is_structural_header(text):
                brand_header_pos = pos
                break

    if brand_header_pos is not None:
        # Find the end: walk forward to the next non-structural header at same level
        start = brand_header_pos
        brand_level = None
        for pos, text, level in index.headers:
            if pos == brand_header_pos:
                brand_level = level
                continue
            if brand_level is not None and pos > brand_header_pos:
                if level <= brand_level and not index.is_structural_header(text):
                    return (start, pos)
        return (start, total)

    # Strategy 2: Find brand in a table cell, then use adaptive scoping
    for pos, table_idx, table_item in index.tables:
        df = index.get_table_df(table_item)
        if df is None:
            continue
        df_str = df.to_string().lower()
        if any(a in df_str for a in aliases):
            return find_section_scope(index, pos)

    # Strategy 3: Find brand in any text item
    for pos, text, level, item_type in index.text_items:
        if any(a in text.lower() for a in aliases):
            # Use table proximity if tables exist nearby
            nearby_tables = [(tp, ti, tt) for tp, ti, tt in index.tables
                             if abs(tp - pos) < 50]
            if nearby_tables:
                return find_section_scope(index, nearby_tables[0][0])
            # Otherwise take a generous window
            return (max(0, pos - 50), min(total, pos + 200))

    # Fallback: entire document
    print(f"  WARNING: Brand '{brand}' not found. Using full document.")
    return (0, total)


def find_section_scope(index: DocumentIndex, anchor_table_pos: int) -> tuple:
    """Adaptive gap-based section scoping around an anchor table."""
    table_positions = [pos for pos, _, _ in index.tables]
    if not table_positions:
        return (0, len(index.items))

    anchor_idx = None
    for i, (pos, _, _) in enumerate(index.tables):
        if pos == anchor_table_pos:
            anchor_idx = i
            break
    if anchor_idx is None:
        return (0, len(index.items))

    # Seed: 10 tables each direction
    seed_start = max(0, anchor_idx - 10)
    seed_end = min(len(table_positions), anchor_idx + 11)
    seed_positions = table_positions[seed_start:seed_end]

    gaps = [seed_positions[i] - seed_positions[i - 1] for i in range(1, len(seed_positions))]
    if not gaps:
        return (0, len(index.items))

    sorted_gaps = sorted(gaps)
    median_gap = sorted_gaps[len(sorted_gaps) // 2]
    initial_threshold = max(median_gap * 3, 10)

    # Walk backward
    start_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, 0, -1):
        if table_positions[i] - table_positions[i - 1] > initial_threshold:
            break
        start_pos = table_positions[i - 1]

    # Walk forward
    end_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, len(table_positions) - 1):
        if table_positions[i + 1] - table_positions[i] > initial_threshold:
            break
        end_pos = table_positions[i + 1]

    # Refine with 2× largest intra-cluster gap
    cluster = [p for p in table_positions if start_pos <= p <= end_pos]
    if len(cluster) > 1:
        intra_gaps = [cluster[i + 1] - cluster[i] for i in range(len(cluster) - 1)]
        refined = max(max(intra_gaps) * 2, 10)
        start_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, 0, -1):
            if table_positions[i] - table_positions[i - 1] > refined:
                break
            start_pos = table_positions[i - 1]
        end_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, len(table_positions) - 1):
            if table_positions[i + 1] - table_positions[i] > refined:
                break
            end_pos = table_positions[i + 1]

    # Extend to capture context
    end_pos = min(end_pos + 10, len(index.items))
    for pos, text, level in reversed(index.headers):
        if pos < start_pos and start_pos - pos < 30:
            start_pos = pos
            break

    return (start_pos, end_pos)


# === PHASE 2: FIND PsO CONTENT WITHIN SECTION ===

def find_pso_by_headings(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy A: Find PsO-related headings within the scoped section."""
    collected = []
    scoped_headers = [(pos, text, level) for pos, text, level in index.headers
                      if scope_start <= pos < scope_end]

    for i, (pos, text, level) in enumerate(scoped_headers):
        is_pso = any(p.search(text) for p in PSO_PATTERNS)
        is_utility = any(p.search(text) for p in UTILITY_PATTERNS)

        if is_pso or is_utility:
            # End at next header at same or higher level within scope
            end_pos = scope_end
            for j in range(i + 1, len(scoped_headers)):
                next_pos, next_text, next_level = scoped_headers[j]
                if next_level <= level and not index.is_structural_header(next_text):
                    end_pos = next_pos
                    break
            collected.append((pos, end_pos))

    return collected


def find_pso_in_tables(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy B: Find tables containing PsO keywords within the scoped section."""
    collected = []
    for pos, table_idx, table_item in index.tables:
        if pos < scope_start or pos >= scope_end:
            continue
        df = index.get_table_df(table_item)
        if df is None:
            continue
        df_str = df.to_string().lower()
        if any(p.search(df_str) for p in PSO_PATTERNS):
            collected.append((max(scope_start, pos - 2), min(scope_end, pos + 2)))
    return collected


def follow_cross_references(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy C: Follow 'Go to #X' routing chains within the scoped section."""
    collected = []

    # Get all tables in scope
    scope_tables = []
    for pos, table_idx, table_item in index.tables:
        if scope_start <= pos <= scope_end:
            df = index.get_table_df(table_item)
            if df is not None:
                scope_tables.append((pos, table_idx, df))

    if not scope_tables:
        return collected

    # Find PsO routing entries: tables with both PsO keyword and "Go to #X"
    refs_to_follow = set()
    initial_tables = set()
    for tbl_pos, tbl_idx, tbl_df in scope_tables:
        for row_idx in range(len(tbl_df)):
            row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
            row_lower = row_text.lower()
            has_pso = any(p.search(row_lower) for p in PSO_PATTERNS)
            refs_in_row = GOTO_PATTERN.findall(row_text)
            if has_pso and refs_in_row:
                initial_tables.add(tbl_pos)
                refs_to_follow.update(int(r) for r in refs_in_row)

    if not refs_to_follow:
        return collected

    # BFS: follow references
    visited = set()
    tables_to_include = set(initial_tables)
    queue = list(refs_to_follow)

    while queue:
        target_num = queue.pop(0)
        if target_num in visited:
            continue
        visited.add(target_num)

        for tbl_pos, tbl_idx, tbl_df in scope_tables:
            for row_idx in range(len(tbl_df)):
                cell_0 = str(tbl_df.iloc[row_idx, 0]).strip()

                # Match criteria number in first cell
                match_bare = re.match(r'^(\d+)\.?\s*$', cell_0)
                match_prefix = CRITERIA_NUM_PATTERN.match(cell_0)

                found_num = None
                if match_bare:
                    found_num = int(match_bare.group(1))
                elif match_prefix:
                    found_num = int(match_prefix.group(1))

                # Also check column 1 for 4-column tables where col 0 is just the number
                if found_num is None and tbl_df.shape[1] >= 3:
                    cell_1 = str(tbl_df.iloc[row_idx, 1] if tbl_df.shape[1] > 1 else '').strip()
                    match_c1 = CRITERIA_NUM_PATTERN.match(cell_1)
                    if match_c1:
                        found_num = int(match_c1.group(1))

                if found_num == target_num:
                    tables_to_include.add(tbl_pos)
                    row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
                    new_refs = GOTO_PATTERN.findall(row_text)
                    for r in new_refs:
                        r_int = int(r)
                        if r_int not in visited:
                            queue.append(r_int)

    for tbl_pos in tables_to_include:
        collected.append((max(scope_start, tbl_pos - 2), min(scope_end, tbl_pos + 2)))

    return collected


# === MERGE AND SERIALIZE ===

def merge_ranges(ranges):
    if not ranges:
        return []
    sorted_r = sorted(ranges, key=lambda x: x[0])
    merged = [sorted_r[0]]
    for start, end in sorted_r[1:]:
        ps, pe = merged[-1]
        if start <= pe + 3:
            merged[-1] = (ps, max(pe, end))
        else:
            merged.append((start, end))
    return merged


def serialize(index, ranges):
    parts = []
    seen = set()
    for start, end in ranges:
        if start not in seen:
            parts.append(index.get_text_between(start, end))
            seen.add(start)
    return "\n\n---\n\n".join(parts)


# === MAIN REDUCER ===

def reduce_document(doc: DoclingDocument, brand: str) -> str:
    index = DocumentIndex(doc)
    print(f"  Reducer: {len(index.items)} items, {len(index.tables)} tables, {len(index.headers)} headers")

    # Phase 1: Find brand's section
    scope_start, scope_end = find_brand_section(index, brand)
    print(f"  Brand section: items {scope_start}-{scope_end} ({scope_end - scope_start} items)")

    # Phase 2: Find PsO content within section
    ranges_a = find_pso_by_headings(index, scope_start, scope_end, brand)
    ranges_b = find_pso_in_tables(index, scope_start, scope_end, brand)
    ranges_c = follow_cross_references(index, scope_start, scope_end, brand)
    print(f"  Strategy A (headings): {len(ranges_a)} ranges")
    print(f"  Strategy B (tables): {len(ranges_b)} ranges")
    print(f"  Strategy C (cross-ref): {len(ranges_c)} ranges")

    # If no PsO-specific content found, include the entire brand section
    all_ranges = ranges_a + ranges_b + ranges_c
    if not all_ranges:
        print(f"  No PsO-specific content found. Including full brand section.")
        all_ranges = [(scope_start, scope_end)]

    merged = merge_ranges(all_ranges)
    print(f"  Merged: {len(merged)} ranges")

    result = serialize(index, merged)

    full_len = len(doc.export_to_markdown())
    reduction = (1 - len(result) / full_len) * 100 if full_len else 0
    print(f"  Output: {len(result):,} chars ({reduction:.0f}% reduction from {full_len:,})")

    return result

In [ ]:
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument

# Paste the full reducer v2 code above this cell, then run:

OBJ_DIR = "work/doc_objects"
OUT_DIR = Path("reduced_v2")
OUT_DIR.mkdir(exist_ok=True)

test_cases = [
    ("56061-4538520", "STELARA", "Simple single-drug policy"),
    ("283789-5060638", "ENBREL", "Formulary table"),
    ("66156-4274314", "CIMZIA", "Indication-organized flowchart"),
]

for stem, brand, label in test_cases:
    obj_path = Path(OBJ_DIR) / f"{stem}.json"
    if not obj_path.exists():
        print(f"SKIP: {obj_path}")
        continue

    print(f"\n{'='*70}")
    print(f"{label}: {stem}.pdf | {brand}")
    print(f"{'='*70}")

    doc = DoclingDocument.load_from_json(obj_path)
    result = reduce_document(doc, brand)

    out_path = OUT_DIR / f"{stem}__{brand}.txt"
    out_path.write_text(result, encoding="utf-8")
    print(f"  Saved: {out_path}")

    print(f"\n  --- Preview (first 800 chars) ---")
    print(result[:800])
    print(f"  --- End preview ---")

print(f"\nOutputs in {OUT_DIR}/")

In [ ]:
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument
import openpyxl

OBJ_DIR = "work/doc_objects"
EXCEL_PATH = "PA_Business_Rules.xlsx"
OUT_DIR = Path("reduced_v2")
OUT_DIR.mkdir(exist_ok=True)

# Load submissions
wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
ws = wb['Submissions']
submissions = []
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
    if row[0] and row[1]:
        submissions.append((str(row[0]).strip(), str(row[1]).strip()))
wb.close()
print(f"Loaded {len(submissions)} submissions")

# Cache loaded documents
doc_cache = {}
stats = {"success": 0, "fallback": 0, "error": 0}

for i, (filename, brand) in enumerate(submissions):
    stem = Path(filename).stem
    print(f"\n[{i+1}/{len(submissions)}] {filename} | {brand}")

    if stem not in doc_cache:
        obj_path = Path(OBJ_DIR) / f"{stem}.json"
        if not obj_path.exists():
            print(f"  ERROR: {obj_path} not found")
            stats["error"] += 1
            continue
        doc_cache[stem] = DoclingDocument.load_from_json(obj_path)

    doc = doc_cache[stem]
    result = reduce_document(doc, brand)

    out_path = OUT_DIR / f"{stem}__{brand}.txt"
    out_path.write_text(result, encoding="utf-8")

    if len(result) < 50:
        stats["fallback"] += 1
        print(f"  ⚠️ SUSPICIOUSLY SHORT: {len(result)} chars")
    else:
        stats["success"] += 1

print(f"\n{'='*60}")
print(f"Done. Success: {stats['success']}, Suspicious: {stats['fallback']}, Error: {stats['error']}")
print(f"Outputs in {OUT_DIR}/")

In [ ]:
"""
Document Reducer v2 — Extracts PsO-relevant sections from Docling structured documents.

Architecture:
  Phase 1: Find the brand's section boundaries
  Phase 2: Within those boundaries, find PsO criteria + utility sections

Three strategies for Phase 2: heading match, table search, cross-reference following.
No LLM calls. Pure algorithmic document understanding.
"""
import re
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument

# === BRAND ALIASES ===
BRAND_ALIASES = {
    "TREMFYA": ["tremfya", "guselkumab"],
    "STELARA": ["stelara", "ustekinumab"],
    "ENBREL": ["enbrel", "etanercept"],
    "AMJEVITA": ["amjevita", "adalimumab-atto", "adalimumab"],
    "COSENTYX": ["cosentyx", "secukinumab"],
    "SKYRIZI": ["skyrizi", "risankizumab"],
    "OTEZLA": ["otezla", "apremilast"],
    "SILIQ": ["siliq", "brodalumab"],
    "CIMZIA": ["cimzia", "certolizumab"],
    "BIMZELX": ["bimzelx", "bimekizumab"],
    "REMICADE": ["remicade", "infliximab"],
    "OTULFI": ["otulfi", "ustekinumab-aauz"],
    "ILUMYA": ["ilumya", "tildrakizumab"],
    "YESINTEK": ["yesintek", "ustekinumab-kfce"],
    "ACITRETIN": ["acitretin"],
}

PSO_PATTERNS = [
    re.compile(r'\bplaque\s+psoriasis\b', re.IGNORECASE),
    re.compile(r'\bPsO\b'),
    re.compile(r'\bpsoriasis\b', re.IGNORECASE),
]

UTILITY_PATTERNS = [
    re.compile(r'\bduration\s+of\s+approval\b', re.IGNORECASE),
    re.compile(r'\bquantity\s+limit', re.IGNORECASE),
    re.compile(r'\breauthorization\b', re.IGNORECASE),
    re.compile(r'\brenewal\s+criteria\b', re.IGNORECASE),
    re.compile(r'\binitial\s+auth', re.IGNORECASE),
    re.compile(r'\binitial\s+approval\b', re.IGNORECASE),
    re.compile(r'\bdenial\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bapproval\s+criteria\b', re.IGNORECASE),
]

# Structural sub-headers that repeat in every section — NOT drug-class boundaries
STRUCTURAL_HEADERS = re.compile(
    r'^\s*('
    r'Products?\s+Affected|'
    r'Goal\(?s?\)?:?|'
    r'Length\s+of\s+Authorization:?|'
    r'Requires\s+PA:?|'
    r'Covered\s+Alternatives:?|'
    r'Approval\s+Criteria|'
    r'Renewal\s+Criteria|'
    r'References?:?|'
    r'Table\s+\d|'
    r'PA\s+Criteria|'
    r'P&T|'
    r'Effective\s+Date|'
    r'Notes?:?'
    r')\s*$',
    re.IGNORECASE
)

GOTO_PATTERN = re.compile(r'Go\s+to\s+#\s*(\d+)', re.IGNORECASE)
CRITERIA_NUM_PATTERN = re.compile(r'^\s*(\d+)\.\s')


# === DOCUMENT INDEX ===

class DocumentIndex:
    def __init__(self, doc: DoclingDocument):
        self.doc = doc
        self.items = list(doc.iterate_items())
        self.headers = []
        self.tables = []
        self.text_items = []
        self._build_index()

    def _build_index(self):
        table_counter = 0
        for pos, (item, level) in enumerate(self.items):
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                self.headers.append((pos, text, level))
            elif item_type == 'TableItem':
                self.tables.append((pos, table_counter, item))
                table_counter += 1
            if text.strip():
                self.text_items.append((pos, text, level, item_type))

    def get_table_df(self, table_item):
        try:
            return table_item.export_to_dataframe(self.doc)
        except:
            try:
                return table_item.export_to_dataframe()
            except:
                return None

    def get_text_between(self, start, end):
        parts = []
        for pos, (item, level) in enumerate(self.items):
            if pos < start:
                continue
            if pos >= end:
                break
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                parts.append(f"\n## {text}\n")
            elif item_type == 'TableItem':
                df = self.get_table_df(item)
                if df is not None:
                    parts.append(df.to_markdown(index=False))
                    parts.append("")
            elif item_type == 'ListItem':
                parts.append(f"- {text}")
            elif text.strip():
                parts.append(text)
        return "\n".join(parts)

    def is_structural_header(self, text):
        return bool(STRUCTURAL_HEADERS.match(text.strip()))


# === PHASE 1: FIND BRAND SECTION ===

def find_brand_section(index: DocumentIndex, brand: str) -> tuple:
    """
    Find the document region that contains PA criteria for this brand.
    Returns (start_pos, end_pos). For small docs, returns (0, total_items).
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    total = len(index.items)

    # Small documents: entire document is the section
    if total < 200:
        return (0, total)

    # Strategy 1: Find brand in a section header
    brand_header_pos = None
    for pos, text, level in index.headers:
        text_lower = text.lower().strip()
        if any(a == text_lower or a in text_lower.split() for a in aliases):
            # Verify it's a drug-name header, not a structural one
            if not index.is_structural_header(text):
                brand_header_pos = pos
                break

    if brand_header_pos is not None:
        # Find the end: walk forward to the next non-structural header at same level
        start = brand_header_pos
        brand_level = None
        for pos, text, level in index.headers:
            if pos == brand_header_pos:
                brand_level = level
                continue
            if brand_level is not None and pos > brand_header_pos:
                if level <= brand_level and not index.is_structural_header(text):
                    return (start, pos)
        return (start, total)

    # Strategy 2: Find brand in a table cell, then use adaptive scoping
    for pos, table_idx, table_item in index.tables:
        df = index.get_table_df(table_item)
        if df is None:
            continue
        df_str = df.to_string().lower()
        if any(a in df_str for a in aliases):
            return find_section_scope(index, pos)

    # Strategy 3: Find brand in any text item
    for pos, text, level, item_type in index.text_items:
        if any(a in text.lower() for a in aliases):
            # Use table proximity if tables exist nearby
            nearby_tables = [(tp, ti, tt) for tp, ti, tt in index.tables
                             if abs(tp - pos) < 50]
            if nearby_tables:
                return find_section_scope(index, nearby_tables[0][0])
            # Otherwise take a generous window
            return (max(0, pos - 50), min(total, pos + 200))

    # Fallback: entire document
    print(f"  WARNING: Brand '{brand}' not found. Using full document.")
    return (0, total)


def find_section_scope(index: DocumentIndex, anchor_table_pos: int) -> tuple:
    """Adaptive gap-based section scoping around an anchor table."""
    table_positions = [pos for pos, _, _ in index.tables]
    if not table_positions:
        return (0, len(index.items))

    anchor_idx = None
    for i, (pos, _, _) in enumerate(index.tables):
        if pos == anchor_table_pos:
            anchor_idx = i
            break
    if anchor_idx is None:
        return (0, len(index.items))

    # Seed: 10 tables each direction
    seed_start = max(0, anchor_idx - 10)
    seed_end = min(len(table_positions), anchor_idx + 11)
    seed_positions = table_positions[seed_start:seed_end]

    gaps = [seed_positions[i] - seed_positions[i - 1] for i in range(1, len(seed_positions))]
    if not gaps:
        return (0, len(index.items))

    sorted_gaps = sorted(gaps)
    median_gap = sorted_gaps[len(sorted_gaps) // 2]
    initial_threshold = max(median_gap * 3, 10)

    # Walk backward
    start_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, 0, -1):
        if table_positions[i] - table_positions[i - 1] > initial_threshold:
            break
        start_pos = table_positions[i - 1]

    # Walk forward
    end_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, len(table_positions) - 1):
        if table_positions[i + 1] - table_positions[i] > initial_threshold:
            break
        end_pos = table_positions[i + 1]

    # Refine with 2× largest intra-cluster gap
    cluster = [p for p in table_positions if start_pos <= p <= end_pos]
    if len(cluster) > 1:
        intra_gaps = [cluster[i + 1] - cluster[i] for i in range(len(cluster) - 1)]
        refined = max(max(intra_gaps) * 2, 10)
        start_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, 0, -1):
            if table_positions[i] - table_positions[i - 1] > refined:
                break
            start_pos = table_positions[i - 1]
        end_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, len(table_positions) - 1):
            if table_positions[i + 1] - table_positions[i] > refined:
                break
            end_pos = table_positions[i + 1]

    # Extend to capture context
    end_pos = min(end_pos + 10, len(index.items))
    for pos, text, level in reversed(index.headers):
        if pos < start_pos and start_pos - pos < 30:
            start_pos = pos
            break

    return (start_pos, end_pos)


# === PHASE 2: FIND PsO CONTENT WITHIN SECTION ===

def find_pso_by_headings(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy A: Find PsO-related headings within the scoped section."""
    collected = []
    scoped_headers = [(pos, text, level) for pos, text, level in index.headers
                      if scope_start <= pos < scope_end]

    for i, (pos, text, level) in enumerate(scoped_headers):
        is_pso = any(p.search(text) for p in PSO_PATTERNS)
        is_utility = any(p.search(text) for p in UTILITY_PATTERNS)

        if is_pso or is_utility:
            # End at next header at same or higher level within scope
            end_pos = scope_end
            for j in range(i + 1, len(scoped_headers)):
                next_pos, next_text, next_level = scoped_headers[j]
                if next_level <= level and not index.is_structural_header(next_text):
                    end_pos = next_pos
                    break
            collected.append((pos, end_pos))

    return collected


def find_pso_in_tables(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy B: Find tables containing PsO keywords within the scoped section.
    
    For small tables (<20 rows): include if PsO keyword appears anywhere.
    For large tables (>=20 rows): only include specific rows that mention
    both PsO keywords and criteria-related content. This prevents
    massive formulary listing tables from flooding the output.
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    collected = []

    # Column names that indicate a PA criteria table (not a formulary listing)
    CRITERIA_TABLE_PATTERNS = re.compile(
        r'PA\s+Criteria|Criteria\s+Details|Approval|Required\s+Medical|'
        r'Coverage\s+Duration|Renewal|Exclusion|Prescriber|Restriction|'
        r'Authorization',
        re.IGNORECASE
    )

    for pos, table_idx, table_item in index.tables:
        if pos < scope_start or pos >= scope_end:
            continue
        df = index.get_table_df(table_item)
        if df is None:
            continue

        num_rows = len(df)
        col_str = ' '.join(str(c) for c in df.columns)

        # Check if this is a PA criteria table by column names
        is_criteria_table = bool(CRITERIA_TABLE_PATTERNS.search(col_str))

        # Small table or criteria-structured table: include if PsO keyword found
        if num_rows < 20 or is_criteria_table:
            df_str = df.to_string().lower()
            if any(p.search(df_str) for p in PSO_PATTERNS):
                collected.append((max(scope_start, pos - 2), min(scope_end, pos + 2)))

        # Large table: only include if a row has BOTH brand/PsO AND criteria content
        else:
            df_str = df.to_string().lower()
            has_brand = any(a in df_str for a in aliases)
            has_pso = any(p.search(df_str) for p in PSO_PATTERNS)
            if has_brand and has_pso:
                # Include but mark as large — the serializer will handle row extraction
                collected.append((max(scope_start, pos - 2), min(scope_end, pos + 2)))

    return collected


def follow_cross_references(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy C: Follow 'Go to #X' routing chains within the scoped section."""
    collected = []

    # Get all tables in scope
    scope_tables = []
    for pos, table_idx, table_item in index.tables:
        if scope_start <= pos <= scope_end:
            df = index.get_table_df(table_item)
            if df is not None:
                scope_tables.append((pos, table_idx, df))

    if not scope_tables:
        return collected

    # Find PsO routing entries: tables with both PsO keyword and "Go to #X"
    refs_to_follow = set()
    initial_tables = set()
    for tbl_pos, tbl_idx, tbl_df in scope_tables:
        for row_idx in range(len(tbl_df)):
            row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
            row_lower = row_text.lower()
            has_pso = any(p.search(row_lower) for p in PSO_PATTERNS)
            refs_in_row = GOTO_PATTERN.findall(row_text)
            if has_pso and refs_in_row:
                initial_tables.add(tbl_pos)
                refs_to_follow.update(int(r) for r in refs_in_row)

    if not refs_to_follow:
        return collected

    # BFS: follow references
    visited = set()
    tables_to_include = set(initial_tables)
    queue = list(refs_to_follow)

    while queue:
        target_num = queue.pop(0)
        if target_num in visited:
            continue
        visited.add(target_num)

        for tbl_pos, tbl_idx, tbl_df in scope_tables:
            for row_idx in range(len(tbl_df)):
                cell_0 = str(tbl_df.iloc[row_idx, 0]).strip()

                # Match criteria number in first cell
                match_bare = re.match(r'^(\d+)\.?\s*$', cell_0)
                match_prefix = CRITERIA_NUM_PATTERN.match(cell_0)

                found_num = None
                if match_bare:
                    found_num = int(match_bare.group(1))
                elif match_prefix:
                    found_num = int(match_prefix.group(1))

                # Also check column 1 for 4-column tables where col 0 is just the number
                if found_num is None and tbl_df.shape[1] >= 3:
                    cell_1 = str(tbl_df.iloc[row_idx, 1] if tbl_df.shape[1] > 1 else '').strip()
                    match_c1 = CRITERIA_NUM_PATTERN.match(cell_1)
                    if match_c1:
                        found_num = int(match_c1.group(1))

                if found_num == target_num:
                    tables_to_include.add(tbl_pos)
                    row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
                    new_refs = GOTO_PATTERN.findall(row_text)
                    for r in new_refs:
                        r_int = int(r)
                        if r_int not in visited:
                            queue.append(r_int)

    for tbl_pos in tables_to_include:
        collected.append((max(scope_start, tbl_pos - 2), min(scope_end, tbl_pos + 2)))

    return collected


# === MERGE AND SERIALIZE ===

def merge_ranges(ranges):
    if not ranges:
        return []
    sorted_r = sorted(ranges, key=lambda x: x[0])
    merged = [sorted_r[0]]
    for start, end in sorted_r[1:]:
        ps, pe = merged[-1]
        if start <= pe + 3:
            merged[-1] = (ps, max(pe, end))
        else:
            merged.append((start, end))
    return merged


def serialize(index, ranges):
    parts = []
    seen = set()
    for start, end in ranges:
        if start not in seen:
            parts.append(index.get_text_between(start, end))
            seen.add(start)
    return "\n\n---\n\n".join(parts)


# === MAIN REDUCER ===

MIN_OUTPUT_CHARS = 100  # Below this, output is suspiciously short

def reduce_document(doc: DoclingDocument, brand: str) -> str:
    index = DocumentIndex(doc)
    total_items = len(index.items)
    print(f"  Reducer: {total_items} items, {len(index.tables)} tables, {len(index.headers)} headers")

    # Phase 1: Find brand's section
    scope_start, scope_end = find_brand_section(index, brand)
    scope_size = scope_end - scope_start
    print(f"  Brand section: items {scope_start}-{scope_end} ({scope_size} items)")

    # Confidence check: if scope is suspiciously narrow for a large document,
    # the brand was found in passing (not in a criteria section).
    # Widen to full document and let Phase 2 search everywhere.
    if scope_size < 10 and total_items > 200:
        print(f"  Scope too narrow ({scope_size} items in {total_items}-item doc). Widening to full document.")
        scope_start, scope_end = 0, total_items

    # Phase 2: Find PsO content within section
    ranges_a = find_pso_by_headings(index, scope_start, scope_end, brand)
    ranges_b = find_pso_in_tables(index, scope_start, scope_end, brand)
    ranges_c = follow_cross_references(index, scope_start, scope_end, brand)
    print(f"  Strategy A (headings): {len(ranges_a)} ranges")
    print(f"  Strategy B (tables): {len(ranges_b)} ranges")
    print(f"  Strategy C (cross-ref): {len(ranges_c)} ranges")

    # If no PsO-specific content found, include the entire brand section
    all_ranges = ranges_a + ranges_b + ranges_c
    if not all_ranges:
        print(f"  No PsO-specific content found. Including full brand section.")
        all_ranges = [(scope_start, scope_end)]

    merged = merge_ranges(all_ranges)
    print(f"  Merged: {len(merged)} ranges")

    result = serialize(index, merged)

    # Recovery: if output is still too short, return full markdown
    if len(result) < MIN_OUTPUT_CHARS and total_items > 50:
        print(f"  Output too short ({len(result)} chars). Falling back to full markdown.")
        result = doc.export_to_markdown()

    full_len = len(doc.export_to_markdown())
    reduction = (1 - len(result) / full_len) * 100 if full_len else 0
    print(f"  Output: {len(result):,} chars ({reduction:.0f}% reduction from {full_len:,})")

    return result

In [ ]:
# Paste the v3 reducer code, then run:
OUT_DIR = Path("work/reduced")
OUT_DIR.mkdir(exist_ok=True)

from pathlib import Path
from docling_core.types.doc.document import DoclingDocument
import openpyxl

OBJ_DIR = "work/doc_objects"
EXCEL_PATH = "PA_Business_Rules.xlsx"
OUT_DIR = Path("reduced_v2")
OUT_DIR.mkdir(exist_ok=True)

# Load submissions
wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
ws = wb['Submissions']
submissions = []
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
    if row[0] and row[1]:
        submissions.append((str(row[0]).strip(), str(row[1]).strip()))
wb.close()
print(f"Loaded {len(submissions)} submissions")

# Cache loaded documents
doc_cache = {}
stats = {"success": 0, "fallback": 0, "error": 0}

for i, (filename, brand) in enumerate(submissions):
    stem = Path(filename).stem
    print(f"\n[{i+1}/{len(submissions)}] {filename} | {brand}")

    if stem not in doc_cache:
        obj_path = Path(OBJ_DIR) / f"{stem}.json"
        if not obj_path.exists():
            print(f"  ERROR: {obj_path} not found")
            stats["error"] += 1
            continue
        doc_cache[stem] = DoclingDocument.load_from_json(obj_path)

    doc = doc_cache[stem]
    result = reduce_document(doc, brand)

    out_path = OUT_DIR / f"{stem}__{brand}.txt"
    out_path.write_text(result, encoding="utf-8")

    if len(result) < 50:
        stats["fallback"] += 1
        print(f"  ⚠️ SUSPICIOUSLY SHORT: {len(result)} chars")
    else:
        stats["success"] += 1

print(f"\n{'='*60}")
print(f"Done. Success: {stats['success']}, Suspicious: {stats['fallback']}, Error: {stats['error']}")
print(f"Outputs in {OUT_DIR}/")

In [ ]:
"""
Document Reducer v2 — Extracts PsO-relevant sections from Docling structured documents.

Architecture:
  Phase 1: Find the brand's section boundaries
  Phase 2: Within those boundaries, find PsO criteria + utility sections

Three strategies for Phase 2: heading match, table search, cross-reference following.
No LLM calls. Pure algorithmic document understanding.
"""
import re
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument

# === BRAND ALIASES ===
BRAND_ALIASES = {
    "TREMFYA": ["tremfya", "guselkumab"],
    "STELARA": ["stelara", "ustekinumab"],
    "ENBREL": ["enbrel", "etanercept"],
    "AMJEVITA": ["amjevita", "adalimumab-atto", "adalimumab"],
    "COSENTYX": ["cosentyx", "secukinumab"],
    "SKYRIZI": ["skyrizi", "risankizumab"],
    "OTEZLA": ["otezla", "apremilast"],
    "SILIQ": ["siliq", "brodalumab"],
    "CIMZIA": ["cimzia", "certolizumab"],
    "BIMZELX": ["bimzelx", "bimekizumab"],
    "REMICADE": ["remicade", "infliximab"],
    "OTULFI": ["otulfi", "ustekinumab-aauz"],
    "ILUMYA": ["ilumya", "tildrakizumab"],
    "YESINTEK": ["yesintek", "ustekinumab-kfce"],
    "ACITRETIN": ["acitretin"],
}

PSO_PATTERNS = [
    re.compile(r'\bplaque\s+psoriasis\b', re.IGNORECASE),
    re.compile(r'\bPsO\b'),
    re.compile(r'\bpsoriasis\b', re.IGNORECASE),
]

UTILITY_PATTERNS = [
    re.compile(r'\bduration\s+of\s+approval\b', re.IGNORECASE),
    re.compile(r'\bquantity\s+limit', re.IGNORECASE),
    re.compile(r'\breauthorization\b', re.IGNORECASE),
    re.compile(r'\brenewal\s+criteria\b', re.IGNORECASE),
    re.compile(r'\binitial\s+auth', re.IGNORECASE),
    re.compile(r'\binitial\s+approval\b', re.IGNORECASE),
    re.compile(r'\bdenial\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bapproval\s+criteria\b', re.IGNORECASE),
]

# Structural sub-headers that repeat in every section — NOT drug-class boundaries
STRUCTURAL_HEADERS = re.compile(
    r'^\s*('
    r'Products?\s+Affected|'
    r'Goal\(?s?\)?:?|'
    r'Length\s+of\s+Authorization:?|'
    r'Requires\s+PA:?|'
    r'Covered\s+Alternatives:?|'
    r'Approval\s+Criteria|'
    r'Renewal\s+Criteria|'
    r'References?:?|'
    r'Table\s+\d|'
    r'PA\s+Criteria|'
    r'P&T|'
    r'Effective\s+Date|'
    r'Notes?:?'
    r')\s*$',
    re.IGNORECASE
)

GOTO_PATTERN = re.compile(r'Go\s+to\s+#\s*(\d+)', re.IGNORECASE)
CRITERIA_NUM_PATTERN = re.compile(r'^\s*(\d+)\.\s')


# === DOCUMENT INDEX ===

class DocumentIndex:
    def __init__(self, doc: DoclingDocument):
        self.doc = doc
        self.items = list(doc.iterate_items())
        self.headers = []
        self.tables = []
        self.text_items = []
        self._build_index()

    def _build_index(self):
        table_counter = 0
        for pos, (item, level) in enumerate(self.items):
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                self.headers.append((pos, text, level))
            elif item_type == 'TableItem':
                self.tables.append((pos, table_counter, item))
                table_counter += 1
            if text.strip():
                self.text_items.append((pos, text, level, item_type))

    def get_table_df(self, table_item):
        try:
            return table_item.export_to_dataframe(self.doc)
        except:
            try:
                return table_item.export_to_dataframe()
            except:
                return None

    def get_text_between(self, start, end):
        parts = []
        for pos, (item, level) in enumerate(self.items):
            if pos < start:
                continue
            if pos >= end:
                break
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                parts.append(f"\n## {text}\n")
            elif item_type == 'TableItem':
                df = self.get_table_df(item)
                if df is not None:
                    parts.append(df.to_markdown(index=False))
                    parts.append("")
            elif item_type == 'ListItem':
                parts.append(f"- {text}")
            elif text.strip():
                parts.append(text)
        return "\n".join(parts)

    def is_structural_header(self, text):
        return bool(STRUCTURAL_HEADERS.match(text.strip()))


# === PHASE 1: FIND BRAND SECTION ===

def find_brand_section(index: DocumentIndex, brand: str) -> tuple:
    """
    Find the document region that contains PA criteria for this brand.
    Returns (start_pos, end_pos). For small docs, returns (0, total_items).
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    total = len(index.items)

    # Small documents: entire document is the section
    if total < 200:
        return (0, total)

    # Strategy 1: Find brand in a section header
    brand_header_pos = None
    for pos, text, level in index.headers:
        text_lower = text.lower().strip()
        if any(a == text_lower or a in text_lower.split() for a in aliases):
            # Verify it's a drug-name header, not a structural one
            if not index.is_structural_header(text):
                brand_header_pos = pos
                break

    if brand_header_pos is not None:
        # Find the end: walk forward to the next non-structural header at same level
        start = brand_header_pos
        brand_level = None
        for pos, text, level in index.headers:
            if pos == brand_header_pos:
                brand_level = level
                continue
            if brand_level is not None and pos > brand_header_pos:
                if level <= brand_level and not index.is_structural_header(text):
                    return (start, pos)
        return (start, total)

    # Strategy 2: Find brand in a table cell, then use adaptive scoping
    for pos, table_idx, table_item in index.tables:
        df = index.get_table_df(table_item)
        if df is None:
            continue
        df_str = df.to_string().lower()
        if any(a in df_str for a in aliases):
            return find_section_scope(index, pos)

    # Strategy 3: Find brand in any text item
    for pos, text, level, item_type in index.text_items:
        if any(a in text.lower() for a in aliases):
            # Use table proximity if tables exist nearby
            nearby_tables = [(tp, ti, tt) for tp, ti, tt in index.tables
                             if abs(tp - pos) < 50]
            if nearby_tables:
                return find_section_scope(index, nearby_tables[0][0])
            # Otherwise take a generous window
            return (max(0, pos - 50), min(total, pos + 200))

    # Fallback: entire document
    print(f"  WARNING: Brand '{brand}' not found. Using full document.")
    return (0, total)


def find_section_scope(index: DocumentIndex, anchor_table_pos: int) -> tuple:
    """Adaptive gap-based section scoping around an anchor table."""
    table_positions = [pos for pos, _, _ in index.tables]
    if not table_positions:
        return (0, len(index.items))

    anchor_idx = None
    for i, (pos, _, _) in enumerate(index.tables):
        if pos == anchor_table_pos:
            anchor_idx = i
            break
    if anchor_idx is None:
        return (0, len(index.items))

    # Seed: 10 tables each direction
    seed_start = max(0, anchor_idx - 10)
    seed_end = min(len(table_positions), anchor_idx + 11)
    seed_positions = table_positions[seed_start:seed_end]

    gaps = [seed_positions[i] - seed_positions[i - 1] for i in range(1, len(seed_positions))]
    if not gaps:
        return (0, len(index.items))

    sorted_gaps = sorted(gaps)
    median_gap = sorted_gaps[len(sorted_gaps) // 2]
    initial_threshold = max(median_gap * 3, 10)

    # Walk backward
    start_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, 0, -1):
        if table_positions[i] - table_positions[i - 1] > initial_threshold:
            break
        start_pos = table_positions[i - 1]

    # Walk forward
    end_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, len(table_positions) - 1):
        if table_positions[i + 1] - table_positions[i] > initial_threshold:
            break
        end_pos = table_positions[i + 1]

    # Refine with 2× largest intra-cluster gap
    cluster = [p for p in table_positions if start_pos <= p <= end_pos]
    if len(cluster) > 1:
        intra_gaps = [cluster[i + 1] - cluster[i] for i in range(len(cluster) - 1)]
        refined = max(max(intra_gaps) * 2, 10)
        start_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, 0, -1):
            if table_positions[i] - table_positions[i - 1] > refined:
                break
            start_pos = table_positions[i - 1]
        end_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, len(table_positions) - 1):
            if table_positions[i + 1] - table_positions[i] > refined:
                break
            end_pos = table_positions[i + 1]

    # Extend to capture context
    end_pos = min(end_pos + 10, len(index.items))
    for pos, text, level in reversed(index.headers):
        if pos < start_pos and start_pos - pos < 30:
            start_pos = pos
            break

    return (start_pos, end_pos)


# === PHASE 2: FIND PsO CONTENT WITHIN SECTION ===

def find_pso_by_headings(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy A: Find PsO-related headings within the scoped section."""
    collected = []
    scoped_headers = [(pos, text, level) for pos, text, level in index.headers
                      if scope_start <= pos < scope_end]

    for i, (pos, text, level) in enumerate(scoped_headers):
        is_pso = any(p.search(text) for p in PSO_PATTERNS)
        is_utility = any(p.search(text) for p in UTILITY_PATTERNS)

        if is_pso or is_utility:
            # End at next header at same or higher level within scope
            end_pos = scope_end
            for j in range(i + 1, len(scoped_headers)):
                next_pos, next_text, next_level = scoped_headers[j]
                if next_level <= level and not index.is_structural_header(next_text):
                    end_pos = next_pos
                    break
            collected.append((pos, end_pos))

    return collected


def find_pso_in_tables(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy B: Find tables containing PsO keywords within the scoped section.
    
    Filters:
    - Skip administrative tables (drug listings, NDC tables, dosing grids)
    - Small tables (<20 rows): include if PsO keyword appears anywhere
    - Criteria-structured tables: include if PsO keyword found
    - Large non-criteria tables: require both brand AND PsO keywords
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    collected = []

    # Column names that indicate a PA criteria table
    CRITERIA_TABLE_PATTERNS = re.compile(
        r'PA\s+Criteria|Criteria\s+Details|Approval|Required\s+Medical|'
        r'Coverage\s+Duration|Renewal|Exclusion|Prescriber|Restriction|'
        r'Authorization|Step\s+Therapy|Clinical\s+Criteria',
        re.IGNORECASE
    )

    # Column names that indicate an administrative/formulary listing table — skip these
    ADMIN_TABLE_PATTERNS = re.compile(
        r'Strength|NDC|Targeted\s+NDC|Day\s+Supply|Dose\s+Form|'
        r'Client\s+Formulary|GPI|HICL|Package\s+Size|Route\s+of\s+Admin|'
        r'AWP|WAC|Unit\s+Price|Wholesale|Generic\s+Name.*Brand\s+Name|'
        r'Brand\s+Name.*Generic\s+Name|HCPCS|J-?Code|CPT',
        re.IGNORECASE
    )

    for pos, table_idx, table_item in index.tables:
        if pos < scope_start or pos >= scope_end:
            continue
        df = index.get_table_df(table_item)
        if df is None:
            continue

        num_rows = len(df)
        col_str = ' '.join(str(c) for c in df.columns)

        # Skip administrative tables entirely
        if ADMIN_TABLE_PATTERNS.search(col_str):
            continue

        # Check if this is a PA criteria table by column names
        is_criteria_table = bool(CRITERIA_TABLE_PATTERNS.search(col_str))

        # Small table or criteria-structured table: include if PsO keyword found
        if num_rows < 20 or is_criteria_table:
            df_str = df.to_string().lower()
            if any(p.search(df_str) for p in PSO_PATTERNS):
                collected.append((max(scope_start, pos - 2), min(scope_end, pos + 2)))

        # Large table: only include if BOTH brand AND PsO found
        else:
            df_str = df.to_string().lower()
            has_brand = any(a in df_str for a in aliases)
            has_pso = any(p.search(df_str) for p in PSO_PATTERNS)
            if has_brand and has_pso:
                collected.append((max(scope_start, pos - 2), min(scope_end, pos + 2)))

    return collected


def follow_cross_references(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy C: Follow 'Go to #X' routing chains within the scoped section."""
    collected = []

    # Get all tables in scope
    scope_tables = []
    for pos, table_idx, table_item in index.tables:
        if scope_start <= pos <= scope_end:
            df = index.get_table_df(table_item)
            if df is not None:
                scope_tables.append((pos, table_idx, df))

    if not scope_tables:
        return collected

    # Find PsO routing entries: tables with both PsO keyword and "Go to #X"
    refs_to_follow = set()
    initial_tables = set()
    for tbl_pos, tbl_idx, tbl_df in scope_tables:
        for row_idx in range(len(tbl_df)):
            row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
            row_lower = row_text.lower()
            has_pso = any(p.search(row_lower) for p in PSO_PATTERNS)
            refs_in_row = GOTO_PATTERN.findall(row_text)
            if has_pso and refs_in_row:
                initial_tables.add(tbl_pos)
                refs_to_follow.update(int(r) for r in refs_in_row)

    if not refs_to_follow:
        return collected

    # BFS: follow references
    visited = set()
    tables_to_include = set(initial_tables)
    queue = list(refs_to_follow)

    while queue:
        target_num = queue.pop(0)
        if target_num in visited:
            continue
        visited.add(target_num)

        for tbl_pos, tbl_idx, tbl_df in scope_tables:
            for row_idx in range(len(tbl_df)):
                cell_0 = str(tbl_df.iloc[row_idx, 0]).strip()

                # Match criteria number in first cell
                match_bare = re.match(r'^(\d+)\.?\s*$', cell_0)
                match_prefix = CRITERIA_NUM_PATTERN.match(cell_0)

                found_num = None
                if match_bare:
                    found_num = int(match_bare.group(1))
                elif match_prefix:
                    found_num = int(match_prefix.group(1))

                # Also check column 1 for 4-column tables where col 0 is just the number
                if found_num is None and tbl_df.shape[1] >= 3:
                    cell_1 = str(tbl_df.iloc[row_idx, 1] if tbl_df.shape[1] > 1 else '').strip()
                    match_c1 = CRITERIA_NUM_PATTERN.match(cell_1)
                    if match_c1:
                        found_num = int(match_c1.group(1))

                if found_num == target_num:
                    tables_to_include.add(tbl_pos)
                    row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
                    new_refs = GOTO_PATTERN.findall(row_text)
                    for r in new_refs:
                        r_int = int(r)
                        if r_int not in visited:
                            queue.append(r_int)

    for tbl_pos in tables_to_include:
        collected.append((max(scope_start, tbl_pos - 2), min(scope_end, tbl_pos + 2)))

    return collected


# === MERGE AND SERIALIZE ===

def merge_ranges(ranges):
    if not ranges:
        return []
    sorted_r = sorted(ranges, key=lambda x: x[0])
    merged = [sorted_r[0]]
    for start, end in sorted_r[1:]:
        ps, pe = merged[-1]
        if start <= pe + 3:
            merged[-1] = (ps, max(pe, end))
        else:
            merged.append((start, end))
    return merged


def serialize(index, ranges):
    parts = []
    seen = set()
    for start, end in ranges:
        if start not in seen:
            parts.append(index.get_text_between(start, end))
            seen.add(start)
    return "\n\n---\n\n".join(parts)


# === MAIN REDUCER ===

MIN_OUTPUT_CHARS = 100  # Below this, output is suspiciously short

def reduce_document(doc: DoclingDocument, brand: str) -> str:
    index = DocumentIndex(doc)
    total_items = len(index.items)
    print(f"  Reducer: {total_items} items, {len(index.tables)} tables, {len(index.headers)} headers")

    # Phase 1: Find brand's section
    scope_start, scope_end = find_brand_section(index, brand)
    scope_size = scope_end - scope_start
    print(f"  Brand section: items {scope_start}-{scope_end} ({scope_size} items)")

    # Confidence check: if scope is suspiciously narrow for a large document,
    # the brand was found in passing (not in a criteria section).
    # Widen to full document and let Phase 2 search everywhere.
    if scope_size < 10 and total_items > 200:
        print(f"  Scope too narrow ({scope_size} items in {total_items}-item doc). Widening to full document.")
        scope_start, scope_end = 0, total_items

    # Phase 2: Find PsO content within section
    ranges_a = find_pso_by_headings(index, scope_start, scope_end, brand)
    ranges_b = find_pso_in_tables(index, scope_start, scope_end, brand)
    ranges_c = follow_cross_references(index, scope_start, scope_end, brand)
    print(f"  Strategy A (headings): {len(ranges_a)} ranges")
    print(f"  Strategy B (tables): {len(ranges_b)} ranges")
    print(f"  Strategy C (cross-ref): {len(ranges_c)} ranges")

    # If no PsO-specific content found, include the entire brand section
    all_ranges = ranges_a + ranges_b + ranges_c
    if not all_ranges:
        print(f"  No PsO-specific content found. Including full brand section.")
        all_ranges = [(scope_start, scope_end)]

    merged = merge_ranges(all_ranges)
    print(f"  Merged: {len(merged)} ranges")

    result = serialize(index, merged)

    # Recovery: if output is still too short, return full markdown
    if len(result) < MIN_OUTPUT_CHARS and total_items > 50:
        print(f"  Output too short ({len(result)} chars). Falling back to full markdown.")
        result = doc.export_to_markdown()

    full_len = len(doc.export_to_markdown())
    reduction = (1 - len(result) / full_len) * 100 if full_len else 0
    print(f"  Output: {len(result):,} chars ({reduction:.0f}% reduction from {full_len:,})")

    return result

In [ ]:
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument
import openpyxl

OBJ_DIR = "work/doc_objects"
EXCEL_PATH = "PA_Business_Rules.xlsx"
OUT_DIR = Path("work/reduced")
OUT_DIR.mkdir(exist_ok=True)

wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
ws = wb['Submissions']
submissions = []
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
    if row[0] and row[1]:
        submissions.append((str(row[0]).strip(), str(row[1]).strip()))
wb.close()
print(f"Loaded {len(submissions)} submissions")

doc_cache = {}
stats = {"success": 0, "suspicious": 0, "error": 0}

for i, (filename, brand) in enumerate(submissions):
    stem = Path(filename).stem
    print(f"\n[{i+1}/{len(submissions)}] {filename} | {brand}")

    if stem not in doc_cache:
        obj_path = Path(OBJ_DIR) / f"{stem}.json"
        if not obj_path.exists():
            print(f"  ERROR: {obj_path} not found")
            stats["error"] += 1
            continue
        doc_cache[stem] = DoclingDocument.load_from_json(obj_path)

    doc = doc_cache[stem]
    result = reduce_document(doc, brand)

    out_path = OUT_DIR / f"{stem}__{brand}.txt"
    out_path.write_text(result, encoding="utf-8")

    if len(result) < 50:
        stats["suspicious"] += 1
        print(f"  ⚠️ SUSPICIOUSLY SHORT: {len(result)} chars")
    else:
        stats["success"] += 1

print(f"\n{'='*60}")
print(f"Done. Success: {stats['success']}, Suspicious: {stats['suspicious']}, Error: {stats['error']}")
print(f"Outputs in {OUT_DIR}/")

In [ ]:
"""
PA Policy Parameter Extraction Pipeline — Integrated with Document Reducer.

Architecture:
  1. Document Reducer (pre-computed): Docling structured output → algorithmic
     section extraction → focused text per (filename, brand) pair.
  2. This pipeline: reads reduced text → Gemini extraction → validation → CSV.

The reducer handles all document structures locally (heading match, table search,
cross-reference following). The LLM only does the hard reasoning: step counting,
AND/OR logic, phototherapy classification.

Inputs:
  work/reduced/   — Reducer output (primary)
  work/md_output/    — Full markdown (fallback if reducer output missing)
  PA_Business_Rules.xlsx — Submission list

Outputs:
  work/result.csv           — 79-row extraction results
  work/_extraction_log.json — Detailed extraction log

Run: GEMINI_API_KEY=your-key python pipeline_integrated.py
"""
import os
import json
import csv
import re
from pathlib import Path
from google import genai
from google.genai import types
import openpyxl

# === CONFIG ===
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    raise RuntimeError("Set GEMINI_API_KEY env var.")

REDUCED_DIR = os.environ.get("REDUCED_DIR", "work/reduced")
MD_DIR = os.environ.get("MD_DIR", "work/md_output")
EXCEL_PATH = os.environ.get("EXCEL_PATH", "PA_Business_Rules.xlsx")
OUTPUT_CSV = os.environ.get("OUTPUT_CSV", "work/result.csv")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_ID = "gemini-2.5-flash"

# === FDA APPROVED AGES FOR PsO BY BRAND ===
FDA_PSO_AGE = {
    "STELARA": ">=6",
    "TREMFYA": ">=6",
    "ENBREL": ">=4",
    "COSENTYX": ">=6",
    "SKYRIZI": ">=18",
    "OTEZLA": ">=6",
    "BIMZELX": ">=18",
    "SILIQ": ">=18",
    "ILUMYA": ">=18",
    "CIMZIA": ">=18",
    "AMJEVITA": ">=18",
    "REMICADE": ">=18",
    "OTULFI": ">=6",
    "YESINTEK": ">=6",
    "ACITRETIN": ">=18",
}

# === EXTRACTION PROMPT ===
SYSTEM_PROMPT = """You are a pharmaceutical prior authorization policy analyst. You extract structured parameters from PA policy documents for the Plaque Psoriasis (PsO) indication.

You will receive a policy document (or relevant excerpt) and a target brand name. Extract the 12 parameters defined below for that brand's PsO coverage ONLY.

=== CRITICAL DISTINCTION: "NA" vs "No" ===

- "NA" = The information COULD NOT BE FOUND in the policy document. The policy simply does not mention this parameter for PsO. Data is missing/absent.
- "No" = The policy EXPLICITLY STATES there is no restriction or requirement.

If a parameter is not mentioned anywhere in the PsO section or universal criteria, output "NA" — NOT "No".

=== CRITICAL RULES ===

INDICATION FILTERING:
- Extract ONLY for Plaque Psoriasis (PsO). Ignore all other indications.
- If the policy separates "moderate-to-severe" and "severe" PsO, extract ONLY moderate-to-severe criteria.
- SPECIALIST TYPES: Only capture specialists listed in the PsO section. Do NOT include Gastroenterologist or other specialists from CD/UC sections.
- STEP THERAPY: Only count steps from the PsO section and universal/general criteria. Do NOT count steps from other indications.
- SELF-REFERENCING: If the target brand appears as one of several options in a step therapy requirement, exclude it from the option list but still count the step — the patient would use one of the other options. Only exclude the entire step if the target brand is the sole option (a drug cannot be the only prerequisite to access itself). This can occur due to document redlining artifacts.

UNIVERSAL vs INDICATION-SPECIFIC CRITERIA:
- Some policies have "General Requirements" or "Universal Criteria" that apply across all indications, PLUS separate PsO-specific criteria.
- These two sets combine with AND — the patient must satisfy BOTH.
- When counting steps, take the UNION of universal steps and PsO-specific steps.

STEP COUNTING — LEAST RESTRICTIVE PATH:
- OR statements → take the path with FEWER total steps (least restrictive).
- AND statements → patient must complete ALL (union).
- "Tried Drug A OR Drug B" = 1 step, NOT 2.
- "Tried Drug A AND Drug B" = 2 steps.
- "must have tried ONE of the following: [list of 8 drugs]" = 1 step, not 8.
- "must have tried TWO of the following: [list]" = 2 steps.
- Maximum realistic: branded steps rarely exceed 4, generic rarely exceed 3.

STEP CLASSIFICATION:
- Branded/biologic: Named biologic drugs, TNF inhibitors, IL-inhibitors, targeted synthetic drugs, biosimilars.
- Generic: Topical agents, conventional systemics (methotrexate, cyclosporine, acitretin), non-biologic drugs.
- Phototherapy: EXCLUDED from both counts — captured separately.

PHOTOTHERAPY:
- "Yes" = mandatory AND with no alternative path.
- "No" = mentioned but optional (in an OR).
- "NA" = not mentioned or no step criteria.

=== PARAMETER DEFINITIONS ===

1. AGE — Explicit threshold (e.g., ">=18"). "adults" → ">=18". "FDA labeled age" → use the FDA age provided. Not mentioned → "NA". Explicitly no restriction → "No". Two ages → youngest.

2. STEP THERAPY REQUIREMENTS — All step therapy language for PsO. "NA" if none.

3. NUMBER OF STEPS THROUGH BRANDS — Integer as string, "No" if criteria exist but no branded steps, "NA" if no criteria.

4. NUMBER OF STEPS THROUGH GENERIC — Same logic.

5. STEP THROUGH PHOTOTHERAPY — "Yes"=mandatory AND. "No"=optional OR. "NA"=not mentioned.

6. TB TEST REQUIRED — "Yes"/"No"/"NA".

7. QUANTITY LIMITS — Only explicitly labeled "quantity limit". Not "dosage"/"dosing". "NA" if not mentioned.

8. SPECIALIST TYPES — PsO only. "NA" if not mentioned.

9. INITIAL AUTHORIZATION DURATION (MONTHS) — Convert weeks/days/years to months. "NA" if not mentioned.

10. REAUTHORIZATION DURATION (MONTHS) — Same. "NA" if not mentioned.

11. REAUTHORIZATION REQUIRED — "Yes"/"No"/"NA". If reauth duration or requirements is non-NA, must be "Yes".

12. REAUTHORIZATION REQUIREMENTS — Text or "NA".

=== FEW-SHOT EXAMPLES ===

EXAMPLE 1 — Phototherapy in OR, no age mentioned
{
  "age": "NA",
  "step_therapy_requirements": "inadequate response to phototherapy or methotrexate/cyclosporine/acitretin",
  "number_of_steps_through_brands": "No",
  "number_of_steps_through_generic": "1",
  "step_through_phototherapy": "No",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "NA",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "NA",
  "reauthorization_required": "No",
  "reauthorization_requirements": "NA"
}

EXAMPLE 2 — Branded + generic steps with AND
{
  "age": ">=18",
  "step_therapy_requirements": "ONE topical AND TWO biologics/targeted synthetics",
  "number_of_steps_through_brands": "2",
  "number_of_steps_through_generic": "1",
  "step_through_phototherapy": "NA",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "Dermatologist",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "12",
  "reauthorization_required": "Yes",
  "reauthorization_requirements": "Positive clinical response: reduction in BSA or improvement in symptoms"
}

EXAMPLE 3 — OR logic, least restrictive path
{
  "age": ">=6",
  "step_therapy_requirements": "(topical + methotrexate) OR (targeted immunomodulator)",
  "number_of_steps_through_brands": "No",
  "number_of_steps_through_generic": "2",
  "step_through_phototherapy": "NA",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "Dermatologist",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "12",
  "reauthorization_required": "Yes",
  "reauthorization_requirements": "Positive clinical response AND not combined with another targeted immunomodulator"
}

=== PsO BRAND REFERENCE ===

BRANDED/BIOLOGIC: Amjevita, Avsola, Bimzelx, Cimzia, Cosentyx, Cyltezo, Enbrel, Humira, Hyrimoz, Idacio, Ilumya, Inflectra, Hulio, Otezla, Remicade, Renflexis, Siliq, Skyrizi, Sotyktu, Stelara, Taltz, Tremfya, Vtama, Yuflyma, Yusimry, Zoryve, Wezlana, Selarsdi, Yesintek, Psychiva/Quallent, Steqeyma, Otulfi

GENERIC/CONVENTIONAL: Acitretin, Cyclosporine, Methotrexate, topical corticosteroids, vitamin D analogs, calcineurin inhibitors, tazarotene, coal tar, anthralin

=== OUTPUT FORMAT ===

Return a JSON object with exactly the 12 fields. No extra fields or commentary."""


# === LOAD SUBMISSIONS ===
def load_submissions():
    wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
    ws = wb['Submissions']
    submissions = []
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
        filename, brand = row[0], row[1]
        if not filename or not brand:
            continue
        submissions.append({"filename": str(filename).strip(), "brand": str(brand).strip()})
    wb.close()
    return submissions


# === LOAD CONTENT ===
def load_content(filename, brand):
    """
    Load pre-reduced content for a (filename, brand) pair.
    Priority 1: Reducer output in REDUCED_DIR (algorithmically extracted)
    Priority 2: Full markdown in MD_DIR (fallback)
    """
    stem = Path(filename).stem

    # Priority 1: Reducer output (brand-specific)
    reduced_path = Path(REDUCED_DIR) / f"{stem}__{brand}.txt"
    if reduced_path.exists():
        text = reduced_path.read_text(encoding="utf-8")
        if text.strip():
            return text, "reduced"

    # Priority 2: Full markdown
    md_path = Path(MD_DIR) / f"{stem}.md"
    if md_path.exists():
        return md_path.read_text(encoding="utf-8"), "markdown-fallback"

    return None, "missing"


# === PARSE JSON ===
def parse_json_response(text):
    """Parse JSON from model response, handling markdown fences and preamble."""
    if text is None:
        raise ValueError("Response text is None")

    cleaned = text.strip()

    # Find JSON block inside markdown fences anywhere in response
    fence_match = re.search(r'```(?:json)?\s*\n(.*?)\n```', cleaned, re.DOTALL)
    if fence_match:
        cleaned = fence_match.group(1).strip()
    # Or find first { ... } block
    elif not cleaned.startswith("{"):
        brace_start = cleaned.find("{")
        brace_end = cleaned.rfind("}")
        if brace_start != -1 and brace_end != -1:
            cleaned = cleaned[brace_start:brace_end + 1]

    return json.loads(cleaned)


# === GEMINI CALL HELPER ===
def call_gemini(contents, system_instruction=None, json_output=False, max_retries=3):
    """Generic Gemini call with retry logic."""
    config_params = {
        "temperature": 0.1,
        "thinking_config": types.ThinkingConfig(thinking_level="HIGH"),
    }
    if system_instruction:
        config_params["system_instruction"] = system_instruction
    if json_output:
        config_params["response_mime_type"] = "application/json"

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=contents,
                config=types.GenerateContentConfig(**config_params),
            )

            if not response.candidates:
                raise ValueError(f"No candidates returned: {getattr(response, 'prompt_feedback', 'unknown')}")

            candidate = response.candidates[0]
            finish_reason = getattr(candidate, "finish_reason", None)
            if finish_reason is not None and "STOP" not in str(finish_reason):
                raise ValueError(f"Stopped: {finish_reason}")

            return response.text, None

        except Exception as e:
            print(f"    Error (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                continue

    return None, "max_retries_exceeded"


# === PARAMETER EXTRACTION ===
def extract_parameters(policy_text, brand):
    """Extract 12 PA parameters from policy text."""
    fda_age = FDA_PSO_AGE.get(brand.upper(), ">=18")

    user_prompt = f"""Extract the 12 PA parameters for the brand "{brand}" on the Plaque Psoriasis (PsO) indication from the following policy document.

FDA APPROVED AGE for {brand} in PsO: {fda_age}
(Use this if the policy says "FDA labelled age" without a number.)

POLICY DOCUMENT:
{policy_text}"""

    text, error = call_gemini(user_prompt, system_instruction=SYSTEM_PROMPT, json_output=True)
    if error:
        return None, error

    try:
        result = parse_json_response(text)
        return result, None
    except (json.JSONDecodeError, ValueError) as e:
        return None, f"JSON parse error: {e}"


# === VALIDATE AND NORMALIZE ===
def validate_and_normalize(params, brand):
    # Resolve FDA labelled age
    age = str(params.get("age", "NA")).strip()
    if "fda" in age.lower():
        params["age"] = FDA_PSO_AGE.get(brand.upper(), ">=18")

    # Binary fields
    for key in ["tb_test_required", "reauthorization_required"]:
        val = str(params.get(key, "NA")).strip().lower()
        if val in ("yes", "y", "true"):
            params[key] = "Yes"
        elif val in ("no", "n", "false"):
            params[key] = "No"
        else:
            params[key] = "NA"

    # Durations — extract leading digits, handle text like "12 months"
    for key in ["initial_authorization_duration", "reauthorization_duration"]:
        val = params.get(key, "")
        if val is None or str(val).strip().lower() in ("", "none", "null", "unspecified"):
            params[key] = "NA"
        elif isinstance(val, (int, float)):
            params[key] = str(int(val))
        else:
            cleaned = str(val).strip()
            if cleaned.lower() in ("na", "n/a"):
                params[key] = "NA"
            else:
                digit_match = re.match(r'(\d+)', cleaned)
                params[key] = digit_match.group(1) if digit_match else "NA"

    # Step counts — extract leading digits, normalize 0 to "No"
    for key in ["number_of_steps_through_brands", "number_of_steps_through_generic"]:
        val = params.get(key, "NA")
        val_str = str(val).strip().lower()
        if val_str in ("na", "n/a", "none", ""):
            params[key] = "NA"
        elif val_str in ("no", "0"):
            params[key] = "No"
        elif isinstance(val, (int, float)):
            params[key] = "No" if int(val) == 0 else str(int(val))
        else:
            cleaned = str(val).strip()
            digit_match = re.match(r'(\d+)', cleaned)
            if digit_match:
                num = int(digit_match.group(1))
                params[key] = "No" if num == 0 else str(num)

    # Phototherapy
    photo = str(params.get("step_through_phototherapy", "NA")).strip().lower()
    if photo in ("yes", "y"):
        params["step_through_phototherapy"] = "Yes"
    elif photo in ("no", "n"):
        params["step_through_phototherapy"] = "No"
    else:
        params["step_through_phototherapy"] = "NA"

    # Cross-field reauth
    reauth_dur = str(params.get("reauthorization_duration", "NA")).strip().lower()
    reauth_req = str(params.get("reauthorization_requirements", "NA")).strip().lower()
    if reauth_dur not in ("na", "n/a", "none", "") or \
       reauth_req not in ("na", "n/a", "none", ""):
        params["reauthorization_required"] = "Yes"

    return params


# === CSV COLUMNS ===
CSV_COLUMNS = [
    ("Filename", lambda row, p: row["filename"]),
    ("Brand", lambda row, p: row["brand"]),
    ("Age", lambda row, p: p.get("age", "")),
    ("Step Therapy Requirements Documented in Policy", lambda row, p: p.get("step_therapy_requirements", "")),
    ("Number of Steps through Brands", lambda row, p: p.get("number_of_steps_through_brands", "")),
    ("Number of Steps through Generic", lambda row, p: p.get("number_of_steps_through_generic", "")),
    ("Step through-Phototherapy", lambda row, p: p.get("step_through_phototherapy", "")),
    ("TB Test required", lambda row, p: p.get("tb_test_required", "")),
    ("Quantity Limits", lambda row, p: p.get("quantity_limits", "")),
    ("Specialist Types", lambda row, p: p.get("specialist_types", "")),
    ("Initial Authorization Duration(in-months)", lambda row, p: p.get("initial_authorization_duration", "")),
    ("Reauthorization Duration(in-months)", lambda row, p: p.get("reauthorization_duration", "")),
    ("Reauthorization Required", lambda row, p: p.get("reauthorization_required", "")),
    ("Reauthorization Requirements Documented in Policy", lambda row, p: p.get("reauthorization_requirements", "")),
    ("Access Score", lambda row, p: p.get("access_score", "")),
]


# === MAIN PIPELINE ===
def run_pipeline():
    submissions = load_submissions()
    print(f"Loaded {len(submissions)} submission rows")
    print(f"Reduced dir: {REDUCED_DIR}")
    print(f"Markdown dir: {MD_DIR}")

    if not submissions:
        print("ERROR: No valid submissions found.")
        return

    # Verify directories exist
    if not Path(REDUCED_DIR).exists():
        print(f"WARNING: {REDUCED_DIR} not found. All files will use markdown fallback.")
    if not Path(MD_DIR).exists():
        print(f"ERROR: {MD_DIR} not found. Cannot proceed.")
        return

    results = []
    errors = []
    source_counts = {"reduced": 0, "markdown-fallback": 0, "missing": 0}

    for i, row in enumerate(submissions):
        filename = row["filename"]
        brand = row["brand"]
        print(f"[{i+1}/{len(submissions)}] {filename} | {brand}")

        # Load content (reducer output preferred, markdown fallback)
        policy_text, source = load_content(filename, brand)
        if policy_text is None:
            print(f"  ERROR: content not found ({source})")
            errors.append({"filename": filename, "brand": brand, "error": "file_not_found"})
            results.append({"row": row, "params": {}})
            source_counts["missing"] += 1
            continue

        if not policy_text.strip():
            print(f"  ERROR: empty content ({source})")
            errors.append({"filename": filename, "brand": brand, "error": "empty_file"})
            results.append({"row": row, "params": {}})
            source_counts["empty"] = source_counts.get("empty", 0) + 1
            continue

        source_counts[source] = source_counts.get(source, 0) + 1
        print(f"  Source: {source} ({len(policy_text):,} chars)")

        # Extract parameters
        params, error = extract_parameters(policy_text, brand)
        if error:
            print(f"  EXTRACTION ERROR: {error}")
            errors.append({"filename": filename, "brand": brand, "error": error})
            results.append({"row": row, "params": {}})
            continue

        params = validate_and_normalize(params, brand)
        params["access_score"] = ""

        results.append({"row": row, "params": params})
        print(f"  OK: age={params.get('age')}, brands={params.get('number_of_steps_through_brands')}, "
              f"generic={params.get('number_of_steps_through_generic')}, "
              f"auth={params.get('initial_authorization_duration')}")

    # Write CSV
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([col[0] for col in CSV_COLUMNS])
        for r in results:
            writer.writerow([col[1](r["row"], r["params"]) for col in CSV_COLUMNS])

    # Summary
    print(f"\n{'='*60}")
    print(f"Done. {len(results) - len(errors)}/{len(results)} succeeded.")
    print(f"Sources: {source_counts}")
    print(f"Errors: {len(errors)}")
    if errors:
        for e in errors:
            print(f"  {e['filename']} | {e['brand']}: {e['error']}")
    print(f"Output: {OUTPUT_CSV}")

    # Log
    log_path = Path(OUTPUT_CSV).parent / "_extraction_log.json"
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(
            [{"filename": r["row"]["filename"], "brand": r["row"]["brand"], "params": r["params"]} for r in results],
            f, indent=2,
        )

run_pipeline()

In [ ]:
"""
Document Reducer v2 — Extracts PsO-relevant sections from Docling structured documents.

Architecture:
  Phase 1: Find the brand's section boundaries
  Phase 2: Within those boundaries, find PsO criteria + utility sections

Three strategies for Phase 2: heading match, table search, cross-reference following.
No LLM calls. Pure algorithmic document understanding.
"""
import re
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument

# === BRAND ALIASES ===
BRAND_ALIASES = {
    "TREMFYA": ["tremfya", "guselkumab"],
    "STELARA": ["stelara", "ustekinumab"],
    "ENBREL": ["enbrel", "etanercept"],
    "AMJEVITA": ["amjevita", "adalimumab-atto", "adalimumab"],
    "COSENTYX": ["cosentyx", "secukinumab"],
    "SKYRIZI": ["skyrizi", "risankizumab"],
    "OTEZLA": ["otezla", "apremilast"],
    "SILIQ": ["siliq", "brodalumab"],
    "CIMZIA": ["cimzia", "certolizumab"],
    "BIMZELX": ["bimzelx", "bimekizumab"],
    "REMICADE": ["remicade", "infliximab"],
    "OTULFI": ["otulfi", "ustekinumab-aauz"],
    "ILUMYA": ["ilumya", "tildrakizumab"],
    "YESINTEK": ["yesintek", "ustekinumab-kfce"],
    "ACITRETIN": ["acitretin"],
}

PSO_PATTERNS = [
    re.compile(r'\bplaque\s+psoriasis\b', re.IGNORECASE),
    re.compile(r'\bPsO\b'),
    re.compile(r'\bpsoriasis\b', re.IGNORECASE),
]

UTILITY_PATTERNS = [
    # Duration and authorization
    re.compile(r'\bduration\s+of\s+approval\b', re.IGNORECASE),
    re.compile(r'\binitial\s+auth', re.IGNORECASE),
    re.compile(r'\binitial\s+approval\b', re.IGNORECASE),
    re.compile(r'\blength\s+of\s+auth', re.IGNORECASE),
    # Reauthorization / renewal
    re.compile(r'\breauthorization\b', re.IGNORECASE),
    re.compile(r'\brenewal\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bcontinuation\s+criteria\b', re.IGNORECASE),
    # Quantity limits
    re.compile(r'\bquantity\s+limit', re.IGNORECASE),
    re.compile(r'\bdosing\s+limit', re.IGNORECASE),
    re.compile(r'\bmax.*\s+dose\b', re.IGNORECASE),
    # Approval / denial
    re.compile(r'\bapproval\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bdenial\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bclinical\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bcoverage\s+criteria\b', re.IGNORECASE),
    # TB test / screening
    re.compile(r'\btuberculosis\b', re.IGNORECASE),
    re.compile(r'\bTB\s+test', re.IGNORECASE),
    re.compile(r'\bTB\s+screen', re.IGNORECASE),
    re.compile(r'\blatent\s+TB\b', re.IGNORECASE),
    re.compile(r'\bscreening\s+requirement', re.IGNORECASE),
    # Specialist / prescriber
    re.compile(r'\bspecialist', re.IGNORECASE),
    re.compile(r'\bprescriber\s+requirement', re.IGNORECASE),
    re.compile(r'\bprescribing\s+provider', re.IGNORECASE),
    re.compile(r'\bdermatolog', re.IGNORECASE),
    re.compile(r'\brheumatolog', re.IGNORECASE),
    # General requirements (often contain TB, specialist, etc.)
    re.compile(r'\bgeneral\s+requirement', re.IGNORECASE),
    re.compile(r'\bgeneral\s+criteria\b', re.IGNORECASE),
    re.compile(r'\buniversal\s+criteria\b', re.IGNORECASE),
    re.compile(r'\brequired\s+medical\s+info', re.IGNORECASE),
    re.compile(r'\bclinical\s+review\s+criteria\b', re.IGNORECASE),
    # Monitoring / testing
    re.compile(r'\bmonitoring\s+requirement', re.IGNORECASE),
    re.compile(r'\blab.*\s+requirement', re.IGNORECASE),
    re.compile(r'\bprecaution', re.IGNORECASE),
    re.compile(r'\bcontraindication', re.IGNORECASE),
]

# Structural sub-headers that repeat in every section — NOT drug-class boundaries
STRUCTURAL_HEADERS = re.compile(
    r'^\s*('
    r'Products?\s+Affected|'
    r'Goal\(?s?\)?:?|'
    r'Length\s+of\s+Authorization:?|'
    r'Requires\s+PA:?|'
    r'Covered\s+Alternatives:?|'
    r'Approval\s+Criteria|'
    r'Renewal\s+Criteria|'
    r'References?:?|'
    r'Table\s+\d|'
    r'PA\s+Criteria|'
    r'P&T|'
    r'Effective\s+Date|'
    r'Notes?:?'
    r')\s*$',
    re.IGNORECASE
)

GOTO_PATTERN = re.compile(r'Go\s+to\s+#\s*(\d+)', re.IGNORECASE)
CRITERIA_NUM_PATTERN = re.compile(r'^\s*(\d+)\.\s')


# === DOCUMENT INDEX ===

class DocumentIndex:
    def __init__(self, doc: DoclingDocument):
        self.doc = doc
        self.items = list(doc.iterate_items())
        self.headers = []
        self.tables = []
        self.text_items = []
        self._build_index()

    def _build_index(self):
        table_counter = 0
        for pos, (item, level) in enumerate(self.items):
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                self.headers.append((pos, text, level))
            elif item_type == 'TableItem':
                self.tables.append((pos, table_counter, item))
                table_counter += 1
            if text.strip():
                self.text_items.append((pos, text, level, item_type))

    def get_table_df(self, table_item):
        try:
            return table_item.export_to_dataframe(self.doc)
        except:
            try:
                return table_item.export_to_dataframe()
            except:
                return None

    def get_text_between(self, start, end):
        parts = []
        for pos, (item, level) in enumerate(self.items):
            if pos < start:
                continue
            if pos >= end:
                break
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                parts.append(f"\n## {text}\n")
            elif item_type == 'TableItem':
                df = self.get_table_df(item)
                if df is not None:
                    parts.append(df.to_markdown(index=False))
                    parts.append("")
            elif item_type == 'ListItem':
                parts.append(f"- {text}")
            elif text.strip():
                parts.append(text)
        return "\n".join(parts)

    def is_structural_header(self, text):
        return bool(STRUCTURAL_HEADERS.match(text.strip()))


# === PHASE 1: FIND BRAND SECTION ===

def find_brand_section(index: DocumentIndex, brand: str) -> tuple:
    """
    Find the document region that contains PA criteria for this brand.
    Returns (start_pos, end_pos). For small docs, returns (0, total_items).
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    total = len(index.items)

    # Small documents: entire document is the section
    if total < 200:
        return (0, total)

    # Strategy 1: Find brand in a section header
    brand_header_pos = None
    for pos, text, level in index.headers:
        text_lower = text.lower().strip()
        if any(a == text_lower or a in text_lower.split() for a in aliases):
            # Verify it's a drug-name header, not a structural one
            if not index.is_structural_header(text):
                brand_header_pos = pos
                break

    if brand_header_pos is not None:
        # Find the end: walk forward to the next non-structural header at same level
        start = brand_header_pos
        brand_level = None
        for pos, text, level in index.headers:
            if pos == brand_header_pos:
                brand_level = level
                continue
            if brand_level is not None and pos > brand_header_pos:
                if level <= brand_level and not index.is_structural_header(text):
                    return (start, pos)
        return (start, total)

    # Strategy 2: Find brand in a table cell, then use adaptive scoping
    for pos, table_idx, table_item in index.tables:
        df = index.get_table_df(table_item)
        if df is None:
            continue
        df_str = df.to_string().lower()
        if any(a in df_str for a in aliases):
            return find_section_scope(index, pos)

    # Strategy 3: Find brand in any text item
    for pos, text, level, item_type in index.text_items:
        if any(a in text.lower() for a in aliases):
            # Use table proximity if tables exist nearby
            nearby_tables = [(tp, ti, tt) for tp, ti, tt in index.tables
                             if abs(tp - pos) < 50]
            if nearby_tables:
                return find_section_scope(index, nearby_tables[0][0])
            # Otherwise take a generous window
            return (max(0, pos - 50), min(total, pos + 200))

    # Fallback: entire document
    print(f"  WARNING: Brand '{brand}' not found. Using full document.")
    return (0, total)


def find_section_scope(index: DocumentIndex, anchor_table_pos: int) -> tuple:
    """Adaptive gap-based section scoping around an anchor table."""
    table_positions = [pos for pos, _, _ in index.tables]
    if not table_positions:
        return (0, len(index.items))

    anchor_idx = None
    for i, (pos, _, _) in enumerate(index.tables):
        if pos == anchor_table_pos:
            anchor_idx = i
            break
    if anchor_idx is None:
        return (0, len(index.items))

    # Seed: 10 tables each direction
    seed_start = max(0, anchor_idx - 10)
    seed_end = min(len(table_positions), anchor_idx + 11)
    seed_positions = table_positions[seed_start:seed_end]

    gaps = [seed_positions[i] - seed_positions[i - 1] for i in range(1, len(seed_positions))]
    if not gaps:
        return (0, len(index.items))

    sorted_gaps = sorted(gaps)
    median_gap = sorted_gaps[len(sorted_gaps) // 2]
    initial_threshold = max(median_gap * 3, 10)

    # Walk backward
    start_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, 0, -1):
        if table_positions[i] - table_positions[i - 1] > initial_threshold:
            break
        start_pos = table_positions[i - 1]

    # Walk forward
    end_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, len(table_positions) - 1):
        if table_positions[i + 1] - table_positions[i] > initial_threshold:
            break
        end_pos = table_positions[i + 1]

    # Refine with 2× largest intra-cluster gap
    cluster = [p for p in table_positions if start_pos <= p <= end_pos]
    if len(cluster) > 1:
        intra_gaps = [cluster[i + 1] - cluster[i] for i in range(len(cluster) - 1)]
        refined = max(max(intra_gaps) * 2, 10)
        start_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, 0, -1):
            if table_positions[i] - table_positions[i - 1] > refined:
                break
            start_pos = table_positions[i - 1]
        end_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, len(table_positions) - 1):
            if table_positions[i + 1] - table_positions[i] > refined:
                break
            end_pos = table_positions[i + 1]

    # Extend to capture context
    end_pos = min(end_pos + 10, len(index.items))
    for pos, text, level in reversed(index.headers):
        if pos < start_pos and start_pos - pos < 30:
            start_pos = pos
            break

    return (start_pos, end_pos)


# === PHASE 2: FIND PsO CONTENT WITHIN SECTION ===

def find_pso_by_headings(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy A: Find PsO-related headings within the scoped section."""
    collected = []
    scoped_headers = [(pos, text, level) for pos, text, level in index.headers
                      if scope_start <= pos < scope_end]

    for i, (pos, text, level) in enumerate(scoped_headers):
        is_pso = any(p.search(text) for p in PSO_PATTERNS)
        is_utility = any(p.search(text) for p in UTILITY_PATTERNS)

        if is_pso or is_utility:
            # End at next header at same or higher level within scope
            end_pos = scope_end
            for j in range(i + 1, len(scoped_headers)):
                next_pos, next_text, next_level = scoped_headers[j]
                if next_level <= level and not index.is_structural_header(next_text):
                    end_pos = next_pos
                    break
            collected.append((pos, end_pos))

    return collected


def find_pso_in_tables(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy B: Find tables containing PsO keywords within the scoped section.
    
    Filters:
    - Skip administrative tables (drug listings, NDC tables, dosing grids)
    - Small tables (<20 rows): include if PsO keyword appears anywhere
    - Criteria-structured tables: include if PsO keyword found
    - Large non-criteria tables: require both brand AND PsO keywords
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    collected = []

    # Column names that indicate a PA criteria table
    CRITERIA_TABLE_PATTERNS = re.compile(
        r'PA\s+Criteria|Criteria\s+Details|Approval|Required\s+Medical|'
        r'Coverage\s+Duration|Renewal|Exclusion|Prescriber|Restriction|'
        r'Authorization|Step\s+Therapy|Clinical\s+Criteria',
        re.IGNORECASE
    )

    # Column names that indicate an administrative/formulary listing table — skip these
    ADMIN_TABLE_PATTERNS = re.compile(
        r'Strength|NDC|Targeted\s+NDC|Day\s+Supply|Dose\s+Form|'
        r'Client\s+Formulary|GPI|HICL|Package\s+Size|Route\s+of\s+Admin|'
        r'AWP|WAC|Unit\s+Price|Wholesale|Generic\s+Name.*Brand\s+Name|'
        r'Brand\s+Name.*Generic\s+Name|HCPCS|J-?Code|CPT',
        re.IGNORECASE
    )

    for pos, table_idx, table_item in index.tables:
        if pos < scope_start or pos >= scope_end:
            continue
        df = index.get_table_df(table_item)
        if df is None:
            continue

        num_rows = len(df)
        col_str = ' '.join(str(c) for c in df.columns)

        # Skip administrative tables entirely
        if ADMIN_TABLE_PATTERNS.search(col_str):
            continue

        # Check if this is a PA criteria table by column names
        is_criteria_table = bool(CRITERIA_TABLE_PATTERNS.search(col_str))

        # Small table or criteria-structured table: include if PsO keyword found
        if num_rows < 20 or is_criteria_table:
            df_str = df.to_string().lower()
            if any(p.search(df_str) for p in PSO_PATTERNS):
                collected.append((max(scope_start, pos - 2), min(scope_end, pos + 2)))

        # Large table: only include if BOTH brand AND PsO found
        else:
            df_str = df.to_string().lower()
            has_brand = any(a in df_str for a in aliases)
            has_pso = any(p.search(df_str) for p in PSO_PATTERNS)
            if has_brand and has_pso:
                collected.append((max(scope_start, pos - 2), min(scope_end, pos + 2)))

    return collected


def follow_cross_references(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy C: Follow 'Go to #X' routing chains within the scoped section."""
    collected = []

    # Get all tables in scope
    scope_tables = []
    for pos, table_idx, table_item in index.tables:
        if scope_start <= pos <= scope_end:
            df = index.get_table_df(table_item)
            if df is not None:
                scope_tables.append((pos, table_idx, df))

    if not scope_tables:
        return collected

    # Find PsO routing entries: tables with both PsO keyword and "Go to #X"
    refs_to_follow = set()
    initial_tables = set()
    for tbl_pos, tbl_idx, tbl_df in scope_tables:
        for row_idx in range(len(tbl_df)):
            row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
            row_lower = row_text.lower()
            has_pso = any(p.search(row_lower) for p in PSO_PATTERNS)
            refs_in_row = GOTO_PATTERN.findall(row_text)
            if has_pso and refs_in_row:
                initial_tables.add(tbl_pos)
                refs_to_follow.update(int(r) for r in refs_in_row)

    if not refs_to_follow:
        return collected

    # BFS: follow references
    visited = set()
    tables_to_include = set(initial_tables)
    queue = list(refs_to_follow)

    while queue:
        target_num = queue.pop(0)
        if target_num in visited:
            continue
        visited.add(target_num)

        for tbl_pos, tbl_idx, tbl_df in scope_tables:
            for row_idx in range(len(tbl_df)):
                cell_0 = str(tbl_df.iloc[row_idx, 0]).strip()

                # Match criteria number in first cell
                match_bare = re.match(r'^(\d+)\.?\s*$', cell_0)
                match_prefix = CRITERIA_NUM_PATTERN.match(cell_0)

                found_num = None
                if match_bare:
                    found_num = int(match_bare.group(1))
                elif match_prefix:
                    found_num = int(match_prefix.group(1))

                # Also check column 1 for 4-column tables where col 0 is just the number
                if found_num is None and tbl_df.shape[1] >= 3:
                    cell_1 = str(tbl_df.iloc[row_idx, 1] if tbl_df.shape[1] > 1 else '').strip()
                    match_c1 = CRITERIA_NUM_PATTERN.match(cell_1)
                    if match_c1:
                        found_num = int(match_c1.group(1))

                if found_num == target_num:
                    tables_to_include.add(tbl_pos)
                    row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
                    new_refs = GOTO_PATTERN.findall(row_text)
                    for r in new_refs:
                        r_int = int(r)
                        if r_int not in visited:
                            queue.append(r_int)

    for tbl_pos in tables_to_include:
        collected.append((max(scope_start, tbl_pos - 2), min(scope_end, tbl_pos + 2)))

    return collected


# === MERGE AND SERIALIZE ===

def merge_ranges(ranges):
    if not ranges:
        return []
    sorted_r = sorted(ranges, key=lambda x: x[0])
    merged = [sorted_r[0]]
    for start, end in sorted_r[1:]:
        ps, pe = merged[-1]
        if start <= pe + 3:
            merged[-1] = (ps, max(pe, end))
        else:
            merged.append((start, end))
    return merged


def serialize(index, ranges):
    parts = []
    seen = set()
    for start, end in ranges:
        if start not in seen:
            parts.append(index.get_text_between(start, end))
            seen.add(start)
    return "\n\n---\n\n".join(parts)


# === MAIN REDUCER ===

MIN_OUTPUT_CHARS = 100
MIN_REDUCTION_RATIO = 0.30  # If output < 30% of brand section, widening triggers

def reduce_document(doc: DoclingDocument, brand: str) -> str:
    index = DocumentIndex(doc)
    total_items = len(index.items)
    print(f"  Reducer: {total_items} items, {len(index.tables)} tables, {len(index.headers)} headers")

    # Phase 1: Find brand's section
    scope_start, scope_end = find_brand_section(index, brand)
    scope_size = scope_end - scope_start
    print(f"  Brand section: items {scope_start}-{scope_end} ({scope_size} items)")

    # Confidence check: if scope is suspiciously narrow for a large document,
    # widen to full document and let Phase 2 search everywhere.
    if scope_size < 10 and total_items > 200:
        print(f"  Scope too narrow ({scope_size} items in {total_items}-item doc). Widening to full document.")
        scope_start, scope_end = 0, total_items
        scope_size = total_items

    # Get the full brand section text for comparison
    brand_section_text = index.get_text_between(scope_start, scope_end)
    brand_section_len = len(brand_section_text)

    # Phase 2: Find PsO content within section
    ranges_a = find_pso_by_headings(index, scope_start, scope_end, brand)
    ranges_b = find_pso_in_tables(index, scope_start, scope_end, brand)
    ranges_c = follow_cross_references(index, scope_start, scope_end, brand)
    print(f"  Strategy A (headings): {len(ranges_a)} ranges")
    print(f"  Strategy B (tables): {len(ranges_b)} ranges")
    print(f"  Strategy C (cross-ref): {len(ranges_c)} ranges")

    # If no PsO-specific content found, include the entire brand section
    all_ranges = ranges_a + ranges_b + ranges_c
    if not all_ranges:
        print(f"  No PsO-specific content found. Including full brand section.")
        all_ranges = [(scope_start, scope_end)]

    merged = merge_ranges(all_ranges)
    print(f"  Merged: {len(merged)} ranges")

    result = serialize(index, merged)

    # Adaptive widening: if the extracted content is less than 30% of the brand
    # section, the reduction was too aggressive — include the full brand section.
    # This preserves TB test, specialist, quantity limit sections that live
    # outside PsO-specific headings but within the brand's PA criteria.
    if brand_section_len > 0 and len(result) < brand_section_len * MIN_REDUCTION_RATIO:
        print(f"  Reduction too aggressive ({len(result):,} < {int(brand_section_len * MIN_REDUCTION_RATIO):,}). "
              f"Widening to full brand section.")
        result = brand_section_text

    # Final recovery: if output is still too short, return full markdown
    if len(result) < MIN_OUTPUT_CHARS and total_items > 50:
        print(f"  Output too short ({len(result)} chars). Falling back to full markdown.")
        result = doc.export_to_markdown()

    full_len = len(doc.export_to_markdown())
    reduction = (1 - len(result) / full_len) * 100 if full_len else 0
    print(f"  Output: {len(result):,} chars ({reduction:.0f}% reduction from {full_len:,})")

    return result

In [ ]:
import shutil
from pathlib import Path

# Clear old reducer output
out = Path("work/reduced")
if out.exists():
    shutil.rmtree(out)
out.mkdir()
print("Cleared work/reduced/")

In [ ]:
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument
import openpyxl

OBJ_DIR = "work/doc_objects"
EXCEL_PATH = "PA_Business_Rules.xlsx"
OUT_DIR = Path("work/reduced")

wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
ws = wb['Submissions']
submissions = []
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
    if row[0] and row[1]:
        submissions.append((str(row[0]).strip(), str(row[1]).strip()))
wb.close()
print(f"Loaded {len(submissions)} submissions")

doc_cache = {}
for i, (filename, brand) in enumerate(submissions):
    stem = Path(filename).stem
    print(f"\n[{i+1}/{len(submissions)}] {filename} | {brand}")

    if stem not in doc_cache:
        obj_path = Path(OBJ_DIR) / f"{stem}.json"
        if not obj_path.exists():
            print(f"  ERROR: {obj_path} not found")
            continue
        doc_cache[stem] = DoclingDocument.load_from_json(obj_path)

    doc = doc_cache[stem]
    result = reduce_document(doc, brand)

    out_path = OUT_DIR / f"{stem}__{brand}.txt"
    out_path.write_text(result, encoding="utf-8")

print(f"\nDone. Outputs in {OUT_DIR}/")

In [ ]:
"""
PA Policy Parameter Extraction Pipeline — Integrated with Document Reducer.

Architecture:
  1. Document Reducer (pre-computed): Docling structured output → algorithmic
     section extraction → focused text per (filename, brand) pair.
  2. This pipeline: reads reduced text → Gemini extraction → validation → CSV.

The reducer handles all document structures locally (heading match, table search,
cross-reference following). The LLM only does the hard reasoning: step counting,
AND/OR logic, phototherapy classification.

Inputs:
  work/reduced/   — Reducer output (primary)
  work/md_output/    — Full markdown (fallback if reducer output missing)
  PA_Business_Rules.xlsx — Submission list

Outputs:
  work/result.csv           — 79-row extraction results
  work/_extraction_log.json — Detailed extraction log

Run: GEMINI_API_KEY=your-key python pipeline_integrated.py
"""
import os
import json
import csv
import re
from pathlib import Path
from google import genai
from google.genai import types
import openpyxl

# === CONFIG ===
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    raise RuntimeError("Set GEMINI_API_KEY env var.")

REDUCED_DIR = os.environ.get("REDUCED_DIR", "work/reduced")
MD_DIR = os.environ.get("MD_DIR", "work/md_output")
EXCEL_PATH = os.environ.get("EXCEL_PATH", "PA_Business_Rules.xlsx")
OUTPUT_CSV = os.environ.get("OUTPUT_CSV", "work/result.csv")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_ID = "gemini-2.5-flash"

# === FDA APPROVED AGES FOR PsO BY BRAND ===
FDA_PSO_AGE = {
    "STELARA": ">=6",
    "TREMFYA": ">=6",
    "ENBREL": ">=4",
    "COSENTYX": ">=6",
    "SKYRIZI": ">=18",
    "OTEZLA": ">=6",
    "BIMZELX": ">=18",
    "SILIQ": ">=18",
    "ILUMYA": ">=18",
    "CIMZIA": ">=18",
    "AMJEVITA": ">=18",
    "REMICADE": ">=18",
    "OTULFI": ">=6",
    "YESINTEK": ">=6",
    "ACITRETIN": ">=18",
}

# === EXTRACTION PROMPT ===
SYSTEM_PROMPT = """You are a pharmaceutical prior authorization policy analyst. You extract structured parameters from PA policy documents for the Plaque Psoriasis (PsO) indication.

You will receive a policy document (or relevant excerpt) and a target brand name. Extract the 12 parameters defined below for that brand's PsO coverage ONLY.

=== CRITICAL DISTINCTION: "NA" vs "No" ===

- "NA" = The information COULD NOT BE FOUND in the policy document. The policy simply does not mention this parameter for PsO. Data is missing/absent.
- "No" = The policy EXPLICITLY STATES there is no restriction or requirement.

If a parameter is not mentioned anywhere in the PsO section or universal criteria, output "NA" — NOT "No".

=== CRITICAL RULES ===

INDICATION FILTERING:
- Extract ONLY for Plaque Psoriasis (PsO). Ignore all other indications.
- If the policy separates "moderate-to-severe" and "severe" PsO, extract ONLY moderate-to-severe criteria.
- SPECIALIST TYPES: Only capture specialists listed in the PsO section. Do NOT include Gastroenterologist or other specialists from CD/UC sections.
- STEP THERAPY: Only count steps from the PsO section and universal/general criteria. Do NOT count steps from other indications.
- SELF-REFERENCING: If the target brand appears as one of several options in a step therapy requirement, exclude it from the option list but still count the step — the patient would use one of the other options. Only exclude the entire step if the target brand is the sole option (a drug cannot be the only prerequisite to access itself). This can occur due to document redlining artifacts.

UNIVERSAL vs INDICATION-SPECIFIC CRITERIA:
- Some policies have "General Requirements" or "Universal Criteria" that apply across all indications, PLUS separate PsO-specific criteria.
- These two sets combine with AND — the patient must satisfy BOTH.
- When counting steps, take the UNION of universal steps and PsO-specific steps.

STEP COUNTING — LEAST RESTRICTIVE PATH:
- OR statements → take the path with FEWER total steps (least restrictive).
- AND statements → patient must complete ALL (union).
- "Tried Drug A OR Drug B" = 1 step, NOT 2.
- "Tried Drug A AND Drug B" = 2 steps.
- "must have tried ONE of the following: [list of 8 drugs]" = 1 step, not 8.
- "must have tried TWO of the following: [list]" = 2 steps.
- Maximum realistic: branded steps rarely exceed 4, generic rarely exceed 3.

STEP CLASSIFICATION:
- Branded/biologic: Named biologic drugs, TNF inhibitors, IL-inhibitors, targeted synthetic drugs, biosimilars.
- Generic: Topical agents, conventional systemics (methotrexate, cyclosporine, acitretin), non-biologic drugs.
- Phototherapy: EXCLUDED from both counts — captured separately.

PHOTOTHERAPY:
- "Yes" = mandatory AND with no alternative path.
- "No" = mentioned but optional (in an OR).
- "NA" = not mentioned or no step criteria.

=== PARAMETER DEFINITIONS ===

1. AGE — Explicit threshold (e.g., ">=18"). "adults" → ">=18". "FDA labeled age" → use the FDA age provided. Not mentioned → "NA". Explicitly no restriction → "No". Two ages → youngest.

2. STEP THERAPY REQUIREMENTS — All step therapy language for PsO. "NA" if none.

3. NUMBER OF STEPS THROUGH BRANDS — Integer as string, "No" if criteria exist but no branded steps, "NA" if no criteria.

4. NUMBER OF STEPS THROUGH GENERIC — Same logic.

5. STEP THROUGH PHOTOTHERAPY — "Yes"=mandatory AND. "No"=optional OR. "NA"=not mentioned.

6. TB TEST REQUIRED — "Yes"/"No"/"NA".

7. QUANTITY LIMITS — Only explicitly labeled "quantity limit". Not "dosage"/"dosing". "NA" if not mentioned.

8. SPECIALIST TYPES — PsO only. "NA" if not mentioned.

9. INITIAL AUTHORIZATION DURATION (MONTHS) — Convert weeks/days/years to months. "NA" if not mentioned.

10. REAUTHORIZATION DURATION (MONTHS) — Same. "NA" if not mentioned.

11. REAUTHORIZATION REQUIRED — "Yes"/"No"/"NA". If reauth duration or requirements is non-NA, must be "Yes".

12. REAUTHORIZATION REQUIREMENTS — Text or "NA".

=== FEW-SHOT EXAMPLES ===

EXAMPLE 1 — Phototherapy in OR, no age mentioned
{
  "age": "NA",
  "step_therapy_requirements": "inadequate response to phototherapy or methotrexate/cyclosporine/acitretin",
  "number_of_steps_through_brands": "No",
  "number_of_steps_through_generic": "1",
  "step_through_phototherapy": "No",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "NA",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "NA",
  "reauthorization_required": "No",
  "reauthorization_requirements": "NA"
}

EXAMPLE 2 — Branded + generic steps with AND
{
  "age": ">=18",
  "step_therapy_requirements": "ONE topical AND TWO biologics/targeted synthetics",
  "number_of_steps_through_brands": "2",
  "number_of_steps_through_generic": "1",
  "step_through_phototherapy": "NA",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "Dermatologist",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "12",
  "reauthorization_required": "Yes",
  "reauthorization_requirements": "Positive clinical response: reduction in BSA or improvement in symptoms"
}

EXAMPLE 3 — OR logic, least restrictive path
{
  "age": ">=6",
  "step_therapy_requirements": "(topical + methotrexate) OR (targeted immunomodulator)",
  "number_of_steps_through_brands": "No",
  "number_of_steps_through_generic": "2",
  "step_through_phototherapy": "NA",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "Dermatologist",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "12",
  "reauthorization_required": "Yes",
  "reauthorization_requirements": "Positive clinical response AND not combined with another targeted immunomodulator"
}

=== PsO BRAND REFERENCE ===

BRANDED/BIOLOGIC: Amjevita, Avsola, Bimzelx, Cimzia, Cosentyx, Cyltezo, Enbrel, Humira, Hyrimoz, Idacio, Ilumya, Inflectra, Hulio, Otezla, Remicade, Renflexis, Siliq, Skyrizi, Sotyktu, Stelara, Taltz, Tremfya, Vtama, Yuflyma, Yusimry, Zoryve, Wezlana, Selarsdi, Yesintek, Psychiva/Quallent, Steqeyma, Otulfi

GENERIC/CONVENTIONAL: Acitretin, Cyclosporine, Methotrexate, topical corticosteroids, vitamin D analogs, calcineurin inhibitors, tazarotene, coal tar, anthralin

=== OUTPUT FORMAT ===

Return a JSON object with exactly the 12 fields. No extra fields or commentary."""


# === LOAD SUBMISSIONS ===
def load_submissions():
    wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
    ws = wb['Submissions']
    submissions = []
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
        filename, brand = row[0], row[1]
        if not filename or not brand:
            continue
        submissions.append({"filename": str(filename).strip(), "brand": str(brand).strip()})
    wb.close()
    return submissions


# === LOAD CONTENT ===
def load_content(filename, brand):
    """
    Load pre-reduced content for a (filename, brand) pair.
    Priority 1: Reducer output in REDUCED_DIR (algorithmically extracted)
    Priority 2: Full markdown in MD_DIR (fallback)
    """
    stem = Path(filename).stem

    # Priority 1: Reducer output (brand-specific)
    reduced_path = Path(REDUCED_DIR) / f"{stem}__{brand}.txt"
    if reduced_path.exists():
        text = reduced_path.read_text(encoding="utf-8")
        if text.strip():
            return text, "reduced"

    # Priority 2: Full markdown
    md_path = Path(MD_DIR) / f"{stem}.md"
    if md_path.exists():
        return md_path.read_text(encoding="utf-8"), "markdown-fallback"

    return None, "missing"


# === PARSE JSON ===
def parse_json_response(text):
    """Parse JSON from model response, handling markdown fences and preamble."""
    if text is None:
        raise ValueError("Response text is None")

    cleaned = text.strip()

    # Find JSON block inside markdown fences anywhere in response
    fence_match = re.search(r'```(?:json)?\s*\n(.*?)\n```', cleaned, re.DOTALL)
    if fence_match:
        cleaned = fence_match.group(1).strip()
    # Or find first { ... } block
    elif not cleaned.startswith("{"):
        brace_start = cleaned.find("{")
        brace_end = cleaned.rfind("}")
        if brace_start != -1 and brace_end != -1:
            cleaned = cleaned[brace_start:brace_end + 1]

    return json.loads(cleaned)


# === GEMINI CALL HELPER ===
def call_gemini(contents, system_instruction=None, json_output=False, max_retries=3):
    """Generic Gemini call with retry logic."""
    config_params = {
        "temperature": 0.1,
        "thinking_config": types.ThinkingConfig(thinking_level="HIGH"),
    }
    if system_instruction:
        config_params["system_instruction"] = system_instruction
    if json_output:
        config_params["response_mime_type"] = "application/json"

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=contents,
                config=types.GenerateContentConfig(**config_params),
            )

            if not response.candidates:
                raise ValueError(f"No candidates returned: {getattr(response, 'prompt_feedback', 'unknown')}")

            candidate = response.candidates[0]
            finish_reason = getattr(candidate, "finish_reason", None)
            if finish_reason is not None and "STOP" not in str(finish_reason):
                raise ValueError(f"Stopped: {finish_reason}")

            return response.text, None

        except Exception as e:
            print(f"    Error (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                continue

    return None, "max_retries_exceeded"


# === PARAMETER EXTRACTION ===
def extract_parameters(policy_text, brand):
    """Extract 12 PA parameters from policy text."""
    fda_age = FDA_PSO_AGE.get(brand.upper(), ">=18")

    user_prompt = f"""Extract the 12 PA parameters for the brand "{brand}" on the Plaque Psoriasis (PsO) indication from the following policy document.

FDA APPROVED AGE for {brand} in PsO: {fda_age}
(Use this if the policy says "FDA labelled age" without a number.)

POLICY DOCUMENT:
{policy_text}"""

    text, error = call_gemini(user_prompt, system_instruction=SYSTEM_PROMPT, json_output=True)
    if error:
        return None, error

    try:
        result = parse_json_response(text)
        return result, None
    except (json.JSONDecodeError, ValueError) as e:
        return None, f"JSON parse error: {e}"


# === VALIDATE AND NORMALIZE ===
def validate_and_normalize(params, brand):
    # Resolve FDA labelled age
    age = str(params.get("age", "NA")).strip()
    if "fda" in age.lower():
        params["age"] = FDA_PSO_AGE.get(brand.upper(), ">=18")

    # Binary fields
    for key in ["tb_test_required", "reauthorization_required"]:
        val = str(params.get(key, "NA")).strip().lower()
        if val in ("yes", "y", "true"):
            params[key] = "Yes"
        elif val in ("no", "n", "false"):
            params[key] = "No"
        else:
            params[key] = "NA"

    # Durations — extract leading digits, handle text like "12 months"
    for key in ["initial_authorization_duration", "reauthorization_duration"]:
        val = params.get(key, "")
        if val is None or str(val).strip().lower() in ("", "none", "null", "unspecified"):
            params[key] = "NA"
        elif isinstance(val, (int, float)):
            params[key] = str(int(val))
        else:
            cleaned = str(val).strip()
            if cleaned.lower() in ("na", "n/a"):
                params[key] = "NA"
            else:
                digit_match = re.match(r'(\d+)', cleaned)
                params[key] = digit_match.group(1) if digit_match else "NA"

    # Step counts — extract leading digits, normalize 0 to "No"
    for key in ["number_of_steps_through_brands", "number_of_steps_through_generic"]:
        val = params.get(key, "NA")
        val_str = str(val).strip().lower()
        if val_str in ("na", "n/a", "none", ""):
            params[key] = "NA"
        elif val_str in ("no", "0"):
            params[key] = "No"
        elif isinstance(val, (int, float)):
            params[key] = "No" if int(val) == 0 else str(int(val))
        else:
            cleaned = str(val).strip()
            digit_match = re.match(r'(\d+)', cleaned)
            if digit_match:
                num = int(digit_match.group(1))
                params[key] = "No" if num == 0 else str(num)

    # Phototherapy
    photo = str(params.get("step_through_phototherapy", "NA")).strip().lower()
    if photo in ("yes", "y"):
        params["step_through_phototherapy"] = "Yes"
    elif photo in ("no", "n"):
        params["step_through_phototherapy"] = "No"
    else:
        params["step_through_phototherapy"] = "NA"

    # Cross-field reauth
    reauth_dur = str(params.get("reauthorization_duration", "NA")).strip().lower()
    reauth_req = str(params.get("reauthorization_requirements", "NA")).strip().lower()
    if reauth_dur not in ("na", "n/a", "none", "") or \
       reauth_req not in ("na", "n/a", "none", ""):
        params["reauthorization_required"] = "Yes"

    return params


# === CSV COLUMNS ===
CSV_COLUMNS = [
    ("Filename", lambda row, p: row["filename"]),
    ("Brand", lambda row, p: row["brand"]),
    ("Age", lambda row, p: p.get("age", "")),
    ("Step Therapy Requirements Documented in Policy", lambda row, p: p.get("step_therapy_requirements", "")),
    ("Number of Steps through Brands", lambda row, p: p.get("number_of_steps_through_brands", "")),
    ("Number of Steps through Generic", lambda row, p: p.get("number_of_steps_through_generic", "")),
    ("Step through-Phototherapy", lambda row, p: p.get("step_through_phototherapy", "")),
    ("TB Test required", lambda row, p: p.get("tb_test_required", "")),
    ("Quantity Limits", lambda row, p: p.get("quantity_limits", "")),
    ("Specialist Types", lambda row, p: p.get("specialist_types", "")),
    ("Initial Authorization Duration(in-months)", lambda row, p: p.get("initial_authorization_duration", "")),
    ("Reauthorization Duration(in-months)", lambda row, p: p.get("reauthorization_duration", "")),
    ("Reauthorization Required", lambda row, p: p.get("reauthorization_required", "")),
    ("Reauthorization Requirements Documented in Policy", lambda row, p: p.get("reauthorization_requirements", "")),
    ("Access Score", lambda row, p: p.get("access_score", "")),
]


# === MAIN PIPELINE ===
def run_pipeline():
    submissions = load_submissions()
    print(f"Loaded {len(submissions)} submission rows")
    print(f"Reduced dir: {REDUCED_DIR}")
    print(f"Markdown dir: {MD_DIR}")

    if not submissions:
        print("ERROR: No valid submissions found.")
        return

    # Verify directories exist
    if not Path(REDUCED_DIR).exists():
        print(f"WARNING: {REDUCED_DIR} not found. All files will use markdown fallback.")
    if not Path(MD_DIR).exists():
        print(f"ERROR: {MD_DIR} not found. Cannot proceed.")
        return

    results = []
    errors = []
    source_counts = {"reduced": 0, "markdown-fallback": 0, "missing": 0}

    for i, row in enumerate(submissions):
        filename = row["filename"]
        brand = row["brand"]
        print(f"[{i+1}/{len(submissions)}] {filename} | {brand}")

        # Load content (reducer output preferred, markdown fallback)
        policy_text, source = load_content(filename, brand)
        if policy_text is None:
            print(f"  ERROR: content not found ({source})")
            errors.append({"filename": filename, "brand": brand, "error": "file_not_found"})
            results.append({"row": row, "params": {}})
            source_counts["missing"] += 1
            continue

        if not policy_text.strip():
            print(f"  ERROR: empty content ({source})")
            errors.append({"filename": filename, "brand": brand, "error": "empty_file"})
            results.append({"row": row, "params": {}})
            source_counts["empty"] = source_counts.get("empty", 0) + 1
            continue

        source_counts[source] = source_counts.get(source, 0) + 1
        print(f"  Source: {source} ({len(policy_text):,} chars)")

        # Extract parameters
        params, error = extract_parameters(policy_text, brand)
        if error:
            print(f"  EXTRACTION ERROR: {error}")
            errors.append({"filename": filename, "brand": brand, "error": error})
            results.append({"row": row, "params": {}})
            continue

        params = validate_and_normalize(params, brand)
        params["access_score"] = ""

        results.append({"row": row, "params": params})
        print(f"  OK: age={params.get('age')}, brands={params.get('number_of_steps_through_brands')}, "
              f"generic={params.get('number_of_steps_through_generic')}, "
              f"auth={params.get('initial_authorization_duration')}")

    # Write CSV
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([col[0] for col in CSV_COLUMNS])
        for r in results:
            writer.writerow([col[1](r["row"], r["params"]) for col in CSV_COLUMNS])

    # Summary
    print(f"\n{'='*60}")
    print(f"Done. {len(results) - len(errors)}/{len(results)} succeeded.")
    print(f"Sources: {source_counts}")
    print(f"Errors: {len(errors)}")
    if errors:
        for e in errors:
            print(f"  {e['filename']} | {e['brand']}: {e['error']}")
    print(f"Output: {OUTPUT_CSV}")

    # Log
    log_path = Path(OUTPUT_CSV).parent / "_extraction_log.json"
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(
            [{"filename": r["row"]["filename"], "brand": r["row"]["brand"], "params": r["params"]} for r in results],
            f, indent=2,
        )

run_pipeline()

In [ ]:
import shutil
from pathlib import Path

out = Path("work/reduced")
if out.exists():
    shutil.rmtree(out)
out.mkdir()
print("Cleared work/reduced/")

In [ ]:
"""
Document Reducer v2 — Extracts PsO-relevant sections from Docling structured documents.

Architecture:
  Phase 1: Find the brand's section boundaries
  Phase 2: Within those boundaries, find PsO criteria + utility sections

Three strategies for Phase 2: heading match, table search, cross-reference following.
No LLM calls. Pure algorithmic document understanding.
"""
import re
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument

# === BRAND ALIASES ===
BRAND_ALIASES = {
    "TREMFYA": ["tremfya", "guselkumab"],
    "STELARA": ["stelara", "ustekinumab"],
    "ENBREL": ["enbrel", "etanercept"],
    "AMJEVITA": ["amjevita", "adalimumab-atto", "adalimumab"],
    "COSENTYX": ["cosentyx", "secukinumab"],
    "SKYRIZI": ["skyrizi", "risankizumab"],
    "OTEZLA": ["otezla", "apremilast"],
    "SILIQ": ["siliq", "brodalumab"],
    "CIMZIA": ["cimzia", "certolizumab"],
    "BIMZELX": ["bimzelx", "bimekizumab"],
    "REMICADE": ["remicade", "infliximab"],
    "OTULFI": ["otulfi", "ustekinumab-aauz"],
    "ILUMYA": ["ilumya", "tildrakizumab"],
    "YESINTEK": ["yesintek", "ustekinumab-kfce"],
    "ACITRETIN": ["acitretin"],
}

PSO_PATTERNS = [
    re.compile(r'\bplaque\s+psoriasis\b', re.IGNORECASE),
    re.compile(r'\bPsO\b'),
    re.compile(r'\bpsoriasis\b', re.IGNORECASE),
]

UTILITY_PATTERNS = [
    # Duration and authorization
    re.compile(r'\bduration\s+of\s+approval\b', re.IGNORECASE),
    re.compile(r'\binitial\s+auth', re.IGNORECASE),
    re.compile(r'\binitial\s+approval\b', re.IGNORECASE),
    re.compile(r'\blength\s+of\s+auth', re.IGNORECASE),
    # Reauthorization / renewal
    re.compile(r'\breauthorization\b', re.IGNORECASE),
    re.compile(r'\brenewal\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bcontinuation\s+criteria\b', re.IGNORECASE),
    # Quantity limits
    re.compile(r'\bquantity\s+limit', re.IGNORECASE),
    re.compile(r'\bdosing\s+limit', re.IGNORECASE),
    re.compile(r'\bmax.*\s+dose\b', re.IGNORECASE),
    # Approval / denial
    re.compile(r'\bapproval\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bdenial\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bclinical\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bcoverage\s+criteria\b', re.IGNORECASE),
    # TB test / screening
    re.compile(r'\btuberculosis\b', re.IGNORECASE),
    re.compile(r'\bTB\s+test', re.IGNORECASE),
    re.compile(r'\bTB\s+screen', re.IGNORECASE),
    re.compile(r'\blatent\s+TB\b', re.IGNORECASE),
    re.compile(r'\bscreening\s+requirement', re.IGNORECASE),
    # Specialist / prescriber
    re.compile(r'\bspecialist', re.IGNORECASE),
    re.compile(r'\bprescriber\s+requirement', re.IGNORECASE),
    re.compile(r'\bprescribing\s+provider', re.IGNORECASE),
    re.compile(r'\bdermatolog', re.IGNORECASE),
    re.compile(r'\brheumatolog', re.IGNORECASE),
    # General requirements (often contain TB, specialist, etc.)
    re.compile(r'\bgeneral\s+requirement', re.IGNORECASE),
    re.compile(r'\bgeneral\s+criteria\b', re.IGNORECASE),
    re.compile(r'\buniversal\s+criteria\b', re.IGNORECASE),
    re.compile(r'\brequired\s+medical\s+info', re.IGNORECASE),
    re.compile(r'\bclinical\s+review\s+criteria\b', re.IGNORECASE),
    # Monitoring / testing
    re.compile(r'\bmonitoring\s+requirement', re.IGNORECASE),
    re.compile(r'\blab.*\s+requirement', re.IGNORECASE),
    re.compile(r'\bprecaution', re.IGNORECASE),
    re.compile(r'\bcontraindication', re.IGNORECASE),
]

# Structural sub-headers that repeat in every section — NOT drug-class boundaries
STRUCTURAL_HEADERS = re.compile(
    r'^\s*('
    r'Products?\s+Affected|'
    r'Goal\(?s?\)?:?|'
    r'Length\s+of\s+Authorization:?|'
    r'Requires\s+PA:?|'
    r'Covered\s+Alternatives:?|'
    r'Approval\s+Criteria|'
    r'Renewal\s+Criteria|'
    r'References?:?|'
    r'Table\s+\d|'
    r'PA\s+Criteria|'
    r'P&T|'
    r'Effective\s+Date|'
    r'Notes?:?'
    r')\s*$',
    re.IGNORECASE
)

GOTO_PATTERN = re.compile(r'Go\s+to\s+#\s*(\d+)', re.IGNORECASE)
CRITERIA_NUM_PATTERN = re.compile(r'^\s*(\d+)\.\s')


# === DOCUMENT INDEX ===

class DocumentIndex:
    def __init__(self, doc: DoclingDocument):
        self.doc = doc
        self.items = list(doc.iterate_items())
        self.headers = []
        self.tables = []
        self.text_items = []
        self._build_index()

    def _build_index(self):
        table_counter = 0
        for pos, (item, level) in enumerate(self.items):
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                self.headers.append((pos, text, level))
            elif item_type == 'TableItem':
                self.tables.append((pos, table_counter, item))
                table_counter += 1
            if text.strip():
                self.text_items.append((pos, text, level, item_type))

    def get_table_df(self, table_item):
        try:
            return table_item.export_to_dataframe(self.doc)
        except:
            try:
                return table_item.export_to_dataframe()
            except:
                return None

    def get_text_between(self, start, end):
        parts = []
        for pos, (item, level) in enumerate(self.items):
            if pos < start:
                continue
            if pos >= end:
                break
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                parts.append(f"\n## {text}\n")
            elif item_type == 'TableItem':
                df = self.get_table_df(item)
                if df is not None:
                    parts.append(df.to_markdown(index=False))
                    parts.append("")
            elif item_type == 'ListItem':
                parts.append(f"- {text}")
            elif text.strip():
                parts.append(text)
        return "\n".join(parts)

    def is_structural_header(self, text):
        return bool(STRUCTURAL_HEADERS.match(text.strip()))


# === PHASE 1: FIND BRAND SECTION ===

def find_brand_section(index: DocumentIndex, brand: str) -> tuple:
    """
    Find the document region that contains PA criteria for this brand.
    Returns (start_pos, end_pos). For small docs, returns (0, total_items).
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    total = len(index.items)

    # Small documents: entire document is the section
    if total < 200:
        return (0, total)

    # Strategy 1: Find brand in a section header
    brand_header_pos = None
    for pos, text, level in index.headers:
        text_lower = text.lower().strip()
        if any(a == text_lower or a in text_lower.split() for a in aliases):
            # Verify it's a drug-name header, not a structural one
            if not index.is_structural_header(text):
                brand_header_pos = pos
                break

    if brand_header_pos is not None:
        # Find the end: walk forward to the next non-structural header at same level
        start = brand_header_pos
        brand_level = None
        for pos, text, level in index.headers:
            if pos == brand_header_pos:
                brand_level = level
                continue
            if brand_level is not None and pos > brand_header_pos:
                if level <= brand_level and not index.is_structural_header(text):
                    return (start, pos)
        return (start, total)

    # Strategy 2: Find brand in a table cell, then use adaptive scoping
    for pos, table_idx, table_item in index.tables:
        df = index.get_table_df(table_item)
        if df is None:
            continue
        df_str = df.to_string().lower()
        if any(a in df_str for a in aliases):
            return find_section_scope(index, pos)

    # Strategy 3: Find brand in any text item
    for pos, text, level, item_type in index.text_items:
        if any(a in text.lower() for a in aliases):
            # Use table proximity if tables exist nearby
            nearby_tables = [(tp, ti, tt) for tp, ti, tt in index.tables
                             if abs(tp - pos) < 50]
            if nearby_tables:
                return find_section_scope(index, nearby_tables[0][0])
            # Otherwise take a generous window
            return (max(0, pos - 50), min(total, pos + 200))

    # Fallback: entire document
    print(f"  WARNING: Brand '{brand}' not found. Using full document.")
    return (0, total)


def find_section_scope(index: DocumentIndex, anchor_table_pos: int) -> tuple:
    """Adaptive gap-based section scoping around an anchor table."""
    table_positions = [pos for pos, _, _ in index.tables]
    if not table_positions:
        return (0, len(index.items))

    anchor_idx = None
    for i, (pos, _, _) in enumerate(index.tables):
        if pos == anchor_table_pos:
            anchor_idx = i
            break
    if anchor_idx is None:
        return (0, len(index.items))

    # Seed: 10 tables each direction
    seed_start = max(0, anchor_idx - 10)
    seed_end = min(len(table_positions), anchor_idx + 11)
    seed_positions = table_positions[seed_start:seed_end]

    gaps = [seed_positions[i] - seed_positions[i - 1] for i in range(1, len(seed_positions))]
    if not gaps:
        return (0, len(index.items))

    sorted_gaps = sorted(gaps)
    median_gap = sorted_gaps[len(sorted_gaps) // 2]
    initial_threshold = max(median_gap * 3, 10)

    # Walk backward
    start_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, 0, -1):
        if table_positions[i] - table_positions[i - 1] > initial_threshold:
            break
        start_pos = table_positions[i - 1]

    # Walk forward
    end_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, len(table_positions) - 1):
        if table_positions[i + 1] - table_positions[i] > initial_threshold:
            break
        end_pos = table_positions[i + 1]

    # Refine with 2× largest intra-cluster gap
    cluster = [p for p in table_positions if start_pos <= p <= end_pos]
    if len(cluster) > 1:
        intra_gaps = [cluster[i + 1] - cluster[i] for i in range(len(cluster) - 1)]
        refined = max(max(intra_gaps) * 2, 10)
        start_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, 0, -1):
            if table_positions[i] - table_positions[i - 1] > refined:
                break
            start_pos = table_positions[i - 1]
        end_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, len(table_positions) - 1):
            if table_positions[i + 1] - table_positions[i] > refined:
                break
            end_pos = table_positions[i + 1]

    # Extend to capture context
    end_pos = min(end_pos + 10, len(index.items))
    for pos, text, level in reversed(index.headers):
        if pos < start_pos and start_pos - pos < 30:
            start_pos = pos
            break

    return (start_pos, end_pos)


# === PHASE 2: FIND PsO CONTENT WITHIN SECTION ===

def find_pso_by_headings(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy A: Find PsO-related headings within the scoped section."""
    collected = []
    scoped_headers = [(pos, text, level) for pos, text, level in index.headers
                      if scope_start <= pos < scope_end]

    for i, (pos, text, level) in enumerate(scoped_headers):
        is_pso = any(p.search(text) for p in PSO_PATTERNS)
        is_utility = any(p.search(text) for p in UTILITY_PATTERNS)

        if is_pso or is_utility:
            # End at next header at same or higher level within scope
            end_pos = scope_end
            for j in range(i + 1, len(scoped_headers)):
                next_pos, next_text, next_level = scoped_headers[j]
                if next_level <= level and not index.is_structural_header(next_text):
                    end_pos = next_pos
                    break
            collected.append((pos, end_pos))

    return collected


def find_pso_in_tables(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy B: Find tables containing PsO keywords within the scoped section.
    
    Filters:
    - Skip administrative tables (drug listings, NDC tables, dosing grids)
    - Small tables (<20 rows): include if PsO keyword appears anywhere
    - Criteria-structured tables: include if PsO keyword found
    - Large non-criteria tables: require both brand AND PsO keywords
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    collected = []

    # Column names that indicate a PA criteria table
    CRITERIA_TABLE_PATTERNS = re.compile(
        r'PA\s+Criteria|Criteria\s+Details|Approval|Required\s+Medical|'
        r'Coverage\s+Duration|Renewal|Exclusion|Prescriber|Restriction|'
        r'Authorization|Step\s+Therapy|Clinical\s+Criteria',
        re.IGNORECASE
    )

    # Column names that indicate an administrative/formulary listing table — skip these
    ADMIN_TABLE_PATTERNS = re.compile(
        r'Strength|NDC|Targeted\s+NDC|Day\s+Supply|Dose\s+Form|'
        r'Client\s+Formulary|GPI|HICL|Package\s+Size|Route\s+of\s+Admin|'
        r'AWP|WAC|Unit\s+Price|Wholesale|Generic\s+Name.*Brand\s+Name|'
        r'Brand\s+Name.*Generic\s+Name|HCPCS|J-?Code|CPT',
        re.IGNORECASE
    )

    for pos, table_idx, table_item in index.tables:
        if pos < scope_start or pos >= scope_end:
            continue
        df = index.get_table_df(table_item)
        if df is None:
            continue

        num_rows = len(df)
        col_str = ' '.join(str(c) for c in df.columns)

        # Skip administrative tables entirely
        if ADMIN_TABLE_PATTERNS.search(col_str):
            continue

        # Check if this is a PA criteria table by column names
        is_criteria_table = bool(CRITERIA_TABLE_PATTERNS.search(col_str))

        # Small table or criteria-structured table: include if PsO keyword found
        if num_rows < 20 or is_criteria_table:
            df_str = df.to_string().lower()
            if any(p.search(df_str) for p in PSO_PATTERNS):
                collected.append((max(scope_start, pos - 2), min(scope_end, pos + 2)))

        # Large table: only include if BOTH brand AND PsO found
        else:
            df_str = df.to_string().lower()
            has_brand = any(a in df_str for a in aliases)
            has_pso = any(p.search(df_str) for p in PSO_PATTERNS)
            if has_brand and has_pso:
                collected.append((max(scope_start, pos - 2), min(scope_end, pos + 2)))

    return collected


def follow_cross_references(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy C: Follow 'Go to #X' routing chains within the scoped section."""
    collected = []

    # Get all tables in scope
    scope_tables = []
    for pos, table_idx, table_item in index.tables:
        if scope_start <= pos <= scope_end:
            df = index.get_table_df(table_item)
            if df is not None:
                scope_tables.append((pos, table_idx, df))

    if not scope_tables:
        return collected

    # Find PsO routing entries: tables with both PsO keyword and "Go to #X"
    refs_to_follow = set()
    initial_tables = set()
    for tbl_pos, tbl_idx, tbl_df in scope_tables:
        for row_idx in range(len(tbl_df)):
            row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
            row_lower = row_text.lower()
            has_pso = any(p.search(row_lower) for p in PSO_PATTERNS)
            refs_in_row = GOTO_PATTERN.findall(row_text)
            if has_pso and refs_in_row:
                initial_tables.add(tbl_pos)
                refs_to_follow.update(int(r) for r in refs_in_row)

    if not refs_to_follow:
        return collected

    # BFS: follow references
    visited = set()
    tables_to_include = set(initial_tables)
    queue = list(refs_to_follow)

    while queue:
        target_num = queue.pop(0)
        if target_num in visited:
            continue
        visited.add(target_num)

        for tbl_pos, tbl_idx, tbl_df in scope_tables:
            for row_idx in range(len(tbl_df)):
                cell_0 = str(tbl_df.iloc[row_idx, 0]).strip()

                # Match criteria number in first cell
                match_bare = re.match(r'^(\d+)\.?\s*$', cell_0)
                match_prefix = CRITERIA_NUM_PATTERN.match(cell_0)

                found_num = None
                if match_bare:
                    found_num = int(match_bare.group(1))
                elif match_prefix:
                    found_num = int(match_prefix.group(1))

                # Also check column 1 for 4-column tables where col 0 is just the number
                if found_num is None and tbl_df.shape[1] >= 3:
                    cell_1 = str(tbl_df.iloc[row_idx, 1] if tbl_df.shape[1] > 1 else '').strip()
                    match_c1 = CRITERIA_NUM_PATTERN.match(cell_1)
                    if match_c1:
                        found_num = int(match_c1.group(1))

                if found_num == target_num:
                    tables_to_include.add(tbl_pos)
                    row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
                    new_refs = GOTO_PATTERN.findall(row_text)
                    for r in new_refs:
                        r_int = int(r)
                        if r_int not in visited:
                            queue.append(r_int)

    for tbl_pos in tables_to_include:
        collected.append((max(scope_start, tbl_pos - 2), min(scope_end, tbl_pos + 2)))

    return collected


# === MERGE AND SERIALIZE ===

def merge_ranges(ranges):
    if not ranges:
        return []
    sorted_r = sorted(ranges, key=lambda x: x[0])
    merged = [sorted_r[0]]
    for start, end in sorted_r[1:]:
        ps, pe = merged[-1]
        if start <= pe + 3:
            merged[-1] = (ps, max(pe, end))
        else:
            merged.append((start, end))
    return merged


def serialize(index, ranges):
    parts = []
    seen = set()
    for start, end in ranges:
        if start not in seen:
            parts.append(index.get_text_between(start, end))
            seen.add(start)
    return "\n\n---\n\n".join(parts)


# === MAIN REDUCER ===

MIN_OUTPUT_CHARS = 100
MIN_REDUCTION_RATIO = 0.30  # If output < 30% of brand section, widening triggers

def reduce_document(doc: DoclingDocument, brand: str) -> str:
    index = DocumentIndex(doc)
    total_items = len(index.items)
    print(f"  Reducer: {total_items} items, {len(index.tables)} tables, {len(index.headers)} headers")

    # Phase 1: Find brand's section
    scope_start, scope_end = find_brand_section(index, brand)
    scope_size = scope_end - scope_start
    print(f"  Brand section: items {scope_start}-{scope_end} ({scope_size} items)")

    # Confidence check: if scope is suspiciously narrow for a large document,
    # widen to full document and let Phase 2 search everywhere.
    if scope_size < 10 and total_items > 200:
        print(f"  Scope too narrow ({scope_size} items in {total_items}-item doc). Widening to full document.")
        scope_start, scope_end = 0, total_items
        scope_size = total_items

    # Get the full brand section text for comparison
    brand_section_text = index.get_text_between(scope_start, scope_end)
    brand_section_len = len(brand_section_text)

    # Phase 2: Find PsO content within section
    ranges_a = find_pso_by_headings(index, scope_start, scope_end, brand)
    ranges_b = find_pso_in_tables(index, scope_start, scope_end, brand)
    ranges_c = follow_cross_references(index, scope_start, scope_end, brand)
    print(f"  Strategy A (headings): {len(ranges_a)} ranges")
    print(f"  Strategy B (tables): {len(ranges_b)} ranges")
    print(f"  Strategy C (cross-ref): {len(ranges_c)} ranges")

    # If no PsO-specific content found, use raw markdown to preserve formatting
    all_ranges = ranges_a + ranges_b + ranges_c
    if not all_ranges:
        print(f"  No PsO-specific content found. Using raw markdown.")
        result = doc.export_to_markdown()
        full_len = len(result)
        print(f"  Output: {len(result):,} chars (0% reduction from {full_len:,})")
        return result

    merged = merge_ranges(all_ranges)
    print(f"  Merged: {len(merged)} ranges")

    result = serialize(index, merged)

    # Adaptive widening: if the extracted content is less than 30% of the brand
    # section, the reduction was too aggressive. Fall back to raw markdown which
    # preserves the original formatting the LLM handles well.
    if brand_section_len > 0 and len(result) < brand_section_len * MIN_REDUCTION_RATIO:
        print(f"  Reduction too aggressive ({len(result):,} < {int(brand_section_len * MIN_REDUCTION_RATIO):,}). "
              f"Widening to raw markdown.")
        result = doc.export_to_markdown()

    # Final recovery: if output is still too short, return full markdown
    if len(result) < MIN_OUTPUT_CHARS and total_items > 50:
        print(f"  Output too short ({len(result)} chars). Falling back to full markdown.")
        result = doc.export_to_markdown()

    full_len = len(doc.export_to_markdown())
    reduction = (1 - len(result) / full_len) * 100 if full_len else 0
    print(f"  Output: {len(result):,} chars ({reduction:.0f}% reduction from {full_len:,})")

    return result

In [ ]:
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument
import openpyxl

OBJ_DIR = "work/doc_objects"
EXCEL_PATH = "PA_Business_Rules.xlsx"
OUT_DIR = Path("work/reduced")

wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
ws = wb['Submissions']
submissions = []
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
    if row[0] and row[1]:
        submissions.append((str(row[0]).strip(), str(row[1]).strip()))
wb.close()
print(f"Loaded {len(submissions)} submissions")

doc_cache = {}
for i, (filename, brand) in enumerate(submissions):
    stem = Path(filename).stem
    print(f"\n[{i+1}/{len(submissions)}] {filename} | {brand}")

    if stem not in doc_cache:
        obj_path = Path(OBJ_DIR) / f"{stem}.json"
        if not obj_path.exists():
            print(f"  ERROR: {obj_path} not found")
            continue
        doc_cache[stem] = DoclingDocument.load_from_json(obj_path)

    doc = doc_cache[stem]
    result = reduce_document(doc, brand)

    out_path = OUT_DIR / f"{stem}__{brand}.txt"
    out_path.write_text(result, encoding="utf-8")

print(f"\nDone. Outputs in {OUT_DIR}/")

In [ ]:
"""
PA Policy Parameter Extraction Pipeline — Integrated with Document Reducer.

Architecture:
  1. Document Reducer (pre-computed): Docling structured output → algorithmic
     section extraction → focused text per (filename, brand) pair.
  2. This pipeline: reads reduced text → Gemini extraction → validation → CSV.

The reducer handles all document structures locally (heading match, table search,
cross-reference following). The LLM only does the hard reasoning: step counting,
AND/OR logic, phototherapy classification.

Inputs:
  work/reduced/   — Reducer output (primary)
  work/md_output/    — Full markdown (fallback if reducer output missing)
  PA_Business_Rules.xlsx — Submission list

Outputs:
  work/result.csv           — 79-row extraction results
  work/_extraction_log.json — Detailed extraction log

Run: GEMINI_API_KEY=your-key python pipeline_integrated.py
"""
import os
import json
import csv
import re
from pathlib import Path
from google import genai
from google.genai import types
import openpyxl

# === CONFIG ===
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    raise RuntimeError("Set GEMINI_API_KEY env var.")

REDUCED_DIR = os.environ.get("REDUCED_DIR", "work/reduced")
MD_DIR = os.environ.get("MD_DIR", "work/md_output")
EXCEL_PATH = os.environ.get("EXCEL_PATH", "PA_Business_Rules.xlsx")
OUTPUT_CSV = os.environ.get("OUTPUT_CSV", "work/result.csv")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_ID = "gemini-2.5-flash"

# === FDA APPROVED AGES FOR PsO BY BRAND ===
FDA_PSO_AGE = {
    "STELARA": ">=6",
    "TREMFYA": ">=6",
    "ENBREL": ">=4",
    "COSENTYX": ">=6",
    "SKYRIZI": ">=18",
    "OTEZLA": ">=6",
    "BIMZELX": ">=18",
    "SILIQ": ">=18",
    "ILUMYA": ">=18",
    "CIMZIA": ">=18",
    "AMJEVITA": ">=18",
    "REMICADE": ">=18",
    "OTULFI": ">=6",
    "YESINTEK": ">=6",
    "ACITRETIN": ">=18",
}

# === EXTRACTION PROMPT ===
SYSTEM_PROMPT = """You are a pharmaceutical prior authorization policy analyst. You extract structured parameters from PA policy documents for the Plaque Psoriasis (PsO) indication.

You will receive a policy document (or relevant excerpt) and a target brand name. Extract the 12 parameters defined below for that brand's PsO coverage ONLY.

=== CRITICAL DISTINCTION: "NA" vs "No" ===

- "NA" = The information COULD NOT BE FOUND in the policy document. The policy simply does not mention this parameter for PsO. Data is missing/absent.
- "No" = The policy EXPLICITLY STATES there is no restriction or requirement.

If a parameter is not mentioned anywhere in the PsO section or universal criteria, output "NA" — NOT "No".

=== CRITICAL RULES ===

INDICATION FILTERING:
- Extract ONLY for Plaque Psoriasis (PsO). Ignore all other indications.
- If the policy separates "moderate-to-severe" and "severe" PsO, extract ONLY moderate-to-severe criteria.
- SPECIALIST TYPES: Only capture specialists listed in the PsO section. Do NOT include Gastroenterologist or other specialists from CD/UC sections.
- STEP THERAPY: Only count steps from the PsO section and universal/general criteria. Do NOT count steps from other indications.
- SELF-REFERENCING: If the target brand appears as one of several options in a step therapy requirement, exclude it from the option list but still count the step — the patient would use one of the other options. Only exclude the entire step if the target brand is the sole option (a drug cannot be the only prerequisite to access itself). This can occur due to document redlining artifacts.

UNIVERSAL vs INDICATION-SPECIFIC CRITERIA:
- Some policies have "General Requirements" or "Universal Criteria" that apply across all indications, PLUS separate PsO-specific criteria.
- These two sets combine with AND — the patient must satisfy BOTH.
- When counting steps, take the UNION of universal steps and PsO-specific steps.

STEP COUNTING — LEAST RESTRICTIVE PATH:
- OR statements → take the path with FEWER total steps (least restrictive).
- AND statements → patient must complete ALL (union).
- "Tried Drug A OR Drug B" = 1 step, NOT 2.
- "Tried Drug A AND Drug B" = 2 steps.
- "must have tried ONE of the following: [list of 8 drugs]" = 1 step, not 8.
- "must have tried TWO of the following: [list]" = 2 steps.
- Maximum realistic: branded steps rarely exceed 4, generic rarely exceed 3.

STEP CLASSIFICATION:
- Branded/biologic: Named biologic drugs, TNF inhibitors, IL-inhibitors, targeted synthetic drugs, biosimilars.
- Generic: Topical agents, conventional systemics (methotrexate, cyclosporine, acitretin), non-biologic drugs.
- Phototherapy: EXCLUDED from both counts — captured separately.

PHOTOTHERAPY:
- "Yes" = mandatory AND with no alternative path.
- "No" = mentioned but optional (in an OR).
- "NA" = not mentioned or no step criteria.

=== PARAMETER DEFINITIONS ===

1. AGE — Explicit threshold (e.g., ">=18"). "adults" → ">=18". "FDA labeled age" → use the FDA age provided. Not mentioned → "NA". Explicitly no restriction → "No". Two ages → youngest.

2. STEP THERAPY REQUIREMENTS — All step therapy language for PsO. "NA" if none.

3. NUMBER OF STEPS THROUGH BRANDS — Integer as string, "No" if criteria exist but no branded steps, "NA" if no criteria.

4. NUMBER OF STEPS THROUGH GENERIC — Same logic.

5. STEP THROUGH PHOTOTHERAPY — "Yes"=mandatory AND. "No"=optional OR. "NA"=not mentioned.

6. TB TEST REQUIRED — "Yes"/"No"/"NA".

7. QUANTITY LIMITS — Only explicitly labeled "quantity limit". Not "dosage"/"dosing". "NA" if not mentioned.

8. SPECIALIST TYPES — PsO only. "NA" if not mentioned.

9. INITIAL AUTHORIZATION DURATION (MONTHS) — Convert weeks/days/years to months. "NA" if not mentioned.

10. REAUTHORIZATION DURATION (MONTHS) — Same. "NA" if not mentioned.

11. REAUTHORIZATION REQUIRED — "Yes"/"No"/"NA". If reauth duration or requirements is non-NA, must be "Yes".

12. REAUTHORIZATION REQUIREMENTS — Text or "NA".

=== FEW-SHOT EXAMPLES ===

EXAMPLE 1 — Phototherapy in OR, no age mentioned
{
  "age": "NA",
  "step_therapy_requirements": "inadequate response to phototherapy or methotrexate/cyclosporine/acitretin",
  "number_of_steps_through_brands": "No",
  "number_of_steps_through_generic": "1",
  "step_through_phototherapy": "No",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "NA",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "NA",
  "reauthorization_required": "No",
  "reauthorization_requirements": "NA"
}

EXAMPLE 2 — Branded + generic steps with AND
{
  "age": ">=18",
  "step_therapy_requirements": "ONE topical AND TWO biologics/targeted synthetics",
  "number_of_steps_through_brands": "2",
  "number_of_steps_through_generic": "1",
  "step_through_phototherapy": "NA",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "Dermatologist",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "12",
  "reauthorization_required": "Yes",
  "reauthorization_requirements": "Positive clinical response: reduction in BSA or improvement in symptoms"
}

EXAMPLE 3 — OR logic, least restrictive path
{
  "age": ">=6",
  "step_therapy_requirements": "(topical + methotrexate) OR (targeted immunomodulator)",
  "number_of_steps_through_brands": "No",
  "number_of_steps_through_generic": "2",
  "step_through_phototherapy": "NA",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "Dermatologist",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "12",
  "reauthorization_required": "Yes",
  "reauthorization_requirements": "Positive clinical response AND not combined with another targeted immunomodulator"
}

=== PsO BRAND REFERENCE ===

BRANDED/BIOLOGIC: Amjevita, Avsola, Bimzelx, Cimzia, Cosentyx, Cyltezo, Enbrel, Humira, Hyrimoz, Idacio, Ilumya, Inflectra, Hulio, Otezla, Remicade, Renflexis, Siliq, Skyrizi, Sotyktu, Stelara, Taltz, Tremfya, Vtama, Yuflyma, Yusimry, Zoryve, Wezlana, Selarsdi, Yesintek, Psychiva/Quallent, Steqeyma, Otulfi

GENERIC/CONVENTIONAL: Acitretin, Cyclosporine, Methotrexate, topical corticosteroids, vitamin D analogs, calcineurin inhibitors, tazarotene, coal tar, anthralin

=== OUTPUT FORMAT ===

Return a JSON object with exactly the 12 fields. No extra fields or commentary."""


# === LOAD SUBMISSIONS ===
def load_submissions():
    wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
    ws = wb['Submissions']
    submissions = []
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
        filename, brand = row[0], row[1]
        if not filename or not brand:
            continue
        submissions.append({"filename": str(filename).strip(), "brand": str(brand).strip()})
    wb.close()
    return submissions


# === LOAD CONTENT ===
def load_content(filename, brand):
    """
    Load pre-reduced content for a (filename, brand) pair.
    Priority 1: Reducer output in REDUCED_DIR (algorithmically extracted)
    Priority 2: Full markdown in MD_DIR (fallback)
    """
    stem = Path(filename).stem

    # Priority 1: Reducer output (brand-specific)
    reduced_path = Path(REDUCED_DIR) / f"{stem}__{brand}.txt"
    if reduced_path.exists():
        text = reduced_path.read_text(encoding="utf-8")
        if text.strip():
            return text, "reduced"

    # Priority 2: Full markdown
    md_path = Path(MD_DIR) / f"{stem}.md"
    if md_path.exists():
        return md_path.read_text(encoding="utf-8"), "markdown-fallback"

    return None, "missing"


# === PARSE JSON ===
def parse_json_response(text):
    """Parse JSON from model response, handling markdown fences and preamble."""
    if text is None:
        raise ValueError("Response text is None")

    cleaned = text.strip()

    # Find JSON block inside markdown fences anywhere in response
    fence_match = re.search(r'```(?:json)?\s*\n(.*?)\n```', cleaned, re.DOTALL)
    if fence_match:
        cleaned = fence_match.group(1).strip()
    # Or find first { ... } block
    elif not cleaned.startswith("{"):
        brace_start = cleaned.find("{")
        brace_end = cleaned.rfind("}")
        if brace_start != -1 and brace_end != -1:
            cleaned = cleaned[brace_start:brace_end + 1]

    return json.loads(cleaned)


# === GEMINI CALL HELPER ===
def call_gemini(contents, system_instruction=None, json_output=False, max_retries=3):
    """Generic Gemini call with retry logic."""
    config_params = {
        "temperature": 0.1,
        "thinking_config": types.ThinkingConfig(thinking_level="HIGH"),
    }
    if system_instruction:
        config_params["system_instruction"] = system_instruction
    if json_output:
        config_params["response_mime_type"] = "application/json"

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=contents,
                config=types.GenerateContentConfig(**config_params),
            )

            if not response.candidates:
                raise ValueError(f"No candidates returned: {getattr(response, 'prompt_feedback', 'unknown')}")

            candidate = response.candidates[0]
            finish_reason = getattr(candidate, "finish_reason", None)
            if finish_reason is not None and "STOP" not in str(finish_reason):
                raise ValueError(f"Stopped: {finish_reason}")

            return response.text, None

        except Exception as e:
            print(f"    Error (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                continue

    return None, "max_retries_exceeded"


# === PARAMETER EXTRACTION ===
def extract_parameters(policy_text, brand):
    """Extract 12 PA parameters from policy text."""
    fda_age = FDA_PSO_AGE.get(brand.upper(), ">=18")

    user_prompt = f"""Extract the 12 PA parameters for the brand "{brand}" on the Plaque Psoriasis (PsO) indication from the following policy document.

FDA APPROVED AGE for {brand} in PsO: {fda_age}
(Use this if the policy says "FDA labelled age" without a number.)

POLICY DOCUMENT:
{policy_text}"""

    text, error = call_gemini(user_prompt, system_instruction=SYSTEM_PROMPT, json_output=True)
    if error:
        return None, error

    try:
        result = parse_json_response(text)
        return result, None
    except (json.JSONDecodeError, ValueError) as e:
        return None, f"JSON parse error: {e}"


# === VALIDATE AND NORMALIZE ===
def validate_and_normalize(params, brand):
    # Resolve FDA labelled age
    age = str(params.get("age", "NA")).strip()
    if "fda" in age.lower():
        params["age"] = FDA_PSO_AGE.get(brand.upper(), ">=18")

    # Binary fields
    for key in ["tb_test_required", "reauthorization_required"]:
        val = str(params.get(key, "NA")).strip().lower()
        if val in ("yes", "y", "true"):
            params[key] = "Yes"
        elif val in ("no", "n", "false"):
            params[key] = "No"
        else:
            params[key] = "NA"

    # Durations — extract leading digits, handle text like "12 months"
    for key in ["initial_authorization_duration", "reauthorization_duration"]:
        val = params.get(key, "")
        if val is None or str(val).strip().lower() in ("", "none", "null", "unspecified"):
            params[key] = "NA"
        elif isinstance(val, (int, float)):
            params[key] = str(int(val))
        else:
            cleaned = str(val).strip()
            if cleaned.lower() in ("na", "n/a"):
                params[key] = "NA"
            else:
                digit_match = re.match(r'(\d+)', cleaned)
                params[key] = digit_match.group(1) if digit_match else "NA"

    # Step counts — extract leading digits, normalize 0 to "No"
    for key in ["number_of_steps_through_brands", "number_of_steps_through_generic"]:
        val = params.get(key, "NA")
        val_str = str(val).strip().lower()
        if val_str in ("na", "n/a", "none", ""):
            params[key] = "NA"
        elif val_str in ("no", "0"):
            params[key] = "No"
        elif isinstance(val, (int, float)):
            params[key] = "No" if int(val) == 0 else str(int(val))
        else:
            cleaned = str(val).strip()
            digit_match = re.match(r'(\d+)', cleaned)
            if digit_match:
                num = int(digit_match.group(1))
                params[key] = "No" if num == 0 else str(num)

    # Phototherapy
    photo = str(params.get("step_through_phototherapy", "NA")).strip().lower()
    if photo in ("yes", "y"):
        params["step_through_phototherapy"] = "Yes"
    elif photo in ("no", "n"):
        params["step_through_phototherapy"] = "No"
    else:
        params["step_through_phototherapy"] = "NA"

    # Cross-field reauth
    reauth_dur = str(params.get("reauthorization_duration", "NA")).strip().lower()
    reauth_req = str(params.get("reauthorization_requirements", "NA")).strip().lower()
    if reauth_dur not in ("na", "n/a", "none", "") or \
       reauth_req not in ("na", "n/a", "none", ""):
        params["reauthorization_required"] = "Yes"

    return params


# === CSV COLUMNS ===
CSV_COLUMNS = [
    ("Filename", lambda row, p: row["filename"]),
    ("Brand", lambda row, p: row["brand"]),
    ("Age", lambda row, p: p.get("age", "")),
    ("Step Therapy Requirements Documented in Policy", lambda row, p: p.get("step_therapy_requirements", "")),
    ("Number of Steps through Brands", lambda row, p: p.get("number_of_steps_through_brands", "")),
    ("Number of Steps through Generic", lambda row, p: p.get("number_of_steps_through_generic", "")),
    ("Step through-Phototherapy", lambda row, p: p.get("step_through_phototherapy", "")),
    ("TB Test required", lambda row, p: p.get("tb_test_required", "")),
    ("Quantity Limits", lambda row, p: p.get("quantity_limits", "")),
    ("Specialist Types", lambda row, p: p.get("specialist_types", "")),
    ("Initial Authorization Duration(in-months)", lambda row, p: p.get("initial_authorization_duration", "")),
    ("Reauthorization Duration(in-months)", lambda row, p: p.get("reauthorization_duration", "")),
    ("Reauthorization Required", lambda row, p: p.get("reauthorization_required", "")),
    ("Reauthorization Requirements Documented in Policy", lambda row, p: p.get("reauthorization_requirements", "")),
    ("Access Score", lambda row, p: p.get("access_score", "")),
]


# === MAIN PIPELINE ===
def run_pipeline():
    submissions = load_submissions()
    print(f"Loaded {len(submissions)} submission rows")
    print(f"Reduced dir: {REDUCED_DIR}")
    print(f"Markdown dir: {MD_DIR}")

    if not submissions:
        print("ERROR: No valid submissions found.")
        return

    # Verify directories exist
    if not Path(REDUCED_DIR).exists():
        print(f"WARNING: {REDUCED_DIR} not found. All files will use markdown fallback.")
    if not Path(MD_DIR).exists():
        print(f"ERROR: {MD_DIR} not found. Cannot proceed.")
        return

    results = []
    errors = []
    source_counts = {"reduced": 0, "markdown-fallback": 0, "missing": 0}

    for i, row in enumerate(submissions):
        filename = row["filename"]
        brand = row["brand"]
        print(f"[{i+1}/{len(submissions)}] {filename} | {brand}")

        # Load content (reducer output preferred, markdown fallback)
        policy_text, source = load_content(filename, brand)
        if policy_text is None:
            print(f"  ERROR: content not found ({source})")
            errors.append({"filename": filename, "brand": brand, "error": "file_not_found"})
            results.append({"row": row, "params": {}})
            source_counts["missing"] += 1
            continue

        if not policy_text.strip():
            print(f"  ERROR: empty content ({source})")
            errors.append({"filename": filename, "brand": brand, "error": "empty_file"})
            results.append({"row": row, "params": {}})
            source_counts["empty"] = source_counts.get("empty", 0) + 1
            continue

        source_counts[source] = source_counts.get(source, 0) + 1
        print(f"  Source: {source} ({len(policy_text):,} chars)")

        # Extract parameters
        params, error = extract_parameters(policy_text, brand)
        if error:
            print(f"  EXTRACTION ERROR: {error}")
            errors.append({"filename": filename, "brand": brand, "error": error})
            results.append({"row": row, "params": {}})
            continue

        params = validate_and_normalize(params, brand)
        params["access_score"] = ""

        results.append({"row": row, "params": params})
        print(f"  OK: age={params.get('age')}, brands={params.get('number_of_steps_through_brands')}, "
              f"generic={params.get('number_of_steps_through_generic')}, "
              f"auth={params.get('initial_authorization_duration')}")

    # Write CSV
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([col[0] for col in CSV_COLUMNS])
        for r in results:
            writer.writerow([col[1](r["row"], r["params"]) for col in CSV_COLUMNS])

    # Summary
    print(f"\n{'='*60}")
    print(f"Done. {len(results) - len(errors)}/{len(results)} succeeded.")
    print(f"Sources: {source_counts}")
    print(f"Errors: {len(errors)}")
    if errors:
        for e in errors:
            print(f"  {e['filename']} | {e['brand']}: {e['error']}")
    print(f"Output: {OUTPUT_CSV}")

    # Log
    log_path = Path(OUTPUT_CSV).parent / "_extraction_log.json"
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(
            [{"filename": r["row"]["filename"], "brand": r["row"]["brand"], "params": r["params"]} for r in results],
            f, indent=2,
        )

run_pipeline()

In [ ]:
import shutil
from pathlib import Path

out = Path("work/reduced")
if out.exists():
    shutil.rmtree(out)
out.mkdir()
print("Cleared work/reduced/")

In [ ]:
"""
Document Reducer v2 — Extracts PsO-relevant sections from Docling structured documents.

Architecture:
  Phase 1: Find the brand's section boundaries
  Phase 2: Within those boundaries, find PsO criteria + utility sections

Three strategies for Phase 2: heading match, table search, cross-reference following.
No LLM calls. Pure algorithmic document understanding.
"""
import re
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument

# === BRAND ALIASES ===
BRAND_ALIASES = {
    "TREMFYA": ["tremfya", "guselkumab"],
    "STELARA": ["stelara", "ustekinumab"],
    "ENBREL": ["enbrel", "etanercept"],
    "AMJEVITA": ["amjevita", "adalimumab-atto", "adalimumab"],
    "COSENTYX": ["cosentyx", "secukinumab"],
    "SKYRIZI": ["skyrizi", "risankizumab"],
    "OTEZLA": ["otezla", "apremilast"],
    "SILIQ": ["siliq", "brodalumab"],
    "CIMZIA": ["cimzia", "certolizumab"],
    "BIMZELX": ["bimzelx", "bimekizumab"],
    "REMICADE": ["remicade", "infliximab"],
    "OTULFI": ["otulfi", "ustekinumab-aauz"],
    "ILUMYA": ["ilumya", "tildrakizumab"],
    "YESINTEK": ["yesintek", "ustekinumab-kfce"],
    "ACITRETIN": ["acitretin"],
}

PSO_PATTERNS = [
    re.compile(r'\bplaque\s+psoriasis\b', re.IGNORECASE),
    re.compile(r'\bPsO\b'),
    re.compile(r'\bpsoriasis\b', re.IGNORECASE),
]

UTILITY_PATTERNS = [
    # Duration and authorization
    re.compile(r'\bduration\s+of\s+approval\b', re.IGNORECASE),
    re.compile(r'\binitial\s+auth', re.IGNORECASE),
    re.compile(r'\binitial\s+approval\b', re.IGNORECASE),
    re.compile(r'\blength\s+of\s+auth', re.IGNORECASE),
    # Reauthorization / renewal
    re.compile(r'\breauthorization\b', re.IGNORECASE),
    re.compile(r'\brenewal\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bcontinuation\s+criteria\b', re.IGNORECASE),
    # Quantity limits
    re.compile(r'\bquantity\s+limit', re.IGNORECASE),
    re.compile(r'\bdosing\s+limit', re.IGNORECASE),
    re.compile(r'\bmax.*\s+dose\b', re.IGNORECASE),
    # Approval / denial
    re.compile(r'\bapproval\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bdenial\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bclinical\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bcoverage\s+criteria\b', re.IGNORECASE),
    # TB test / screening
    re.compile(r'\btuberculosis\b', re.IGNORECASE),
    re.compile(r'\bTB\s+test', re.IGNORECASE),
    re.compile(r'\bTB\s+screen', re.IGNORECASE),
    re.compile(r'\blatent\s+TB\b', re.IGNORECASE),
    re.compile(r'\bscreening\s+requirement', re.IGNORECASE),
    # Specialist / prescriber
    re.compile(r'\bspecialist', re.IGNORECASE),
    re.compile(r'\bprescriber\s+requirement', re.IGNORECASE),
    re.compile(r'\bprescribing\s+provider', re.IGNORECASE),
    re.compile(r'\bdermatolog', re.IGNORECASE),
    re.compile(r'\brheumatolog', re.IGNORECASE),
    # General requirements (often contain TB, specialist, etc.)
    re.compile(r'\bgeneral\s+requirement', re.IGNORECASE),
    re.compile(r'\bgeneral\s+criteria\b', re.IGNORECASE),
    re.compile(r'\buniversal\s+criteria\b', re.IGNORECASE),
    re.compile(r'\brequired\s+medical\s+info', re.IGNORECASE),
    re.compile(r'\bclinical\s+review\s+criteria\b', re.IGNORECASE),
    # Monitoring / testing
    re.compile(r'\bmonitoring\s+requirement', re.IGNORECASE),
    re.compile(r'\blab.*\s+requirement', re.IGNORECASE),
    re.compile(r'\bprecaution', re.IGNORECASE),
    re.compile(r'\bcontraindication', re.IGNORECASE),
]

# Structural sub-headers that repeat in every section — NOT drug-class boundaries
STRUCTURAL_HEADERS = re.compile(
    r'^\s*('
    r'Products?\s+Affected|'
    r'Goal\(?s?\)?:?|'
    r'Length\s+of\s+Authorization:?|'
    r'Requires\s+PA:?|'
    r'Covered\s+Alternatives:?|'
    r'Approval\s+Criteria|'
    r'Renewal\s+Criteria|'
    r'References?:?|'
    r'Table\s+\d|'
    r'PA\s+Criteria|'
    r'P&T|'
    r'Effective\s+Date|'
    r'Notes?:?'
    r')\s*$',
    re.IGNORECASE
)

GOTO_PATTERN = re.compile(r'Go\s+to\s+#\s*(\d+)', re.IGNORECASE)
CRITERIA_NUM_PATTERN = re.compile(r'^\s*(\d+)\.\s')


# === DOCUMENT INDEX ===

class DocumentIndex:
    def __init__(self, doc: DoclingDocument):
        self.doc = doc
        self.items = list(doc.iterate_items())
        self.headers = []
        self.tables = []
        self.text_items = []
        self._build_index()

    def _build_index(self):
        table_counter = 0
        for pos, (item, level) in enumerate(self.items):
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                self.headers.append((pos, text, level))
            elif item_type == 'TableItem':
                self.tables.append((pos, table_counter, item))
                table_counter += 1
            if text.strip():
                self.text_items.append((pos, text, level, item_type))

    def get_table_df(self, table_item):
        try:
            return table_item.export_to_dataframe(self.doc)
        except:
            try:
                return table_item.export_to_dataframe()
            except:
                return None

    def get_text_between(self, start, end):
        parts = []
        for pos, (item, level) in enumerate(self.items):
            if pos < start:
                continue
            if pos >= end:
                break
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                parts.append(f"\n## {text}\n")
            elif item_type == 'TableItem':
                df = self.get_table_df(item)
                if df is not None:
                    parts.append(df.to_markdown(index=False))
                    parts.append("")
            elif item_type == 'ListItem':
                parts.append(f"- {text}")
            elif text.strip():
                parts.append(text)
        return "\n".join(parts)

    def is_structural_header(self, text):
        return bool(STRUCTURAL_HEADERS.match(text.strip()))


# === PHASE 1: FIND BRAND SECTION ===

def find_brand_section(index: DocumentIndex, brand: str) -> tuple:
    """
    Find the document region that contains PA criteria for this brand.
    Returns (start_pos, end_pos). For small docs, returns (0, total_items).
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    total = len(index.items)

    # Small documents: entire document is the section
    if total < 200:
        return (0, total)

    # Strategy 1: Find brand in a section header
    brand_header_pos = None
    for pos, text, level in index.headers:
        text_lower = text.lower().strip()
        if any(a == text_lower or a in text_lower.split() for a in aliases):
            # Verify it's a drug-name header, not a structural one
            if not index.is_structural_header(text):
                brand_header_pos = pos
                break

    if brand_header_pos is not None:
        # Find the end: walk forward to the next non-structural header at same level
        start = brand_header_pos
        brand_level = None
        for pos, text, level in index.headers:
            if pos == brand_header_pos:
                brand_level = level
                continue
            if brand_level is not None and pos > brand_header_pos:
                if level <= brand_level and not index.is_structural_header(text):
                    return (start, pos)
        return (start, total)

    # Strategy 2: Find brand in a table cell, then use adaptive scoping
    for pos, table_idx, table_item in index.tables:
        df = index.get_table_df(table_item)
        if df is None:
            continue
        df_str = df.to_string().lower()
        if any(a in df_str for a in aliases):
            return find_section_scope(index, pos)

    # Strategy 3: Find brand in any text item
    for pos, text, level, item_type in index.text_items:
        if any(a in text.lower() for a in aliases):
            # Use table proximity if tables exist nearby
            nearby_tables = [(tp, ti, tt) for tp, ti, tt in index.tables
                             if abs(tp - pos) < 50]
            if nearby_tables:
                return find_section_scope(index, nearby_tables[0][0])
            # Otherwise take a generous window
            return (max(0, pos - 50), min(total, pos + 200))

    # Fallback: entire document
    print(f"  WARNING: Brand '{brand}' not found. Using full document.")
    return (0, total)


def find_section_scope(index: DocumentIndex, anchor_table_pos: int) -> tuple:
    """Adaptive gap-based section scoping around an anchor table."""
    table_positions = [pos for pos, _, _ in index.tables]
    if not table_positions:
        return (0, len(index.items))

    anchor_idx = None
    for i, (pos, _, _) in enumerate(index.tables):
        if pos == anchor_table_pos:
            anchor_idx = i
            break
    if anchor_idx is None:
        return (0, len(index.items))

    # Seed: 10 tables each direction
    seed_start = max(0, anchor_idx - 10)
    seed_end = min(len(table_positions), anchor_idx + 11)
    seed_positions = table_positions[seed_start:seed_end]

    gaps = [seed_positions[i] - seed_positions[i - 1] for i in range(1, len(seed_positions))]
    if not gaps:
        return (0, len(index.items))

    sorted_gaps = sorted(gaps)
    median_gap = sorted_gaps[len(sorted_gaps) // 2]
    initial_threshold = max(median_gap * 3, 10)

    # Walk backward
    start_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, 0, -1):
        if table_positions[i] - table_positions[i - 1] > initial_threshold:
            break
        start_pos = table_positions[i - 1]

    # Walk forward
    end_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, len(table_positions) - 1):
        if table_positions[i + 1] - table_positions[i] > initial_threshold:
            break
        end_pos = table_positions[i + 1]

    # Refine with 2× largest intra-cluster gap
    cluster = [p for p in table_positions if start_pos <= p <= end_pos]
    if len(cluster) > 1:
        intra_gaps = [cluster[i + 1] - cluster[i] for i in range(len(cluster) - 1)]
        refined = max(max(intra_gaps) * 2, 10)
        start_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, 0, -1):
            if table_positions[i] - table_positions[i - 1] > refined:
                break
            start_pos = table_positions[i - 1]
        end_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, len(table_positions) - 1):
            if table_positions[i + 1] - table_positions[i] > refined:
                break
            end_pos = table_positions[i + 1]

    # Extend to capture context
    end_pos = min(end_pos + 10, len(index.items))
    for pos, text, level in reversed(index.headers):
        if pos < start_pos and start_pos - pos < 30:
            start_pos = pos
            break

    return (start_pos, end_pos)


# === PHASE 2: FIND PsO CONTENT WITHIN SECTION ===

def find_pso_by_headings(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy A: Find PsO-related headings within the scoped section."""
    collected = []
    scoped_headers = [(pos, text, level) for pos, text, level in index.headers
                      if scope_start <= pos < scope_end]

    for i, (pos, text, level) in enumerate(scoped_headers):
        is_pso = any(p.search(text) for p in PSO_PATTERNS)
        is_utility = any(p.search(text) for p in UTILITY_PATTERNS)

        if is_pso or is_utility:
            # End at next header at same or higher level within scope
            end_pos = scope_end
            for j in range(i + 1, len(scoped_headers)):
                next_pos, next_text, next_level = scoped_headers[j]
                if next_level <= level and not index.is_structural_header(next_text):
                    end_pos = next_pos
                    break
            collected.append((pos, end_pos))

    return collected


def find_pso_in_tables(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy B: Find tables containing PsO keywords within the scoped section.
    
    Filters:
    - Skip administrative tables (drug listings, NDC tables, dosing grids)
    - Small tables (<20 rows): include if PsO keyword appears anywhere
    - Criteria-structured tables: include if PsO keyword found
    - Large non-criteria tables: require both brand AND PsO keywords
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    collected = []

    # Column names that indicate a PA criteria table
    CRITERIA_TABLE_PATTERNS = re.compile(
        r'PA\s+Criteria|Criteria\s+Details|Approval|Required\s+Medical|'
        r'Coverage\s+Duration|Renewal|Exclusion|Prescriber|Restriction|'
        r'Authorization|Step\s+Therapy|Clinical\s+Criteria',
        re.IGNORECASE
    )

    # Column names that indicate an administrative/formulary listing table — skip these
    ADMIN_TABLE_PATTERNS = re.compile(
        r'Strength|NDC|Targeted\s+NDC|Day\s+Supply|Dose\s+Form|'
        r'Client\s+Formulary|GPI|HICL|Package\s+Size|Route\s+of\s+Admin|'
        r'AWP|WAC|Unit\s+Price|Wholesale|Generic\s+Name.*Brand\s+Name|'
        r'Brand\s+Name.*Generic\s+Name|HCPCS|J-?Code|CPT',
        re.IGNORECASE
    )

    for pos, table_idx, table_item in index.tables:
        if pos < scope_start or pos >= scope_end:
            continue
        df = index.get_table_df(table_item)
        if df is None:
            continue

        num_rows = len(df)
        col_str = ' '.join(str(c) for c in df.columns)

        # Skip administrative tables entirely
        if ADMIN_TABLE_PATTERNS.search(col_str):
            continue

        # Check if this is a PA criteria table by column names
        is_criteria_table = bool(CRITERIA_TABLE_PATTERNS.search(col_str))

        # Small table or criteria-structured table: include if PsO keyword found
        if num_rows < 20 or is_criteria_table:
            df_str = df.to_string().lower()
            if any(p.search(df_str) for p in PSO_PATTERNS):
                collected.append((max(scope_start, pos - 2), min(scope_end, pos + 2)))

        # Large table: only include if BOTH brand AND PsO found
        else:
            df_str = df.to_string().lower()
            has_brand = any(a in df_str for a in aliases)
            has_pso = any(p.search(df_str) for p in PSO_PATTERNS)
            if has_brand and has_pso:
                collected.append((max(scope_start, pos - 2), min(scope_end, pos + 2)))

    return collected


def follow_cross_references(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy C: Follow 'Go to #X' routing chains within the scoped section."""
    collected = []

    # Get all tables in scope
    scope_tables = []
    for pos, table_idx, table_item in index.tables:
        if scope_start <= pos <= scope_end:
            df = index.get_table_df(table_item)
            if df is not None:
                scope_tables.append((pos, table_idx, df))

    if not scope_tables:
        return collected

    # Find PsO routing entries: tables with both PsO keyword and "Go to #X"
    refs_to_follow = set()
    initial_tables = set()
    for tbl_pos, tbl_idx, tbl_df in scope_tables:
        for row_idx in range(len(tbl_df)):
            row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
            row_lower = row_text.lower()
            has_pso = any(p.search(row_lower) for p in PSO_PATTERNS)
            refs_in_row = GOTO_PATTERN.findall(row_text)
            if has_pso and refs_in_row:
                initial_tables.add(tbl_pos)
                refs_to_follow.update(int(r) for r in refs_in_row)

    if not refs_to_follow:
        return collected

    # BFS: follow references
    visited = set()
    tables_to_include = set(initial_tables)
    queue = list(refs_to_follow)

    while queue:
        target_num = queue.pop(0)
        if target_num in visited:
            continue
        visited.add(target_num)

        for tbl_pos, tbl_idx, tbl_df in scope_tables:
            for row_idx in range(len(tbl_df)):
                cell_0 = str(tbl_df.iloc[row_idx, 0]).strip()

                # Match criteria number in first cell
                match_bare = re.match(r'^(\d+)\.?\s*$', cell_0)
                match_prefix = CRITERIA_NUM_PATTERN.match(cell_0)

                found_num = None
                if match_bare:
                    found_num = int(match_bare.group(1))
                elif match_prefix:
                    found_num = int(match_prefix.group(1))

                # Also check column 1 for 4-column tables where col 0 is just the number
                if found_num is None and tbl_df.shape[1] >= 3:
                    cell_1 = str(tbl_df.iloc[row_idx, 1] if tbl_df.shape[1] > 1 else '').strip()
                    match_c1 = CRITERIA_NUM_PATTERN.match(cell_1)
                    if match_c1:
                        found_num = int(match_c1.group(1))

                if found_num == target_num:
                    tables_to_include.add(tbl_pos)
                    row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
                    new_refs = GOTO_PATTERN.findall(row_text)
                    for r in new_refs:
                        r_int = int(r)
                        if r_int not in visited:
                            queue.append(r_int)

    for tbl_pos in tables_to_include:
        collected.append((max(scope_start, tbl_pos - 2), min(scope_end, tbl_pos + 2)))

    return collected


# === MERGE AND SERIALIZE ===

def merge_ranges(ranges):
    if not ranges:
        return []
    sorted_r = sorted(ranges, key=lambda x: x[0])
    merged = [sorted_r[0]]
    for start, end in sorted_r[1:]:
        ps, pe = merged[-1]
        if start <= pe + 3:
            merged[-1] = (ps, max(pe, end))
        else:
            merged.append((start, end))
    return merged


def serialize(index, ranges):
    parts = []
    seen = set()
    for start, end in ranges:
        if start not in seen:
            parts.append(index.get_text_between(start, end))
            seen.add(start)
    return "\n\n---\n\n".join(parts)


# === MAIN REDUCER ===

MIN_OUTPUT_CHARS = 100
MIN_REDUCTION_RATIO = 0.30   # If output < 30% of brand section, widening triggers
MIN_FULLDOC_REDUCTION = 0.85 # Reducer output trusted only if it cuts >=85% of full doc;
                             # a weaker (messy) reduction means it pulled in noise and
                             # likely missed shared sections -> fall back to raw markdown

def reduce_document(doc: DoclingDocument, brand: str) -> str:
    index = DocumentIndex(doc)
    total_items = len(index.items)
    full_md = doc.export_to_markdown()
    full_len = len(full_md)
    print(f"  Reducer: {total_items} items, {len(index.tables)} tables, {len(index.headers)} headers")

    # Small documents: raw markdown is better than structured serialization.
    # The reducer only adds value for large files where the LLM can't process
    # the full content. For small files, pass through raw markdown.
    if full_len < 100_000:
        print(f"  Small document ({full_len:,} chars). Using raw markdown.")
        return full_md

    # Phase 1: Find brand's section
    scope_start, scope_end = find_brand_section(index, brand)
    scope_size = scope_end - scope_start
    print(f"  Brand section: items {scope_start}-{scope_end} ({scope_size} items)")

    # Confidence check: if scope is suspiciously narrow for a large document,
    # widen to full document and let Phase 2 search everywhere.
    if scope_size < 10 and total_items > 200:
        print(f"  Scope too narrow ({scope_size} items in {total_items}-item doc). Widening to full document.")
        scope_start, scope_end = 0, total_items
        scope_size = total_items

    # Get the full brand section text for comparison
    brand_section_text = index.get_text_between(scope_start, scope_end)
    brand_section_len = len(brand_section_text)

    # Phase 2: Find PsO content within section
    ranges_a = find_pso_by_headings(index, scope_start, scope_end, brand)
    ranges_b = find_pso_in_tables(index, scope_start, scope_end, brand)
    ranges_c = follow_cross_references(index, scope_start, scope_end, brand)
    print(f"  Strategy A (headings): {len(ranges_a)} ranges")
    print(f"  Strategy B (tables): {len(ranges_b)} ranges")
    print(f"  Strategy C (cross-ref): {len(ranges_c)} ranges")

    # If no PsO-specific content found, use raw markdown to preserve formatting
    all_ranges = ranges_a + ranges_b + ranges_c
    if not all_ranges:
        print(f"  No PsO-specific content found. Using raw markdown.")
        print(f"  Output: {full_len:,} chars (0% reduction from {full_len:,})")
        return full_md

    merged = merge_ranges(all_ranges)
    print(f"  Merged: {len(merged)} ranges")

    result = serialize(index, merged)

    # Adaptive widening: if the extracted content is less than 30% of the brand
    # section, the reduction was too aggressive. Fall back to raw markdown which
    # preserves the original formatting the LLM handles well.
    if brand_section_len > 0 and len(result) < brand_section_len * MIN_REDUCTION_RATIO:
        print(f"  Reduction too aggressive ({len(result):,} < {int(brand_section_len * MIN_REDUCTION_RATIO):,}). "
              f"Widening to raw markdown.")
        result = full_md

    # Cleanliness gate: a clean reduction cuts most of the document. If the output
    # is still a large fraction of the full doc (reduction < 85%), the reducer pulled
    # in too much noise (many tables for other drugs/indications) and likely missed
    # shared sections. In that "messy reduction" case the raw markdown is the safer
    # input — it preserves shared duration/specialist/TB sections the LLM can read.
    # Clean reductions (CIMZIA 97%, formulary 99%) pass and keep their focused output.
    if result is not full_md and len(result) > full_len * (1 - MIN_FULLDOC_REDUCTION):
        achieved = (1 - len(result) / full_len) * 100 if full_len else 0
        print(f"  Messy reduction ({achieved:.0f}% < {int(MIN_FULLDOC_REDUCTION*100)}% of full doc). "
              f"Falling back to raw markdown.")
        result = full_md

    # Final recovery: if output is still too short, return full markdown
    if len(result) < MIN_OUTPUT_CHARS and total_items > 50:
        print(f"  Output too short ({len(result)} chars). Falling back to full markdown.")
        result = full_md

    reduction = (1 - len(result) / full_len) * 100 if full_len else 0
    print(f"  Output: {len(result):,} chars ({reduction:.0f}% reduction from {full_len:,})")

    return result

In [ ]:
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument
import openpyxl

OBJ_DIR = "work/doc_objects"
EXCEL_PATH = "PA_Business_Rules.xlsx"
OUT_DIR = Path("work/reduced")

wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
ws = wb['Submissions']
submissions = []
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
    if row[0] and row[1]:
        submissions.append((str(row[0]).strip(), str(row[1]).strip()))
wb.close()
print(f"Loaded {len(submissions)} submissions")

doc_cache = {}
for i, (filename, brand) in enumerate(submissions):
    stem = Path(filename).stem
    print(f"\n[{i+1}/{len(submissions)}] {filename} | {brand}")
    if stem not in doc_cache:
        obj_path = Path(OBJ_DIR) / f"{stem}.json"
        if not obj_path.exists():
            print(f"  ERROR: {obj_path} not found")
            continue
        doc_cache[stem] = DoclingDocument.load_from_json(obj_path)
    result = reduce_document(doc_cache[stem], brand)
    (OUT_DIR / f"{stem}__{brand}.txt").write_text(result, encoding="utf-8")

print(f"\nDone. Outputs in {OUT_DIR}/")

In [ ]:
"""
PA Policy Parameter Extraction Pipeline — Integrated with Document Reducer.

Architecture:
  1. Document Reducer (pre-computed): Docling structured output → algorithmic
     section extraction → focused text per (filename, brand) pair.
  2. This pipeline: reads reduced text → Gemini extraction → validation → CSV.

The reducer handles all document structures locally (heading match, table search,
cross-reference following). The LLM only does the hard reasoning: step counting,
AND/OR logic, phototherapy classification.

Inputs:
  work/reduced/   — Reducer output (primary)
  work/md_output/    — Full markdown (fallback if reducer output missing)
  PA_Business_Rules.xlsx — Submission list

Outputs:
  work/result.csv           — 79-row extraction results
  work/_extraction_log.json — Detailed extraction log

Run: GEMINI_API_KEY=your-key python pipeline_integrated.py
"""
import os
import json
import csv
import re
from pathlib import Path
from google import genai
from google.genai import types
import openpyxl

# === CONFIG ===
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    raise RuntimeError("Set GEMINI_API_KEY env var.")

REDUCED_DIR = os.environ.get("REDUCED_DIR", "work/reduced")
MD_DIR = os.environ.get("MD_DIR", "work/md_output")
EXCEL_PATH = os.environ.get("EXCEL_PATH", "PA_Business_Rules.xlsx")
OUTPUT_CSV = os.environ.get("OUTPUT_CSV", "work/result.csv")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_ID = "gemini-2.5-flash"

# === FDA APPROVED AGES FOR PsO BY BRAND ===
FDA_PSO_AGE = {
    "STELARA": ">=6",
    "TREMFYA": ">=6",
    "ENBREL": ">=4",
    "COSENTYX": ">=6",
    "SKYRIZI": ">=18",
    "OTEZLA": ">=6",
    "BIMZELX": ">=18",
    "SILIQ": ">=18",
    "ILUMYA": ">=18",
    "CIMZIA": ">=18",
    "AMJEVITA": ">=18",
    "REMICADE": ">=18",
    "OTULFI": ">=6",
    "YESINTEK": ">=6",
    "ACITRETIN": ">=18",
}

# === EXTRACTION PROMPT ===
SYSTEM_PROMPT = """You are a pharmaceutical prior authorization policy analyst. You extract structured parameters from PA policy documents for the Plaque Psoriasis (PsO) indication.

You will receive a policy document (or relevant excerpt) and a target brand name. Extract the 12 parameters defined below for that brand's PsO coverage ONLY.

=== CRITICAL DISTINCTION: "NA" vs "No" ===

- "NA" = The information COULD NOT BE FOUND in the policy document. The policy simply does not mention this parameter for PsO. Data is missing/absent.
- "No" = The policy EXPLICITLY STATES there is no restriction or requirement.

If a parameter is not mentioned anywhere in the PsO section or universal criteria, output "NA" — NOT "No".

=== CRITICAL RULES ===

INDICATION FILTERING:
- Extract ONLY for Plaque Psoriasis (PsO). Ignore all other indications.
- If the policy separates "moderate-to-severe" and "severe" PsO, extract ONLY moderate-to-severe criteria.
- SPECIALIST TYPES: Only capture specialists listed in the PsO section. Do NOT include Gastroenterologist or other specialists from CD/UC sections.
- STEP THERAPY: Only count steps from the PsO section and universal/general criteria. Do NOT count steps from other indications.
- SELF-REFERENCING: If the target brand appears as one of several options in a step therapy requirement, exclude it from the option list but still count the step — the patient would use one of the other options. Only exclude the entire step if the target brand is the sole option (a drug cannot be the only prerequisite to access itself). This can occur due to document redlining artifacts.

UNIVERSAL vs INDICATION-SPECIFIC CRITERIA:
- Some policies have "General Requirements" or "Universal Criteria" that apply across all indications, PLUS separate PsO-specific criteria.
- These two sets combine with AND — the patient must satisfy BOTH.
- When counting steps, take the UNION of universal steps and PsO-specific steps.

STEP COUNTING — LEAST RESTRICTIVE PATH:
- OR statements → take the path with FEWER total steps (least restrictive).
- AND statements → patient must complete ALL (union).
- "Tried Drug A OR Drug B" = 1 step, NOT 2.
- "Tried Drug A AND Drug B" = 2 steps.
- "must have tried ONE of the following: [list of 8 drugs]" = 1 step, not 8.
- "must have tried TWO of the following: [list]" = 2 steps.
- Maximum realistic: branded steps rarely exceed 4, generic rarely exceed 3.

STEP CLASSIFICATION:
- Branded/biologic: Named biologic drugs, TNF inhibitors, IL-inhibitors, targeted synthetic drugs, biosimilars.
- Generic: Topical agents, conventional systemics (methotrexate, cyclosporine, acitretin), non-biologic drugs.
- Phototherapy: EXCLUDED from both counts — captured separately.

PHOTOTHERAPY:
- "Yes" = mandatory AND with no alternative path.
- "No" = mentioned but optional (in an OR).
- "NA" = not mentioned or no step criteria.

=== PARAMETER DEFINITIONS ===

1. AGE — Explicit threshold (e.g., ">=18"). "adults" → ">=18". "FDA labeled age" → use the FDA age provided. Not mentioned → "NA". Explicitly no restriction → "No". Two ages → youngest.

2. STEP THERAPY REQUIREMENTS — All step therapy language for PsO. "NA" if none.

3. NUMBER OF STEPS THROUGH BRANDS — Integer as string, "No" if criteria exist but no branded steps, "NA" if no criteria.

4. NUMBER OF STEPS THROUGH GENERIC — Same logic.

5. STEP THROUGH PHOTOTHERAPY — "Yes"=mandatory AND. "No"=optional OR. "NA"=not mentioned.

6. TB TEST REQUIRED — "Yes"/"No"/"NA".

7. QUANTITY LIMITS — Only explicitly labeled "quantity limit". Not "dosage"/"dosing". "NA" if not mentioned.

8. SPECIALIST TYPES — PsO only. "NA" if not mentioned.

9. INITIAL AUTHORIZATION DURATION (MONTHS) — Convert weeks/days/years to months. "NA" if not mentioned.

10. REAUTHORIZATION DURATION (MONTHS) — Same. "NA" if not mentioned.

11. REAUTHORIZATION REQUIRED — "Yes"/"No"/"NA". If reauth duration or requirements is non-NA, must be "Yes".

12. REAUTHORIZATION REQUIREMENTS — Text or "NA".

=== FEW-SHOT EXAMPLES ===

EXAMPLE 1 — Phototherapy in OR, no age mentioned
{
  "age": "NA",
  "step_therapy_requirements": "inadequate response to phototherapy or methotrexate/cyclosporine/acitretin",
  "number_of_steps_through_brands": "No",
  "number_of_steps_through_generic": "1",
  "step_through_phototherapy": "No",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "NA",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "NA",
  "reauthorization_required": "No",
  "reauthorization_requirements": "NA"
}

EXAMPLE 2 — Branded + generic steps with AND
{
  "age": ">=18",
  "step_therapy_requirements": "ONE topical AND TWO biologics/targeted synthetics",
  "number_of_steps_through_brands": "2",
  "number_of_steps_through_generic": "1",
  "step_through_phototherapy": "NA",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "Dermatologist",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "12",
  "reauthorization_required": "Yes",
  "reauthorization_requirements": "Positive clinical response: reduction in BSA or improvement in symptoms"
}

EXAMPLE 3 — OR logic, least restrictive path
{
  "age": ">=6",
  "step_therapy_requirements": "(topical + methotrexate) OR (targeted immunomodulator)",
  "number_of_steps_through_brands": "No",
  "number_of_steps_through_generic": "2",
  "step_through_phototherapy": "NA",
  "tb_test_required": "Yes",
  "quantity_limits": "NA",
  "specialist_types": "Dermatologist",
  "initial_authorization_duration": "12",
  "reauthorization_duration": "12",
  "reauthorization_required": "Yes",
  "reauthorization_requirements": "Positive clinical response AND not combined with another targeted immunomodulator"
}

=== PsO BRAND REFERENCE ===

BRANDED/BIOLOGIC: Amjevita, Avsola, Bimzelx, Cimzia, Cosentyx, Cyltezo, Enbrel, Humira, Hyrimoz, Idacio, Ilumya, Inflectra, Hulio, Otezla, Remicade, Renflexis, Siliq, Skyrizi, Sotyktu, Stelara, Taltz, Tremfya, Vtama, Yuflyma, Yusimry, Zoryve, Wezlana, Selarsdi, Yesintek, Psychiva/Quallent, Steqeyma, Otulfi

GENERIC/CONVENTIONAL: Acitretin, Cyclosporine, Methotrexate, topical corticosteroids, vitamin D analogs, calcineurin inhibitors, tazarotene, coal tar, anthralin

=== OUTPUT FORMAT ===

Return a JSON object with exactly the 12 fields. No extra fields or commentary."""


# === LOAD SUBMISSIONS ===
def load_submissions():
    wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
    ws = wb['Submissions']
    submissions = []
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
        filename, brand = row[0], row[1]
        if not filename or not brand:
            continue
        submissions.append({"filename": str(filename).strip(), "brand": str(brand).strip()})
    wb.close()
    return submissions


# === LOAD CONTENT ===
def load_content(filename, brand):
    """
    Load pre-reduced content for a (filename, brand) pair.
    Priority 1: Reducer output in REDUCED_DIR (algorithmically extracted)
    Priority 2: Full markdown in MD_DIR (fallback)
    """
    stem = Path(filename).stem

    # Priority 1: Reducer output (brand-specific)
    reduced_path = Path(REDUCED_DIR) / f"{stem}__{brand}.txt"
    if reduced_path.exists():
        text = reduced_path.read_text(encoding="utf-8")
        if text.strip():
            return text, "reduced"

    # Priority 2: Full markdown
    md_path = Path(MD_DIR) / f"{stem}.md"
    if md_path.exists():
        return md_path.read_text(encoding="utf-8"), "markdown-fallback"

    return None, "missing"


# === PARSE JSON ===
def parse_json_response(text):
    """Parse JSON from model response, handling markdown fences and preamble."""
    if text is None:
        raise ValueError("Response text is None")

    cleaned = text.strip()

    # Find JSON block inside markdown fences anywhere in response
    fence_match = re.search(r'```(?:json)?\s*\n(.*?)\n```', cleaned, re.DOTALL)
    if fence_match:
        cleaned = fence_match.group(1).strip()
    # Or find first { ... } block
    elif not cleaned.startswith("{"):
        brace_start = cleaned.find("{")
        brace_end = cleaned.rfind("}")
        if brace_start != -1 and brace_end != -1:
            cleaned = cleaned[brace_start:brace_end + 1]

    return json.loads(cleaned)


# === GEMINI CALL HELPER ===
def call_gemini(contents, system_instruction=None, json_output=False, max_retries=3):
    """Generic Gemini call with retry logic."""
    config_params = {
        "temperature": 0.1,
        "thinking_config": types.ThinkingConfig(thinking_level="HIGH"),
    }
    if system_instruction:
        config_params["system_instruction"] = system_instruction
    if json_output:
        config_params["response_mime_type"] = "application/json"

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=contents,
                config=types.GenerateContentConfig(**config_params),
            )

            if not response.candidates:
                raise ValueError(f"No candidates returned: {getattr(response, 'prompt_feedback', 'unknown')}")

            candidate = response.candidates[0]
            finish_reason = getattr(candidate, "finish_reason", None)
            if finish_reason is not None and "STOP" not in str(finish_reason):
                raise ValueError(f"Stopped: {finish_reason}")

            return response.text, None

        except Exception as e:
            print(f"    Error (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                continue

    return None, "max_retries_exceeded"


# === PARAMETER EXTRACTION ===
def extract_parameters(policy_text, brand):
    """Extract 12 PA parameters from policy text."""
    fda_age = FDA_PSO_AGE.get(brand.upper(), ">=18")

    user_prompt = f"""Extract the 12 PA parameters for the brand "{brand}" on the Plaque Psoriasis (PsO) indication from the following policy document.

FDA APPROVED AGE for {brand} in PsO: {fda_age}
(Use this if the policy says "FDA labelled age" without a number.)

POLICY DOCUMENT:
{policy_text}"""

    text, error = call_gemini(user_prompt, system_instruction=SYSTEM_PROMPT, json_output=True)
    if error:
        return None, error

    try:
        result = parse_json_response(text)
        return result, None
    except (json.JSONDecodeError, ValueError) as e:
        return None, f"JSON parse error: {e}"


# === VALIDATE AND NORMALIZE ===
def validate_and_normalize(params, brand):
    # Resolve FDA labelled age
    age = str(params.get("age", "NA")).strip()
    if "fda" in age.lower():
        params["age"] = FDA_PSO_AGE.get(brand.upper(), ">=18")

    # Binary fields
    for key in ["tb_test_required", "reauthorization_required"]:
        val = str(params.get(key, "NA")).strip().lower()
        if val in ("yes", "y", "true"):
            params[key] = "Yes"
        elif val in ("no", "n", "false"):
            params[key] = "No"
        else:
            params[key] = "NA"

    # Durations — extract leading digits, handle text like "12 months"
    for key in ["initial_authorization_duration", "reauthorization_duration"]:
        val = params.get(key, "")
        if val is None or str(val).strip().lower() in ("", "none", "null", "unspecified"):
            params[key] = "NA"
        elif isinstance(val, (int, float)):
            params[key] = str(int(val))
        else:
            cleaned = str(val).strip()
            if cleaned.lower() in ("na", "n/a"):
                params[key] = "NA"
            else:
                digit_match = re.match(r'(\d+)', cleaned)
                params[key] = digit_match.group(1) if digit_match else "NA"

    # Step counts — extract leading digits, normalize 0 to "No"
    for key in ["number_of_steps_through_brands", "number_of_steps_through_generic"]:
        val = params.get(key, "NA")
        val_str = str(val).strip().lower()
        if val_str in ("na", "n/a", "none", ""):
            params[key] = "NA"
        elif val_str in ("no", "0"):
            params[key] = "No"
        elif isinstance(val, (int, float)):
            params[key] = "No" if int(val) == 0 else str(int(val))
        else:
            cleaned = str(val).strip()
            digit_match = re.match(r'(\d+)', cleaned)
            if digit_match:
                num = int(digit_match.group(1))
                params[key] = "No" if num == 0 else str(num)

    # Phototherapy
    photo = str(params.get("step_through_phototherapy", "NA")).strip().lower()
    if photo in ("yes", "y"):
        params["step_through_phototherapy"] = "Yes"
    elif photo in ("no", "n"):
        params["step_through_phototherapy"] = "No"
    else:
        params["step_through_phototherapy"] = "NA"

    # Cross-field reauth
    reauth_dur = str(params.get("reauthorization_duration", "NA")).strip().lower()
    reauth_req = str(params.get("reauthorization_requirements", "NA")).strip().lower()
    if reauth_dur not in ("na", "n/a", "none", "") or \
       reauth_req not in ("na", "n/a", "none", ""):
        params["reauthorization_required"] = "Yes"

    return params


# === CSV COLUMNS ===
CSV_COLUMNS = [
    ("Filename", lambda row, p: row["filename"]),
    ("Brand", lambda row, p: row["brand"]),
    ("Age", lambda row, p: p.get("age", "")),
    ("Step Therapy Requirements Documented in Policy", lambda row, p: p.get("step_therapy_requirements", "")),
    ("Number of Steps through Brands", lambda row, p: p.get("number_of_steps_through_brands", "")),
    ("Number of Steps through Generic", lambda row, p: p.get("number_of_steps_through_generic", "")),
    ("Step through-Phototherapy", lambda row, p: p.get("step_through_phototherapy", "")),
    ("TB Test required", lambda row, p: p.get("tb_test_required", "")),
    ("Quantity Limits", lambda row, p: p.get("quantity_limits", "")),
    ("Specialist Types", lambda row, p: p.get("specialist_types", "")),
    ("Initial Authorization Duration(in-months)", lambda row, p: p.get("initial_authorization_duration", "")),
    ("Reauthorization Duration(in-months)", lambda row, p: p.get("reauthorization_duration", "")),
    ("Reauthorization Required", lambda row, p: p.get("reauthorization_required", "")),
    ("Reauthorization Requirements Documented in Policy", lambda row, p: p.get("reauthorization_requirements", "")),
    ("Access Score", lambda row, p: p.get("access_score", "")),
]


# === MAIN PIPELINE ===
def run_pipeline():
    submissions = load_submissions()
    print(f"Loaded {len(submissions)} submission rows")
    print(f"Reduced dir: {REDUCED_DIR}")
    print(f"Markdown dir: {MD_DIR}")

    if not submissions:
        print("ERROR: No valid submissions found.")
        return

    # Verify directories exist
    if not Path(REDUCED_DIR).exists():
        print(f"WARNING: {REDUCED_DIR} not found. All files will use markdown fallback.")
    if not Path(MD_DIR).exists():
        print(f"ERROR: {MD_DIR} not found. Cannot proceed.")
        return

    results = []
    errors = []
    source_counts = {"reduced": 0, "markdown-fallback": 0, "missing": 0}

    for i, row in enumerate(submissions):
        filename = row["filename"]
        brand = row["brand"]
        print(f"[{i+1}/{len(submissions)}] {filename} | {brand}")

        # Load content (reducer output preferred, markdown fallback)
        policy_text, source = load_content(filename, brand)
        if policy_text is None:
            print(f"  ERROR: content not found ({source})")
            errors.append({"filename": filename, "brand": brand, "error": "file_not_found"})
            results.append({"row": row, "params": {}})
            source_counts["missing"] += 1
            continue

        if not policy_text.strip():
            print(f"  ERROR: empty content ({source})")
            errors.append({"filename": filename, "brand": brand, "error": "empty_file"})
            results.append({"row": row, "params": {}})
            source_counts["empty"] = source_counts.get("empty", 0) + 1
            continue

        source_counts[source] = source_counts.get(source, 0) + 1
        print(f"  Source: {source} ({len(policy_text):,} chars)")

        # Extract parameters
        params, error = extract_parameters(policy_text, brand)
        if error:
            print(f"  EXTRACTION ERROR: {error}")
            errors.append({"filename": filename, "brand": brand, "error": error})
            results.append({"row": row, "params": {}})
            continue

        params = validate_and_normalize(params, brand)
        params["access_score"] = ""

        results.append({"row": row, "params": params})
        print(f"  OK: age={params.get('age')}, brands={params.get('number_of_steps_through_brands')}, "
              f"generic={params.get('number_of_steps_through_generic')}, "
              f"auth={params.get('initial_authorization_duration')}")

    # Write CSV
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([col[0] for col in CSV_COLUMNS])
        for r in results:
            writer.writerow([col[1](r["row"], r["params"]) for col in CSV_COLUMNS])

    # Summary
    print(f"\n{'='*60}")
    print(f"Done. {len(results) - len(errors)}/{len(results)} succeeded.")
    print(f"Sources: {source_counts}")
    print(f"Errors: {len(errors)}")
    if errors:
        for e in errors:
            print(f"  {e['filename']} | {e['brand']}: {e['error']}")
    print(f"Output: {OUTPUT_CSV}")

    # Log
    log_path = Path(OUTPUT_CSV).parent / "_extraction_log.json"
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(
            [{"filename": r["row"]["filename"], "brand": r["row"]["brand"], "params": r["params"]} for r in results],
            f, indent=2,
        )


if __name__ == "__main__":
    run_pipeline()

In [ ]:
# Cell 1 — clear old reduced output
import shutil
from pathlib import Path
out = Path("work/reduced")
if out.exists(): shutil.rmtree(out)
out.mkdir()
print("Cleared")

In [ ]:
"""
Document Reducer v2 — Extracts PsO-relevant sections from Docling structured documents.

Architecture:
  Phase 1: Find the brand's section boundaries
  Phase 2: Within those boundaries, find PsO criteria + utility sections

Three strategies for Phase 2: heading match, table search, cross-reference following.
No LLM calls. Pure algorithmic document understanding.
"""
import re
from pathlib import Path
from docling_core.types.doc.document import DoclingDocument


def alias_in_text(alias: str, text: str) -> bool:
    """
    Word-boundary-aware alias match. Prevents an INN like "ustekinumab" from
    matching inside a biosimilar token like "ustekinumab-kfce" (Yesintek) or
    "ustekinumab-aauz" (Otulfi), which would silently mis-scope a multi-drug
    formulary. The alias must appear as a whole token, not as a prefix of a
    longer hyphenated biosimilar name. Generalizes to any biosimilar family.
    """
    # \b...\b alone treats '-' as a boundary, so "ustekinumab" would still match
    # "ustekinumab-kfce". Require that the alias is NOT immediately followed by a
    # hyphen+suffix (the biosimilar pattern) and not part of a longer word.
    pattern = r'(?<![A-Za-z0-9])' + re.escape(alias) + r'(?![A-Za-z0-9-])'
    return re.search(pattern, text) is not None


def any_alias_in_text(aliases, text: str) -> bool:
    return any(alias_in_text(a, text) for a in aliases)


# === BRAND ALIASES ===
BRAND_ALIASES = {
    "TREMFYA": ["tremfya", "guselkumab"],
    "STELARA": ["stelara", "ustekinumab"],
    "ENBREL": ["enbrel", "etanercept"],
    "AMJEVITA": ["amjevita", "adalimumab-atto", "adalimumab"],
    "COSENTYX": ["cosentyx", "secukinumab"],
    "SKYRIZI": ["skyrizi", "risankizumab"],
    "OTEZLA": ["otezla", "apremilast"],
    "SILIQ": ["siliq", "brodalumab"],
    "CIMZIA": ["cimzia", "certolizumab"],
    "BIMZELX": ["bimzelx", "bimekizumab"],
    "REMICADE": ["remicade", "infliximab"],
    "OTULFI": ["otulfi", "ustekinumab-aauz"],
    "ILUMYA": ["ilumya", "tildrakizumab"],
    "YESINTEK": ["yesintek", "ustekinumab-kfce"],
    "ACITRETIN": ["acitretin"],
}

PSO_PATTERNS = [
    re.compile(r'\bplaque\s+psoriasis\b', re.IGNORECASE),
    re.compile(r'\bPsO\b'),
    re.compile(r'\bpsoriasis\b', re.IGNORECASE),
]

UTILITY_PATTERNS = [
    # Duration and authorization
    re.compile(r'\bduration\s+of\s+approval\b', re.IGNORECASE),
    re.compile(r'\binitial\s+auth', re.IGNORECASE),
    re.compile(r'\binitial\s+approval\b', re.IGNORECASE),
    re.compile(r'\blength\s+of\s+auth', re.IGNORECASE),
    # Reauthorization / renewal
    re.compile(r'\breauthorization\b', re.IGNORECASE),
    re.compile(r'\brenewal\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bcontinuation\s+criteria\b', re.IGNORECASE),
    # Quantity limits
    re.compile(r'\bquantity\s+limit', re.IGNORECASE),
    re.compile(r'\bdosing\s+limit', re.IGNORECASE),
    re.compile(r'\bmax.*\s+dose\b', re.IGNORECASE),
    # Approval / denial
    re.compile(r'\bapproval\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bdenial\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bclinical\s+criteria\b', re.IGNORECASE),
    re.compile(r'\bcoverage\s+criteria\b', re.IGNORECASE),
    # TB test / screening
    re.compile(r'\btuberculosis\b', re.IGNORECASE),
    re.compile(r'\bTB\s+test', re.IGNORECASE),
    re.compile(r'\bTB\s+screen', re.IGNORECASE),
    re.compile(r'\blatent\s+TB\b', re.IGNORECASE),
    re.compile(r'\bscreening\s+requirement', re.IGNORECASE),
    # Specialist / prescriber
    re.compile(r'\bspecialist', re.IGNORECASE),
    re.compile(r'\bprescriber\s+requirement', re.IGNORECASE),
    re.compile(r'\bprescribing\s+provider', re.IGNORECASE),
    re.compile(r'\bdermatolog', re.IGNORECASE),
    re.compile(r'\brheumatolog', re.IGNORECASE),
    # General requirements (often contain TB, specialist, etc.)
    re.compile(r'\bgeneral\s+requirement', re.IGNORECASE),
    re.compile(r'\bgeneral\s+criteria\b', re.IGNORECASE),
    re.compile(r'\buniversal\s+criteria\b', re.IGNORECASE),
    re.compile(r'\brequired\s+medical\s+info', re.IGNORECASE),
    re.compile(r'\bclinical\s+review\s+criteria\b', re.IGNORECASE),
    # Monitoring / testing
    re.compile(r'\bmonitoring\s+requirement', re.IGNORECASE),
    re.compile(r'\blab.*\s+requirement', re.IGNORECASE),
    re.compile(r'\bprecaution', re.IGNORECASE),
    re.compile(r'\bcontraindication', re.IGNORECASE),
]

# Structural sub-headers that repeat in every section — NOT drug-class boundaries
STRUCTURAL_HEADERS = re.compile(
    r'^\s*('
    r'Products?\s+Affected|'
    r'Goal\(?s?\)?:?|'
    r'Length\s+of\s+Authorization:?|'
    r'Requires\s+PA:?|'
    r'Covered\s+Alternatives:?|'
    r'Approval\s+Criteria|'
    r'Renewal\s+Criteria|'
    r'References?:?|'
    r'Table\s+\d|'
    r'PA\s+Criteria|'
    r'P&T|'
    r'Effective\s+Date|'
    r'Notes?:?'
    r')\s*$',
    re.IGNORECASE
)

GOTO_PATTERN = re.compile(r'Go\s+to\s+#\s*(\d+)', re.IGNORECASE)
CRITERIA_NUM_PATTERN = re.compile(r'^\s*(\d+)\.\s')


# === DOCUMENT INDEX ===

class DocumentIndex:
    def __init__(self, doc: DoclingDocument):
        self.doc = doc
        self.items = list(doc.iterate_items())
        self.headers = []
        self.tables = []
        self.text_items = []
        self._build_index()

    def _build_index(self):
        table_counter = 0
        for pos, (item, level) in enumerate(self.items):
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                self.headers.append((pos, text, level))
            elif item_type == 'TableItem':
                self.tables.append((pos, table_counter, item))
                table_counter += 1
            if text.strip():
                self.text_items.append((pos, text, level, item_type))

    def get_table_df(self, table_item):
        try:
            return table_item.export_to_dataframe(self.doc)
        except:
            try:
                return table_item.export_to_dataframe()
            except:
                return None

    def get_text_between(self, start, end):
        parts = []
        for pos, (item, level) in enumerate(self.items):
            if pos < start:
                continue
            if pos >= end:
                break
            item_type = type(item).__name__
            text = getattr(item, 'text', '') or ''
            if item_type == 'SectionHeaderItem':
                parts.append(f"\n## {text}\n")
            elif item_type == 'TableItem':
                df = self.get_table_df(item)
                if df is not None:
                    parts.append(df.to_markdown(index=False))
                    parts.append("")
            elif item_type == 'ListItem':
                parts.append(f"- {text}")
            elif text.strip():
                parts.append(text)
        return "\n".join(parts)

    def is_structural_header(self, text):
        return bool(STRUCTURAL_HEADERS.match(text.strip()))


# === PHASE 1: FIND BRAND SECTION ===

def find_brand_section(index: DocumentIndex, brand: str) -> tuple:
    """
    Find the document region that contains PA criteria for this brand.
    Returns (start_pos, end_pos). For small docs, returns (0, total_items).
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    total = len(index.items)

    # Small documents: entire document is the section
    if total < 200:
        return (0, total)

    # Strategy 1: Find brand in a section header
    brand_header_pos = None
    for pos, text, level in index.headers:
        text_lower = text.lower().strip()
        if any(a == text_lower or alias_in_text(a, text_lower) for a in aliases):
            # Verify it's a drug-name header, not a structural one
            if not index.is_structural_header(text):
                brand_header_pos = pos
                break

    if brand_header_pos is not None:
        # Find the end: walk forward to the next non-structural header at same level
        start = brand_header_pos
        brand_level = None
        for pos, text, level in index.headers:
            if pos == brand_header_pos:
                brand_level = level
                continue
            if brand_level is not None and pos > brand_header_pos:
                if level <= brand_level and not index.is_structural_header(text):
                    return (start, pos)
        return (start, total)

    # Strategy 2: Find brand in a table cell, then use adaptive scoping
    for pos, table_idx, table_item in index.tables:
        df = index.get_table_df(table_item)
        if df is None:
            continue
        df_str = df.to_string().lower()
        if any_alias_in_text(aliases, df_str):
            return find_section_scope(index, pos)

    # Strategy 3: Find brand in any text item
    for pos, text, level, item_type in index.text_items:
        if any_alias_in_text(aliases, text.lower()):
            # Use table proximity if tables exist nearby
            nearby_tables = [(tp, ti, tt) for tp, ti, tt in index.tables
                             if abs(tp - pos) < 50]
            if nearby_tables:
                return find_section_scope(index, nearby_tables[0][0])
            # Otherwise take a generous window
            return (max(0, pos - 50), min(total, pos + 200))

    # Fallback: entire document
    print(f"  WARNING: Brand '{brand}' not found. Using full document.")
    return (0, total)


def find_section_scope(index: DocumentIndex, anchor_table_pos: int) -> tuple:
    """Adaptive gap-based section scoping around an anchor table."""
    table_positions = [pos for pos, _, _ in index.tables]
    if not table_positions:
        return (0, len(index.items))

    anchor_idx = None
    for i, (pos, _, _) in enumerate(index.tables):
        if pos == anchor_table_pos:
            anchor_idx = i
            break
    if anchor_idx is None:
        return (0, len(index.items))

    # Seed: 10 tables each direction
    seed_start = max(0, anchor_idx - 10)
    seed_end = min(len(table_positions), anchor_idx + 11)
    seed_positions = table_positions[seed_start:seed_end]

    gaps = [seed_positions[i] - seed_positions[i - 1] for i in range(1, len(seed_positions))]
    if not gaps:
        return (0, len(index.items))

    sorted_gaps = sorted(gaps)
    median_gap = sorted_gaps[len(sorted_gaps) // 2]
    initial_threshold = max(median_gap * 3, 10)

    # Walk backward
    start_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, 0, -1):
        if table_positions[i] - table_positions[i - 1] > initial_threshold:
            break
        start_pos = table_positions[i - 1]

    # Walk forward
    end_pos = table_positions[anchor_idx]
    for i in range(anchor_idx, len(table_positions) - 1):
        if table_positions[i + 1] - table_positions[i] > initial_threshold:
            break
        end_pos = table_positions[i + 1]

    # Refine with 2× largest intra-cluster gap
    cluster = [p for p in table_positions if start_pos <= p <= end_pos]
    if len(cluster) > 1:
        intra_gaps = [cluster[i + 1] - cluster[i] for i in range(len(cluster) - 1)]
        refined = max(max(intra_gaps) * 2, 10)
        start_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, 0, -1):
            if table_positions[i] - table_positions[i - 1] > refined:
                break
            start_pos = table_positions[i - 1]
        end_pos = table_positions[anchor_idx]
        for i in range(anchor_idx, len(table_positions) - 1):
            if table_positions[i + 1] - table_positions[i] > refined:
                break
            end_pos = table_positions[i + 1]

    # Extend to capture context
    end_pos = min(end_pos + 10, len(index.items))
    for pos, text, level in reversed(index.headers):
        if pos < start_pos and start_pos - pos < 30:
            start_pos = pos
            break

    return (start_pos, end_pos)


# === PHASE 2: FIND PsO CONTENT WITHIN SECTION ===

def find_pso_by_headings(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy A: Find PsO-related headings within the scoped section."""
    collected = []
    scoped_headers = [(pos, text, level) for pos, text, level in index.headers
                      if scope_start <= pos < scope_end]

    for i, (pos, text, level) in enumerate(scoped_headers):
        is_pso = any(p.search(text) for p in PSO_PATTERNS)
        is_utility = any(p.search(text) for p in UTILITY_PATTERNS)

        if is_pso or is_utility:
            # End at next header at same or higher level within scope
            end_pos = scope_end
            for j in range(i + 1, len(scoped_headers)):
                next_pos, next_text, next_level = scoped_headers[j]
                if next_level <= level and not index.is_structural_header(next_text):
                    end_pos = next_pos
                    break
            collected.append((pos, end_pos))

    return collected


def find_pso_in_tables(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy B: Find tables containing PsO keywords within the scoped section.
    
    Filters:
    - Skip administrative tables (drug listings, NDC tables, dosing grids)
    - Small tables (<20 rows): include if PsO keyword appears anywhere
    - Criteria-structured tables: include if PsO keyword found
    - Large non-criteria tables: require both brand AND PsO keywords
    """
    aliases = BRAND_ALIASES.get(brand.upper(), [brand.lower()])
    collected = []

    # Column names that indicate a PA criteria table
    CRITERIA_TABLE_PATTERNS = re.compile(
        r'PA\s+Criteria|Criteria\s+Details|Approval|Required\s+Medical|'
        r'Coverage\s+Duration|Renewal|Exclusion|Prescriber|Restriction|'
        r'Authorization|Step\s+Therapy|Clinical\s+Criteria',
        re.IGNORECASE
    )

    # Column names that indicate an administrative/formulary listing table — skip these
    ADMIN_TABLE_PATTERNS = re.compile(
        r'Strength|NDC|Targeted\s+NDC|Day\s+Supply|Dose\s+Form|'
        r'Client\s+Formulary|GPI|HICL|Package\s+Size|Route\s+of\s+Admin|'
        r'AWP|WAC|Unit\s+Price|Wholesale|Generic\s+Name.*Brand\s+Name|'
        r'Brand\s+Name.*Generic\s+Name|HCPCS|J-?Code|CPT',
        re.IGNORECASE
    )

    for pos, table_idx, table_item in index.tables:
        if pos < scope_start or pos >= scope_end:
            continue
        df = index.get_table_df(table_item)
        if df is None:
            continue

        num_rows = len(df)
        col_str = ' '.join(str(c) for c in df.columns)

        # Skip administrative tables entirely
        if ADMIN_TABLE_PATTERNS.search(col_str):
            continue

        # Check if this is a PA criteria table by column names
        is_criteria_table = bool(CRITERIA_TABLE_PATTERNS.search(col_str))

        # Small table or criteria-structured table: include if PsO keyword found
        if num_rows < 20 or is_criteria_table:
            df_str = df.to_string().lower()
            if any(p.search(df_str) for p in PSO_PATTERNS):
                collected.append((max(scope_start, pos - 2), min(scope_end, pos + 2)))

        # Large table: only include if BOTH brand AND PsO found
        else:
            df_str = df.to_string().lower()
            has_brand = any_alias_in_text(aliases, df_str)
            has_pso = any(p.search(df_str) for p in PSO_PATTERNS)
            if has_brand and has_pso:
                collected.append((max(scope_start, pos - 2), min(scope_end, pos + 2)))

    return collected


def follow_cross_references(index: DocumentIndex, scope_start: int, scope_end: int, brand: str) -> list:
    """Strategy C: Follow 'Go to #X' routing chains within the scoped section."""
    collected = []

    # Get all tables in scope
    scope_tables = []
    for pos, table_idx, table_item in index.tables:
        if scope_start <= pos <= scope_end:
            df = index.get_table_df(table_item)
            if df is not None:
                scope_tables.append((pos, table_idx, df))

    if not scope_tables:
        return collected

    # Find PsO routing entries: tables with both PsO keyword and "Go to #X"
    refs_to_follow = set()
    initial_tables = set()
    for tbl_pos, tbl_idx, tbl_df in scope_tables:
        for row_idx in range(len(tbl_df)):
            row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
            row_lower = row_text.lower()
            has_pso = any(p.search(row_lower) for p in PSO_PATTERNS)
            refs_in_row = GOTO_PATTERN.findall(row_text)
            if has_pso and refs_in_row:
                initial_tables.add(tbl_pos)
                refs_to_follow.update(int(r) for r in refs_in_row)

    if not refs_to_follow:
        return collected

    # BFS: follow references
    visited = set()
    tables_to_include = set(initial_tables)
    queue = list(refs_to_follow)

    while queue:
        target_num = queue.pop(0)
        if target_num in visited:
            continue
        visited.add(target_num)

        for tbl_pos, tbl_idx, tbl_df in scope_tables:
            for row_idx in range(len(tbl_df)):
                cell_0 = str(tbl_df.iloc[row_idx, 0]).strip()

                # Match criteria number in first cell
                match_bare = re.match(r'^(\d+)\.?\s*$', cell_0)
                match_prefix = CRITERIA_NUM_PATTERN.match(cell_0)

                found_num = None
                if match_bare:
                    found_num = int(match_bare.group(1))
                elif match_prefix:
                    found_num = int(match_prefix.group(1))

                # Also check column 1 for 4-column tables where col 0 is just the number
                if found_num is None and tbl_df.shape[1] >= 3:
                    cell_1 = str(tbl_df.iloc[row_idx, 1] if tbl_df.shape[1] > 1 else '').strip()
                    match_c1 = CRITERIA_NUM_PATTERN.match(cell_1)
                    if match_c1:
                        found_num = int(match_c1.group(1))

                if found_num == target_num:
                    tables_to_include.add(tbl_pos)
                    row_text = ' '.join(str(v) for v in tbl_df.iloc[row_idx].tolist())
                    new_refs = GOTO_PATTERN.findall(row_text)
                    for r in new_refs:
                        r_int = int(r)
                        if r_int not in visited:
                            queue.append(r_int)

    for tbl_pos in tables_to_include:
        collected.append((max(scope_start, tbl_pos - 2), min(scope_end, tbl_pos + 2)))

    return collected


# === MERGE AND SERIALIZE ===

def merge_ranges(ranges):
    if not ranges:
        return []
    sorted_r = sorted(ranges, key=lambda x: x[0])
    merged = [sorted_r[0]]
    for start, end in sorted_r[1:]:
        ps, pe = merged[-1]
        if start <= pe + 3:
            merged[-1] = (ps, max(pe, end))
        else:
            merged.append((start, end))
    return merged


def serialize(index, ranges):
    parts = []
    seen = set()
    for start, end in ranges:
        if start not in seen:
            parts.append(index.get_text_between(start, end))
            seen.add(start)
    return "\n\n---\n\n".join(parts)


# === MAIN REDUCER ===

MIN_OUTPUT_CHARS = 100
MIN_REDUCTION_RATIO = 0.30   # If output < 30% of brand section, widening triggers
MIN_FULLDOC_REDUCTION = 0.85 # Reducer output trusted only if it cuts >=85% of full doc;
                             # a weaker (messy) reduction means it pulled in noise and
                             # likely missed shared sections -> fall back to raw markdown

def reduce_document(doc: DoclingDocument, brand: str) -> str:
    index = DocumentIndex(doc)
    total_items = len(index.items)
    full_md = doc.export_to_markdown()
    full_len = len(full_md)
    print(f"  Reducer: {total_items} items, {len(index.tables)} tables, {len(index.headers)} headers")

    # Small documents: raw markdown is better than structured serialization.
    # The reducer only adds value for large files where the LLM can't process
    # the full content. For small files, pass through raw markdown.
    if full_len < 100_000:
        print(f"  Small document ({full_len:,} chars). Using raw markdown.")
        return full_md

    # Phase 1: Find brand's section
    scope_start, scope_end = find_brand_section(index, brand)
    scope_size = scope_end - scope_start
    print(f"  Brand section: items {scope_start}-{scope_end} ({scope_size} items)")

    # Confidence check: if scope is suspiciously narrow for a large document,
    # widen to full document and let Phase 2 search everywhere.
    if scope_size < 10 and total_items > 200:
        print(f"  Scope too narrow ({scope_size} items in {total_items}-item doc). Widening to full document.")
        scope_start, scope_end = 0, total_items
        scope_size = total_items

    # Get the full brand section text for comparison
    brand_section_text = index.get_text_between(scope_start, scope_end)
    brand_section_len = len(brand_section_text)

    # Phase 2: Find PsO content within section
    ranges_a = find_pso_by_headings(index, scope_start, scope_end, brand)
    ranges_b = find_pso_in_tables(index, scope_start, scope_end, brand)
    ranges_c = follow_cross_references(index, scope_start, scope_end, brand)
    print(f"  Strategy A (headings): {len(ranges_a)} ranges")
    print(f"  Strategy B (tables): {len(ranges_b)} ranges")
    print(f"  Strategy C (cross-ref): {len(ranges_c)} ranges")

    # If no PsO-specific content found, use raw markdown to preserve formatting
    all_ranges = ranges_a + ranges_b + ranges_c
    if not all_ranges:
        print(f"  No PsO-specific content found. Using raw markdown.")
        print(f"  Output: {full_len:,} chars (0% reduction from {full_len:,})")
        return full_md

    merged = merge_ranges(all_ranges)
    print(f"  Merged: {len(merged)} ranges")

    result = serialize(index, merged)

    # Adaptive widening: if the extracted content is less than 30% of the brand
    # section, the reduction was too aggressive. Fall back to raw markdown which
    # preserves the original formatting the LLM handles well.
    if brand_section_len > 0 and len(result) < brand_section_len * MIN_REDUCTION_RATIO:
        print(f"  Reduction too aggressive ({len(result):,} < {int(brand_section_len * MIN_REDUCTION_RATIO):,}). "
              f"Widening to raw markdown.")
        result = full_md

    # Cleanliness gate: a clean reduction cuts most of the document. If the output
    # is still a large fraction of the full doc (reduction < 85%), the reducer pulled
    # in too much noise (many tables for other drugs/indications) and likely missed
    # shared sections. In that "messy reduction" case the raw markdown is the safer
    # input — it preserves shared duration/specialist/TB sections the LLM can read.
    # Clean reductions (CIMZIA 97%, formulary 99%) pass and keep their focused output.
    if result is not full_md and len(result) > full_len * (1 - MIN_FULLDOC_REDUCTION):
        achieved = (1 - len(result) / full_len) * 100 if full_len else 0
        print(f"  Messy reduction ({achieved:.0f}% < {int(MIN_FULLDOC_REDUCTION*100)}% of full doc). "
              f"Falling back to raw markdown.")
        result = full_md

    # Final recovery: if output is still too short, return full markdown
    if len(result) < MIN_OUTPUT_CHARS and total_items > 50:
        print(f"  Output too short ({len(result)} chars). Falling back to full markdown.")
        result = full_md

    reduction = (1 - len(result) / full_len) * 100 if full_len else 0
    print(f"  Output: {len(result):,} chars ({reduction:.0f}% reduction from {full_len:,})")

    return result

In [ ]:
# First: paste the ENTIRE contents of document_reducer_v3.py and run it
# (defines reduce_document, find_brand_section, the matcher, etc.)
# Then run this batch loop:

from pathlib import Path
from docling_core.types.doc.document import DoclingDocument
import openpyxl

OBJ_DIR = "work/doc_objects"
EXCEL_PATH = "PA_Business_Rules.xlsx"
OUT_DIR = Path("work/reduced")

wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
ws = wb['Submissions']
submissions = []
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
    if row[0] and row[1]:
        submissions.append((str(row[0]).strip(), str(row[1]).strip()))
wb.close()
print(f"Loaded {len(submissions)} submissions")

doc_cache = {}
for i, (filename, brand) in enumerate(submissions):
    stem = Path(filename).stem
    print(f"\n[{i+1}/{len(submissions)}] {filename} | {brand}")
    if stem not in doc_cache:
        obj_path = Path(OBJ_DIR) / f"{stem}.json"
        if not obj_path.exists():
            print(f"  ERROR: {obj_path} not found")
            continue
        doc_cache[stem] = DoclingDocument.load_from_json(obj_path)
    result = reduce_document(doc_cache[stem], brand)
    (OUT_DIR / f"{stem}__{brand}.txt").write_text(result, encoding="utf-8")

print(f"\nDone. Outputs in {OUT_DIR}/")